In [0]:
%pip install pymc arviz numpyro jax jaxlib pytensor scikit-learn openpyxl scipy --quiet
dbutils.library.restartPython()

In [0]:
import os, json, warnings, numpy as np, pandas as pd
import pytensor.tensor as pt
from sklearn.metrics import mean_absolute_percentage_error
warnings.simplefilter("ignore")

# ── CONFIG ──
PATH = "/Volumes/data_exploration_hub/marketing_hub/model_data/MMM_Model_Data/LBE/June_2026/lbe_model_input_2026(June).csv"
OUTDIR = "/Volumes/data_exploration_hub/marketing_hub/model_data/MMM_Model_Data/LBE/June_2026/MMM_Training_Outputs"
ADSTOCK_L = 8
VAL_WEEKS = 18   # single holdout: last 18 weeks for validation, rest for training
DRAWS, TUNE, CHAINS = 2000, 2000, 4
TRAIN_FRAC = 0.95

# ── LOAD DATA ──
d = pd.read_csv(PATH)
d["week_date"] = pd.to_datetime(d["week_date"])
d = d.dropna(subset=["week_date"]).sort_values("week_date").reset_index(drop=True)

# Encode the single Y/N flag
d["is_long_weekend"] = (d["is_long_weekend"].astype(str) == "Y").astype(int)

# ── BUILD BRAND-LEVEL SERIES ──
d["TRR_Rev"] = d[["Revenue_TRR_HPL", "Revenue_TRR_LPL", "Revenue_TRR_NL"]].sum(axis=1)
d["PDM_Rev"] = d[["Revenue_PDM_NL", "Revenue_PDM_OL"]].sum(axis=1)
d["TRR_Txns"] = d[["TRR_HPL_Txns", "TRR_LPL_Txns", "TRR_NL_Txns"]].sum(axis=1)
d["PDM_Txns"] = d[["PDM_NL_Txns", "PDM_OL_Txns"]].sum(axis=1)

# ── MEDIA SPEND ROLLUPS (folded tier schema — no National/Other) ──
CH = {
    "TRR": {
        "Awareness":     ["Awareness_TRR_HPL", "Awareness_TRR_LPL", "Awareness_TRR_NL"],
        "Consideration": ["Consideration_TRR_HPL", "Consideration_TRR_LPL", "Consideration_TRR_NL"],
    },
    "PDM": {
        "Awareness":     ["Awareness_PDM_NL", "Awareness_PDM_OL"],
        "Consideration": ["Consideration_PDM_NL", "Consideration_PDM_OL"],
    },
}
for b in CH:
    for stage, cols in CH[b].items():
        cols = [c for c in cols if c in d.columns and d[c].sum() > 0]
        d[f"{b}_{stage}_spend"] = d[cols].sum(axis=1)

CONTROLS = ["national_mean_temp", "national_rain_mm", "headline_cpi", "is_new_year_wk",
            "is_march_break", "is_pre_xmas", "is_summer_peak", "is_back_to_school",
            "is_spring_lull", "is_long_weekend"]

print(f"✓ Data loaded: {len(d)} weeks ({d['week_date'].min().date()} → {d['week_date'].max().date()})")
print(f"  TRR: ${d['TRR_Rev'].sum():,.0f} revenue, {d['TRR_Txns'].sum():,.0f} txns")
print(f"  PDM: ${d['PDM_Rev'].sum():,.0f} revenue, {d['PDM_Txns'].sum():,.0f} txns")

In [0]:
import pymc as pm
import pytensor.tensor as pt
import arviz as az

def adstock_np(x, a, L=ADSTOCK_L):
    """Numpy geometric adstock (for reconstruction)."""
    out = x.astype(float).copy()
    for lag in range(1, L):
        out[lag:] += a ** lag * x[:-lag]
    return out

def adstock_pt(x, a, L=ADSTOCK_L):
    """PyTensor geometric adstock (for model)."""
    acc = x
    for lag in range(1, L):
        shifted = pt.concatenate([pt.zeros(lag), x[:-lag]])
        acc = acc + (a ** lag) * shifted
    return acc

print("✓ Helpers ready")
print(f"  Adstock: geometric, {ADSTOCK_L}-week window")
print(f"  Saturation: Hill (half-saturation)")
# print(f"  Train/val: {TRAIN_FRAC:.0%} / {1-TRAIN_FRAC:.0%}")

In [0]:
# ── V5e: Dual-Target Model — sat_half FIXED at data anchor ──

# Reference adstock for computing the anchor (prior means)
REF_ADSTOCK_AW = 6/(6+3)   # Beta(6,3) mean = 0.667
REF_ADSTOCK_CO = 3/(3+3)   # Beta(3,3) mean = 0.5

def fit_brand_v5e(brand):
    """Dual-target model: Model A targets log(Rev/Txn), Model B targets log(Txns).
    Combined ROI = value_lift + volume_lift.
    Saturation midpoint FIXED at median of adstocked normalized spend (>0)."""
    n = len(d); n_tr = int(n * TRAIN_FRAC)

    rev = d[f"{brand}_Rev"].values.astype(float)
    txn = d[f"{brand}_Txns"].values.astype(float)

    rpt = np.where(txn > 0, rev / txn, np.nan)
    rpt_clean = np.where(~np.isnan(rpt), rpt, np.nanmedian(rpt))
    y_rpt = np.log(np.maximum(rpt_clean, 1.0))
    y_txn = np.log(np.maximum(txn, 1.0))

    M = d[[f"{brand}_Awareness_spend", f"{brand}_Consideration_spend"]].values.astype(float)
    X = d[CONTROLS].values.astype(float)
    m_s = M[:n_tr].max(0); m_s[m_s == 0] = 1.0
    Mz = np.nan_to_num(M / m_s)
    x_mu, x_sd = X[:n_tr].mean(0), X[:n_tr].std(0); x_sd[x_sd == 0] = 1.0
    Xz = np.nan_to_num((X - x_mu) / x_sd)
    aw_n = Mz[:n_tr, 0]; co_n = Mz[:n_tr, 1]

    # ── Compute anchors: median of adstocked spend (>0), using reference adstock ──
    aw_ref = adstock_np(aw_n, REF_ADSTOCK_AW)
    co_ref = adstock_np(co_n, REF_ADSTOCK_CO)
    anchor_aw = float(np.median(aw_ref[aw_ref > 0])) if (aw_ref > 0).any() else 1.0
    anchor_co = float(np.median(co_ref[co_ref > 0])) if (co_ref > 0).any() else 1.0
    print(f"    Anchors: aw={anchor_aw:.3f}, co={anchor_co:.3f}")

    def _build_and_sample(target, label):
        """Adstock (sampled) → Hill saturation (h fixed at data anchor) → linear beta."""
        with pm.Model() as model:
            alpha_aw = pm.Beta("adstock_aw", 6, 3)
            alpha_co = pm.Beta("adstock_co", 3, 3)
            aw_ad = adstock_pt(pt.as_tensor(aw_n), alpha_aw)
            co_ad = adstock_pt(pt.as_tensor(co_n), alpha_co)

            # Hill saturation with FIXED midpoint (no ridge with beta)
            aw_sat = aw_ad / (aw_ad + anchor_aw)
            co_sat = co_ad / (co_ad + anchor_co)

            beta = pm.HalfNormal("beta_media", 0.5, shape=2)

            gamma_raw = pm.Normal("beta_ctrl_raw", 0, 1, shape=Xz.shape[1])
            gamma = pm.Deterministic("beta_ctrl", gamma_raw * 0.5)

            intercept_raw = pm.Normal("intercept_raw", 0, 1)
            intercept = pm.Deterministic("intercept", target[:n_tr].mean() + intercept_raw * 0.5)

            sigma = pm.HalfNormal("sigma", 0.3)
            nu = pm.Gamma("nu", 2, 0.1)

            mu = intercept + pt.dot(pt.as_tensor(Xz[:n_tr]), gamma) + beta[0]*aw_sat + beta[1]*co_sat
            pm.StudentT("y", nu=nu, mu=mu, sigma=sigma, observed=target[:n_tr])

            idata = pm.sample(DRAWS, tune=TUNE, chains=CHAINS, target_accept=0.995,
                              nuts_sampler="numpyro", random_seed=42)
        return idata

    print(f"    Fitting Model A: log(Rev/Txn)...")
    idata_rpt = _build_and_sample(y_rpt, "rpt")
    print(f"    Fitting Model B: log(Txns)...")
    idata_txn = _build_and_sample(y_txn, "txn")

    def _reconstruct(idata, Mz_full, Xz_full, h_aw, h_co):
        po = idata.posterior
        a_aw = float(po["adstock_aw"].mean())
        a_co = float(po["adstock_co"].mean())
        b = po["beta_media"].mean(dim=["chain","draw"]).values
        g = po["beta_ctrl"].mean(dim=["chain","draw"]).values
        ic = float(po["intercept"].mean())

        aw_full = adstock_np(Mz_full[:,0], a_aw)
        co_full = adstock_np(Mz_full[:,1], a_co)
        aw_sat = aw_full / (aw_full + h_aw)
        co_sat = co_full / (co_full + h_co)
        m_aw = b[0] * aw_sat
        m_co = b[1] * co_sat
        log_pred = ic + Xz_full @ g + m_aw + m_co
        return log_pred, m_aw, m_co, b, a_aw, a_co

    log_pred_rpt, m_aw_rpt, m_co_rpt, b_rpt, a_aw_rpt, a_co_rpt = _reconstruct(idata_rpt, Mz, Xz, anchor_aw, anchor_co)
    log_pred_txn, m_aw_txn, m_co_txn, b_txn, a_aw_txn, a_co_txn = _reconstruct(idata_txn, Mz, Xz, anchor_aw, anchor_co)

    pred_rpt = np.exp(log_pred_rpt)
    pred_txn = np.exp(log_pred_txn)

    co_incr_rpt = (pred_rpt - np.exp(log_pred_rpt - m_co_rpt)).sum()
    aw_incr_rpt = (pred_rpt - np.exp(log_pred_rpt - m_aw_rpt)).sum()
    co_incr_txn = (pred_txn - np.exp(log_pred_txn - m_co_txn)).sum()
    aw_incr_txn = (pred_txn - np.exp(log_pred_txn - m_aw_txn)).sum()

    avg_rpt = rpt_clean.mean(); avg_txn = txn.mean()
    co_incr_rev = co_incr_txn * avg_rpt + avg_txn * co_incr_rpt
    aw_incr_rev = aw_incr_txn * avg_rpt + avg_txn * aw_incr_rpt

    spend_aw = M[:,0].sum(); spend_co = M[:,1].sum()
    roi_aw = aw_incr_rev / spend_aw if spend_aw > 0 else 0
    roi_co = co_incr_rev / spend_co if spend_co > 0 else 0

    mape_rpt = np.nanmean(np.abs((rpt_clean[n_tr:] - pred_rpt[n_tr:]) /
                                  np.where(rpt_clean[n_tr:]==0, np.nan, rpt_clean[n_tr:]))) * 100
    mape_txn = np.nanmean(np.abs((txn[n_tr:] - pred_txn[n_tr:]) /
                                  np.where(txn[n_tr:]==0, np.nan, txn[n_tr:]))) * 100

    co_lift_rpt = (np.exp(float(b_rpt[1])) - 1) * 100
    co_lift_txn = (np.exp(float(b_txn[1])) - 1) * 100
    aw_lift_rpt = (np.exp(float(b_rpt[0])) - 1) * 100
    aw_lift_txn = (np.exp(float(b_txn[0])) - 1) * 100

    return {
        "brand": brand,
        "roi_aw": roi_aw, "roi_co": roi_co,
        "aw_incr_rev": aw_incr_rev, "co_incr_rev": co_incr_rev,
        "spend_aw": spend_aw, "spend_co": spend_co,
        "co_lift_rpt": co_lift_rpt, "co_lift_txn": co_lift_txn,
        "aw_lift_rpt": aw_lift_rpt, "aw_lift_txn": aw_lift_txn,
        "mape_rpt": round(mape_rpt, 1), "mape_txn": round(mape_txn, 1),
        "avg_rpt": avg_rpt, "avg_txn": avg_txn,
        "pred_rpt": pred_rpt, "pred_txn": pred_txn,
        "actual_rpt": rpt_clean, "actual_txn": txn,
        "m_aw_rpt": m_aw_rpt, "m_co_rpt": m_co_rpt,
        "m_aw_txn": m_aw_txn, "m_co_txn": m_co_txn,
        "log_pred_rpt": log_pred_rpt, "log_pred_txn": log_pred_txn,
        "adstock_aw_rpt": a_aw_rpt, "adstock_co_rpt": a_co_rpt,
        "adstock_aw_txn": a_aw_txn, "adstock_co_txn": a_co_txn,
        "anchor_aw": anchor_aw, "anchor_co": anchor_co,
        "div_rpt": int(idata_rpt.sample_stats.diverging.sum()),
        "div_txn": int(idata_txn.sample_stats.diverging.sum()),
        "idata_rpt": idata_rpt,
        "idata_txn": idata_txn,
    }

print("✓ V5e model function defined (sat_half fixed at data anchor)")

In [0]:
# ── Fit V5e for both brands ──
results = {}
for brand in ["TRR", "PDM"]:
    print(f"\n{'='*65}")
    print(f"  Fitting V5e: {brand}")
    print(f"{'='*65}")
    results[brand] = fit_brand_v5e(brand)
    r = results[brand]
    print(f"\n  Rev/Txn model MAPE: {r['mape_rpt']}%  Txn model MAPE: {r['mape_txn']}%")
    print(f"  Divergences: RPT={r['div_rpt']}, TXN={r['div_txn']}")
    print(f"\n  {'Channel':<15} {'Rev/Txn lift':>12} {'Volume lift':>12} {'Combined ROI':>13} {'Spend':>12}")
    print(f"  {'-'*65}")
    print(f"  {'Awareness':<15} {r['aw_lift_rpt']:>+11.1f}% {r['aw_lift_txn']:>+11.1f}% {r['roi_aw']:>12.2f}× ${r['spend_aw']:>10,.0f}")
    print(f"  {'Consideration':<15} {r['co_lift_rpt']:>+11.1f}% {r['co_lift_txn']:>+11.1f}% {r['roi_co']:>12.2f}× ${r['spend_co']:>10,.0f}")

In [0]:
# ── beta_media identifiability audit ──
# Prior: HalfNormal(0.5) → sd ≈ 0.301
PRIOR_SD = 0.5 * np.sqrt(1 - 2/np.pi)

rows = []
for brand in ["TRR", "PDM"]:
    r = results[brand]
    for target, key in [("RPT", "idata_rpt"), ("TXN", "idata_txn")]:
        bm = r[key].posterior["beta_media"]
        summary = az.summary(r[key], var_names=["beta_media"])
        for ch_idx, ch_name in enumerate(["Awareness", "Consideration"]):
            post = bm.isel(beta_media_dim_0=ch_idx).values.flatten()
            post_sd = post.std()
            ratio = post_sd / PRIOR_SD
            rows.append({
                "brand": brand, "target": target, "channel": ch_name,
                "post_mean": round(post.mean(), 3),
                "post_sd": round(post_sd, 3),
                "sd_ratio": round(ratio, 2),
                "rhat": round(summary.iloc[ch_idx]["r_hat"], 3),
                "ess": int(summary.iloc[ch_idx]["ess_bulk"]),
                "identified": "✓" if ratio < 0.5 else ("~" if ratio < 0.8 else "✗"),
            })

audit = pd.DataFrame(rows)
print(audit.to_string(index=False))
print(f"\nRule: sd_ratio < 0.5 = well identified (✓), 0.5–0.8 = moderate (~), ≥0.8 = weak (✗)")

In [0]:
# ── ROI credible intervals (posterior draws through the full ROI pipeline) ──

def _incr_draws(idata, Mz, Xz, h_aw, h_co, n_draws=1000):
    """Push each posterior draw through reconstruction → incremental revenue.
    Returns per-draw arrays of aw_incr and co_incr (in target units)."""
    po = idata.posterior
    if hasattr(po, "to_dataset"):     # DataTree in arviz ≥0.18
        po = po.to_dataset()
    po = po.stack(sample=("chain", "draw"))
    total = po.sizes["sample"]
    idx = np.random.default_rng(42).choice(total, size=min(n_draws, total), replace=False)

    a_aw = po["adstock_aw"].values[idx]
    a_co = po["adstock_co"].values[idx]
    b    = po["beta_media"].values[:, idx]
    g    = po["beta_ctrl"].values[:, idx]
    ic   = po["intercept"].values[idx]

    aw_incr = np.zeros(len(idx))
    co_incr = np.zeros(len(idx))
    for i in range(len(idx)):
        aw_full = adstock_np(Mz[:, 0], float(a_aw[i]))
        co_full = adstock_np(Mz[:, 1], float(a_co[i]))
        aw_sat = aw_full / (aw_full + h_aw)
        co_sat = co_full / (co_full + h_co)
        m_aw = b[0, i] * aw_sat
        m_co = b[1, i] * co_sat
        log_pred = ic[i] + Xz @ g[:, i] + m_aw + m_co
        pred = np.exp(log_pred)
        aw_incr[i] = (pred - np.exp(log_pred - m_aw)).sum()
        co_incr[i] = (pred - np.exp(log_pred - m_co)).sum()
    return aw_incr, co_incr

rows = []
for brand in ["TRR", "PDM"]:
    r = results[brand]
    M = d[[f"{brand}_Awareness_spend", f"{brand}_Consideration_spend"]].values.astype(float)
    X = d[CONTROLS].values.astype(float)
    n_tr = int(len(d) * TRAIN_FRAC)
    m_s = M[:n_tr].max(0); m_s[m_s == 0] = 1.0
    Mz = np.nan_to_num(M / m_s)
    x_mu, x_sd = X[:n_tr].mean(0), X[:n_tr].std(0); x_sd[x_sd == 0] = 1.0
    Xz = np.nan_to_num((X - x_mu) / x_sd)

    h_aw = r["anchor_aw"]; h_co = r["anchor_co"]
    avg_rpt = r["avg_rpt"]; avg_txn = r["avg_txn"]
    spend_aw = r["spend_aw"]; spend_co = r["spend_co"]

    aw_incr_rpt_d, co_incr_rpt_d = _incr_draws(r["idata_rpt"], Mz, Xz, h_aw, h_co)
    aw_incr_txn_d, co_incr_txn_d = _incr_draws(r["idata_txn"], Mz, Xz, h_aw, h_co)

    # Combined incremental revenue per draw (pair RPT and TXN draws index-wise)
    aw_rev_d = aw_incr_txn_d * avg_rpt + avg_txn * aw_incr_rpt_d
    co_rev_d = co_incr_txn_d * avg_rpt + avg_txn * co_incr_rpt_d

    roi_aw_d = aw_rev_d / spend_aw
    roi_co_d = co_rev_d / spend_co

    for ch, draws, spend in [("Awareness", roi_aw_d, spend_aw),
                              ("Consideration", roi_co_d, spend_co)]:
        q05, q50, q95 = np.quantile(draws, [0.05, 0.5, 0.95])
        p_gt1 = (draws > 1).mean()
        rows.append({
            "brand": brand, "channel": ch,
            "roi_p05": round(q05, 2), "roi_p50": round(q50, 2), "roi_p95": round(q95, 2),
            "P(ROI>1)": round(p_gt1, 2),
            "spend": f"${spend:,.0f}",
        })

ci_table = pd.DataFrame(rows)
print(ci_table.to_string(index=False))
print("""
Reportability guide:
  P(ROI>1) ≥ 0.95  → confident, report point estimate
  P(ROI>1) 0.80–0.95 → directional, report with CI
  P(ROI>1) < 0.80  → too uncertain, report range only or flag
""")

In [0]:
# ── Final ROI Summary ──
import matplotlib.pyplot as plt

# Cineplex palette
CPX_NAVY = "#002B5C"; CPX_RED = "#E31837"; CPX_GOLD = "#F5A800"
CPX_LGREY = "#F2F2F2"; CPX_DGREY = "#4A4A4A"; CPX_MGREY = "#8A8A8A"; WHITE = "#FFFFFF"
FUNNEL_COLORS = {"Awareness": CPX_NAVY, "Consideration": CPX_RED}

plt.rcParams.update({"figure.dpi": 150, "axes.titlesize": 13, "axes.titleweight": "bold",
                     "axes.labelsize": 11, "font.weight": "bold", "legend.frameon": False,
                     "figure.facecolor": WHITE, "axes.facecolor": WHITE})

print("\n" + "="*70)
print("  V5e FINAL ROI SUMMARY")
print("="*70)
print(f"\n{'Brand':<6} {'Channel':<15} {'ROI':>6} {'Incr Rev':>12} {'Spend':>12} {'Rev/Txn lift':>13} {'Vol lift':>9}")
print("-"*75)
for b in ["TRR", "PDM"]:
    r = results[b]
    print(f"{b:<6} {'Awareness':<15} {r['roi_aw']:>5.2f}× ${r['aw_incr_rev']:>10,.0f} ${r['spend_aw']:>10,.0f} {r['aw_lift_rpt']:>+12.1f}% {r['aw_lift_txn']:>+8.1f}%")
    print(f"{b:<6} {'Consideration':<15} {r['roi_co']:>5.2f}× ${r['co_incr_rev']:>10,.0f} ${r['spend_co']:>10,.0f} {r['co_lift_rpt']:>+12.1f}% {r['co_lift_txn']:>+8.1f}%")
    print()

# ── ROI Bar Chart ──
fig, axes = plt.subplots(1, 2, figsize=(14, 6))
for ax, b in zip(axes, ["TRR", "PDM"]):
    r = results[b]
    channels = ["Awareness", "Consideration"]
    rois = [r["roi_aw"], r["roi_co"]]
    colors = [FUNNEL_COLORS[c] for c in channels]
    bars = ax.bar(channels, rois, color=colors, width=0.5)
    ax.axhline(1.0, color=CPX_MGREY, ls="--", lw=1.2, label="Break-even")
    for bar, val in zip(bars, rois):
        ax.text(bar.get_x() + bar.get_width()/2, val + 0.1,
                f"{val:.2f}×", ha="center", fontsize=12, fontweight="bold", color=CPX_DGREY)
    ax.set_title(f"{b} — V5e Combined ROI")
    ax.set_ylabel("ROI (×)")
    ax.set_ylim(0, max(rois) * 1.3)
    ax.legend()
plt.tight_layout()
plt.show()

In [0]:
# ── Channel Mechanism: Value vs Volume decomposition ──
from matplotlib.patches import Patch

fig, axes = plt.subplots(1, 2, figsize=(14, 6))

for ax, b in zip(axes, ["TRR", "PDM"]):
    r = results[b]
    # Stacked bar: Value lift vs Volume lift
    channels = ["Awareness", "Consideration"]
    value_lifts = [r["aw_lift_rpt"], r["co_lift_rpt"]]
    volume_lifts = [r["aw_lift_txn"], r["co_lift_txn"]]
    x = np.arange(len(channels))
    w = 0.35
    
    ax.bar(x - w/2, value_lifts, w, color=CPX_GOLD, label="Rev/Txn lift (value)")
    ax.bar(x + w/2, volume_lifts, w, color=CPX_NAVY, label="Transaction lift (volume)")
    
    ax.set_xticks(x)
    ax.set_xticklabels(channels)
    ax.set_ylabel("% lift at saturation")
    ax.set_title(f"{b} — Value vs Volume Mechanism")
    ax.legend()
    ax.axhline(0, color=CPX_DGREY, lw=0.8)
    
    # Annotate
    for i, (v, vol) in enumerate(zip(value_lifts, volume_lifts)):
        ax.text(i - w/2, v + 0.3, f"+{v:.1f}%", ha="center", fontsize=9, fontweight="bold", color=CPX_GOLD)
        ax.text(i + w/2, vol + 0.3, f"+{vol:.1f}%", ha="center", fontsize=9, fontweight="bold", color=CPX_NAVY)

plt.tight_layout()
plt.show()

print("\nKey insight:")
print("  Awareness drives BOTH value and volume roughly equally")
print("  Consideration drives primarily VALUE (spend-per-visit) with modest volume")
print("  This is why modeling total revenue conflates and undercredits Consideration")

In [0]:
# ── Predicted vs Actual: Rev/Txn and Transactions ──
dates = d["week_date"]
n_tr = int(len(d) * TRAIN_FRAC)

fig, axes = plt.subplots(2, 2, figsize=(16, 10))

for col, b in enumerate(["TRR", "PDM"]):
    r = results[b]
    
    # Top row: Rev/Txn
    ax = axes[0, col]
    ax.plot(dates, r["actual_rpt"], color=CPX_DGREY, lw=0.8, alpha=0.7, label="Actual")
    ax.plot(dates, r["pred_rpt"], color=CPX_RED, lw=1.5, label="Predicted")
    ax.axvline(dates.iloc[n_tr], ls="--", color=CPX_GOLD, lw=1.2)
    ax.set_title(f"{b} — Rev/Transaction (MAPE {r['mape_rpt']}%)")
    ax.set_ylabel("$/Transaction")
    ax.legend(loc="upper right", fontsize=8)
    
    # Bottom row: Transactions
    ax = axes[1, col]
    ax.plot(dates, r["actual_txn"], color=CPX_DGREY, lw=0.8, alpha=0.7, label="Actual")
    ax.plot(dates, r["pred_txn"], color=CPX_NAVY, lw=1.5, label="Predicted")
    ax.axvline(dates.iloc[n_tr], ls="--", color=CPX_GOLD, lw=1.2)
    ax.set_title(f"{b} — Transactions (MAPE {r['mape_txn']}%)")
    ax.set_ylabel("Weekly Txns")
    ax.legend(loc="upper right", fontsize=8)

plt.tight_layout()
plt.show()

In [0]:
from sklearn.metrics import r2_score, mean_absolute_percentage_error
n_tr = int(len(d) * TRAIN_FRAC)

rows = []
for b in ["TRR", "PDM"]:
    r = results[b]
    for target, act_key, pred_key in [("Rev/Txn", "actual_rpt", "pred_rpt"), ("Txns", "actual_txn", "pred_txn")]:
        act, pred = r[act_key], r[pred_key]
        for split, s, e in [("Train", 0, n_tr), ("Val", n_tr, len(d)), ("Full", 0, len(d))]:
            a, p = act[s:e], pred[s:e]
            mask = a > 0
            rows.append({"Brand": b, "Target": target, "Split": split,
                         "R²": round(r2_score(a[mask], p[mask]), 3),
                         "MAPE": f"{mean_absolute_percentage_error(a[mask], p[mask])*100:.1f}%",
                         "N_weeks": int(mask.sum())})

df_metrics = pd.DataFrame(rows)
display(df_metrics)

In [0]:
# ━━ SAVE V5e RESULTS & DECOMPOSITION ARTIFACTS ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
# Persists model outputs so downstream viz cells load from disk
# and produce identical numbers regardless of MCMC re-runs.
# Run ONCE after model training; downstream cells load from these files.
import json
%pip install netCDF4 --q
os.makedirs(OUTDIR, exist_ok=True)
ARTIFACT_PATH = f"{OUTDIR}/lbe_v5e"

# ── 1. Summary JSON (high-level ROI metrics) ───────────────────────────────
output = {}
for b in ["TRR", "PDM"]:
    r = results[b]
    output[b] = {
        "model": "V5e_dual_target",
        "roi_awareness": round(r["roi_aw"], 3),
        "roi_consideration": round(r["roi_co"], 3),
        "awareness_incr_revenue": round(r["aw_incr_rev"], 0),
        "consideration_incr_revenue": round(r["co_incr_rev"], 0),
        "awareness_spend": round(r["spend_aw"], 0),
        "consideration_spend": round(r["spend_co"], 0),
        "consideration_rev_txn_lift_pct": round(r["co_lift_rpt"], 1),
        "consideration_volume_lift_pct": round(r["co_lift_txn"], 1),
        "awareness_rev_txn_lift_pct": round(r["aw_lift_rpt"], 1),
        "awareness_volume_lift_pct": round(r["aw_lift_txn"], 1),
        "mape_rev_txn": r["mape_rpt"],
        "mape_transactions": r["mape_txn"],
        "avg_rev_per_txn": round(r["avg_rpt"], 2),
        "avg_weekly_txn": round(r["avg_txn"], 0),
    }

with open(f"{ARTIFACT_PATH}_results.json", "w") as f:
    json.dump(output, f, indent=2)
print(f"✓ lbe_v5e_results.json")

# ── 2. Adstock alphas, sat anchors & scalar model params ───────────────────
artifacts = {}
for brand in ["TRR", "PDM"]:
    r = results[brand]
    artifacts[brand] = {
        "adstock_aw_rpt": float(r["adstock_aw_rpt"]),
        "adstock_co_rpt": float(r["adstock_co_rpt"]),
        "adstock_aw_txn": float(r["adstock_aw_txn"]),
        "adstock_co_txn": float(r["adstock_co_txn"]),
        "anchor_aw": float(r["anchor_aw"]),
        "anchor_co": float(r["anchor_co"]),
        "roi_aw": float(r["roi_aw"]),
        "roi_co": float(r["roi_co"]),
        "spend_aw": float(r["spend_aw"]),
        "spend_co": float(r["spend_co"]),
        "aw_incr_rev": float(r["aw_incr_rev"]),
        "co_incr_rev": float(r["co_incr_rev"]),
        "mape_rpt": float(r["mape_rpt"]),
        "mape_txn": float(r["mape_txn"]),
    }

with open(f"{ARTIFACT_PATH}_artifacts.json", "w") as f:
    json.dump(artifacts, f, indent=2)
print(f"✓ lbe_v5e_artifacts.json")

# ── 3. Full reconstruction arrays per brand ───────────────────────────────
for brand in ["TRR", "PDM"]:
    r = results[brand]
    pd.DataFrame({
        "pred_rpt": r["pred_rpt"],
        "pred_txn": r["pred_txn"],
        "log_pred_rpt": r["log_pred_rpt"],
        "log_pred_txn": r["log_pred_txn"],
        "m_aw_rpt": r["m_aw_rpt"],
        "m_co_rpt": r["m_co_rpt"],
        "m_aw_txn": r["m_aw_txn"],
        "m_co_txn": r["m_co_txn"],
    }).to_parquet(f"{ARTIFACT_PATH}_{brand.lower()}_arrays.parquet", index=False)
    print(f"✓ lbe_v5e_{brand.lower()}_arrays.parquet")


# ── 4. Posterior InferenceData (netCDF) — needed for posterior-based scenarios ──
# HDF5 can't write directly to Volumes (locking issues) — write local first,
# then copy bytes directly with plain Python I/O (avoids Hadoop checksum issues
# from dbutils.fs.cp / shutil.copy on Volume-backed paths).
_local_tmp = "/local_disk0/tmp_idata"
os.makedirs(_local_tmp, exist_ok=True)

for brand in ["TRR", "PDM"]:
    r = results[brand]
    for tgt in ["rpt", "txn"]:
        local_path = f"{_local_tmp}/{brand.lower()}_idata_{tgt}.nc"
        r[f"idata_{tgt}"].to_netcdf(local_path)
        with open(local_path, "rb") as _src, open(f"{ARTIFACT_PATH}_{brand.lower()}_idata_{tgt}.nc", "wb") as _dst:
            _dst.write(_src.read())
    print(f"✓ lbe_v5e_{brand.lower()}_idata_rpt.nc / idata_txn.nc")

print(f"\n✓ All artifacts saved to {OUTDIR}/")
print(f"  Downstream cells load from these files for stable outputs.")

In [0]:
# ── Validation Diagnostics: R², MAPE & Annual Bias ──────────────────────────
# Self-contained: loads data + artifacts from disk
import numpy as np, pandas as pd, json
from sklearn.metrics import r2_score, mean_absolute_percentage_error

# Load source data
d = pd.read_csv(PATH)
d["week_date"] = pd.to_datetime(d["week_date"])
d = d.dropna(subset=["week_date"]).sort_values("week_date").reset_index(drop=True)
d["TRR_Rev"] = d[["Revenue_TRR_HPL", "Revenue_TRR_LPL", "Revenue_TRR_NL"]].sum(axis=1)
d["PDM_Rev"] = d[["Revenue_PDM_NL", "Revenue_PDM_OL"]].sum(axis=1)
d["TRR_Txns"] = d[["TRR_HPL_Txns", "TRR_LPL_Txns", "TRR_NL_Txns"]].sum(axis=1)
d["PDM_Txns"] = d[["PDM_NL_Txns", "PDM_OL_Txns"]].sum(axis=1)

# Load predictions from saved artifacts
ARTIFACT_PATH = f"{OUTDIR}/lbe_v5e"
results = {}
for brand in ["TRR", "PDM"]:
    arr = pd.read_parquet(f"{ARTIFACT_PATH}_{brand.lower()}_arrays.parquet")
    txn = d[f"{brand}_Txns"].values.astype(float)
    rev = d[f"{brand}_Rev"].values.astype(float)
    rpt = np.where(txn > 0, rev / txn, np.nan)
    rpt_clean = np.where(~np.isnan(rpt), rpt, np.nanmedian(rpt))
    results[brand] = {
        "pred_rpt": arr["pred_rpt"].values,
        "pred_txn": arr["pred_txn"].values,
        "actual_rpt": rpt_clean,
        "actual_txn": txn,
    }

n = len(d); n_tr = int(n * TRAIN_FRAC)
dates = d["week_date"]
years = dates.dt.year.values

print("═" * 70)
print("  V5e VALIDATION DIAGNOSTICS")
print("═" * 70)

# ── 1. Validation R² and MAPE ────────────────────────────────────────────────
print(f"\n{'Brand':<6} {'Sub-Model':<16} {'Val MAPE':>9} {'Val R²':>8} {'Train R²':>9}")
print("-" * 55)
for b in ["TRR", "PDM"]:
    r = results[b]
    # Rev/Txn
    valid_rpt = r["actual_rpt"][n_tr:] > 0
    mape_rpt = np.nanmean(np.abs((r["actual_rpt"][n_tr:][valid_rpt] - r["pred_rpt"][n_tr:][valid_rpt]) / r["actual_rpt"][n_tr:][valid_rpt])) * 100
    r2_val_rpt = r2_score(r["actual_rpt"][n_tr:], r["pred_rpt"][n_tr:])
    r2_trn_rpt = r2_score(r["actual_rpt"][:n_tr], r["pred_rpt"][:n_tr])
    # Transactions
    valid_txn = r["actual_txn"][n_tr:] > 0
    mape_txn = np.nanmean(np.abs((r["actual_txn"][n_tr:][valid_txn] - r["pred_txn"][n_tr:][valid_txn]) / r["actual_txn"][n_tr:][valid_txn])) * 100
    r2_val_txn = r2_score(r["actual_txn"][n_tr:], r["pred_txn"][n_tr:])
    r2_trn_txn = r2_score(r["actual_txn"][:n_tr], r["pred_txn"][:n_tr])
    print(f"{b:<6} {'Rev/Txn':<16} {mape_rpt:>8.1f}% {r2_val_rpt:>8.3f} {r2_trn_rpt:>9.3f}")
    print(f"{b:<6} {'Transactions':<16} {mape_txn:>8.1f}% {r2_val_txn:>8.3f} {r2_trn_txn:>9.3f}")

# ── 2. Annual Bias (predicted total / actual total − 1) ─────────────────────
print(f"\n{'─' * 70}")
print(f"  ANNUAL BIAS  (positive = model over-predicts)")
print(f"{'─' * 70}")
print(f"\n{'Brand':<6} {'Sub-Model':<16} {'Year':>5} {'Actual':>12} {'Predicted':>12} {'Bias':>8}")
print("-" * 65)
for b in ["TRR", "PDM"]:
    r = results[b]
    for label, actual, pred in [("Rev/Txn", r["actual_rpt"], r["pred_rpt"]),
                                 ("Transactions", r["actual_txn"], r["pred_txn"])]:
        for yr in sorted(set(years)):
            mask = years == yr
            act_sum = actual[mask].sum()
            pred_sum = pred[mask].sum()
            bias_pct = (pred_sum / act_sum - 1) * 100 if act_sum != 0 else np.nan
            print(f"{b:<6} {label:<16} {yr:>5} {act_sum:>12,.0f} {pred_sum:>12,.0f} {bias_pct:>+7.1f}%")
        print()

print("Note: Annual bias < ±5% indicates well-calibrated model at yearly level.")

In [0]:
# ── Seasonal Flag Attribution on Attendance (Transactions) ──────────────────
# Recovers control coefficients from saved arrays (log_pred - media = ctrl + intercept)
import matplotlib.pyplot as plt, numpy as np, pandas as pd

CPX_NAVY = "#002B5C"; CPX_RED = "#E31837"; CPX_GOLD = "#F5A800"
CPX_DGREY = "#4A4A4A"; CPX_MGREY = "#8A8A8A"; WHITE = "#FFFFFF"
plt.rcParams.update({"figure.dpi": 150, "axes.titlesize": 13, "axes.titleweight": "bold",
                     "axes.labelsize": 11, "font.weight": "bold", "legend.frameon": False,
                     "figure.facecolor": WHITE, "axes.facecolor": WHITE})

CONTROLS = ["national_mean_temp", "national_rain_mm", "headline_cpi", "is_new_year_wk",
            "is_march_break", "is_pre_xmas", "is_summer_peak", "is_back_to_school",
            "is_spring_lull", "is_long_weekend"]
SEASON_FLAGS = ["is_new_year_wk", "is_march_break", "is_pre_xmas",
                "is_summer_peak", "is_back_to_school", "is_spring_lull", "is_long_weekend"]
FLAG_LABELS = ["New Year", "March Break", "Pre-Christmas",
               "Summer Peak", "Back to School", "Spring Lull", "Long Weekend"]

n_tr = int(len(d) * TRAIN_FRAC)
if d["is_long_weekend"].dtype == object:
    d["is_long_weekend"] = (d["is_long_weekend"].astype(str) == "Y").astype(int)
X = d[CONTROLS].values.astype(float)
x_mu, x_sd = X[:n_tr].mean(0), X[:n_tr].std(0); x_sd[x_sd == 0] = 1.0
Xz = (X - x_mu) / x_sd

ARTIFACT_PATH = f"{OUTDIR}/lbe_v5e"

rows = []
for brand in ["TRR", "PDM"]:
    # Load full arrays parquet (has log_pred_txn, m_aw_txn, m_co_txn)
    arr = pd.read_parquet(f"{ARTIFACT_PATH}_{brand.lower()}_arrays.parquet")
    # ctrl + intercept = log_pred_txn - media components
    ctrl_plus_intercept = arr["log_pred_txn"].values - arr["m_aw_txn"].values - arr["m_co_txn"].values
    # Recover gamma via OLS: ctrl_plus_intercept = intercept + Xz @ gamma
    Xz_with_const = np.column_stack([np.ones(len(Xz)), Xz])
    coefs, _, _, _ = np.linalg.lstsq(Xz_with_const, ctrl_plus_intercept, rcond=None)
    gamma = coefs[1:]  # drop intercept, shape (n_controls,)

    for flag, label in zip(SEASON_FLAGS, FLAG_LABELS):
        idx = CONTROLS.index(flag)
        # When flag=1: standardized value = (1 - mu) / sd
        xz_active = (1.0 - x_mu[idx]) / x_sd[idx]
        log_lift = gamma[idx] * xz_active
        pct_lift = (np.exp(log_lift) - 1) * 100

        # Total attributed txns = sum over weeks where flag=1
        flag_weeks = d[flag].values == 1
        attr_txn = (np.exp(gamma[idx] * Xz[:, idx]) - 1) * results[brand]["pred_txn"]
        total_attr = attr_txn[flag_weeks].sum()

        rows.append({"Brand": brand, "Season": label,
                     "% Lift (when active)": round(pct_lift, 1),
                     "Weeks Active": int(flag_weeks.sum()),
                     "Attributed Txns": f"{total_attr:+,.0f}"})

df_season = pd.DataFrame(rows)
print(df_season.to_string(index=False))

# ── Bar chart: % lift per flag, side-by-side TRR vs PDM ──
fig, ax = plt.subplots(figsize=(14, 6))
x_pos = np.arange(len(FLAG_LABELS))
w = 0.35

trr_lifts = [r["% Lift (when active)"] for r in rows if r["Brand"] == "TRR"]
pdm_lifts = [r["% Lift (when active)"] for r in rows if r["Brand"] == "PDM"]

bars1 = ax.bar(x_pos - w/2, trr_lifts, w, color=CPX_NAVY, label="TRR")
bars2 = ax.bar(x_pos + w/2, pdm_lifts, w, color=CPX_RED, label="PDM")

ax.set_xticks(x_pos)
ax.set_xticklabels(FLAG_LABELS, rotation=30, ha="right")
ax.set_ylabel("% Lift in Attendance (Transactions)")
ax.set_title("Seasonal Flag Attribution \u2014 Impact on Attendance")
ax.axhline(0, color=CPX_DGREY, lw=0.8)
ax.legend()

for bar in bars1 + bars2:
    val = bar.get_height()
    ax.text(bar.get_x() + bar.get_width()/2, val + (0.3 if val >= 0 else -1.5),
            f"{val:+.1f}%", ha="center", fontsize=8, fontweight="bold")

plt.tight_layout()
plt.show()

## Findings & Caveats

### What V5e Tells Us
1. **Consideration ROI is ~3×** (not 1× as V3 suggested) — it drives spend-per-visit, not traffic
2. **Awareness ROI is ~5×** (TRR) / ~2.4× (PDM) — drives both traffic AND spend, consistent with V3
3. The Rev/Txn model has excellent predictive accuracy (5–7% MAPE), more stable than the volume model
4. Combined media drives ~12% of revenue through two distinct mechanisms

### Business Implication
- **Awareness** = "get people in the door" (volume play)
- **Consideration** = "get them to spend more once there" (upsell/premium play)
- Both are well above breakeven → justify continued investment
- Consideration is particularly efficient for PDM where spend-per-visit is the binding constraint

### Caveats
- Transaction volume model has higher MAPE (26% TRR) — noisy target, many external drivers
- The dual-target decomposition assumes independence between value and volume channels
- 170 weeks nationally — direction is solid, point estimates have wide CIs
- Seasonal confounds partially removed by controls but not perfectly

### Model Progression
| Version | Target | Co ROI | Key Insight |
|---|---|---|---|
| V3 | log(Revenue) | 1.15× | Signals cancel in combined target |
| V5c | log(Revenue) + spike | 0.86× | Spike not the issue |
| V5d | log(Revenue) Weibull 16wk | 0.75× | Lag shape not the issue |
| **V5e** | **log(Rev/Txn) + log(Txn)** | **3.26×** | **Correct decomposition** |

# PLOTS

In [0]:
CH = {
    "TRR": {
        "Awareness":     ["Awareness_TRR_HPL", "Awareness_TRR_LPL", "Awareness_TRR_NL"],
        "Consideration": ["Consideration_TRR_HPL", "Consideration_TRR_LPL", "Consideration_TRR_NL"],
    },
    "PDM": {
        "Awareness":     ["Awareness_PDM_NL", "Awareness_PDM_OL"],
        "Consideration": ["Consideration_PDM_NL", "Consideration_PDM_OL"],
    },
}
for b in CH:
    for stage, cols in CH[b].items():
        cols = [c for c in cols if c in d.columns and d[c].sum() > 0]
        d[f"{b}_{stage}_spend"] = d[cols].sum(axis=1)

In [0]:
# ── Load posterior InferenceData (needed for posterior-based scenario cells) ──
_local_tmp = "/local_disk0/tmp_idata"
os.makedirs(_local_tmp, exist_ok=True)

for brand in ["TRR", "PDM"]:
    for tgt in ["rpt", "txn"]:
        local_path = f"{_local_tmp}/{brand.lower()}_idata_{tgt}.nc"
        with open(f"{ARTIFACT_PATH}_{brand.lower()}_idata_{tgt}.nc", "rb") as _src, open(local_path, "wb") as _dst:
            _dst.write(_src.read())
        results[brand][f"idata_{tgt}"] = az.from_netcdf(local_path)
print("✓ Posterior idata loaded (idata_rpt / idata_txn per brand)")

In [0]:
# ── PLOTS CONFIG — single source of truth for all visualization cells ──
# Loads saved artifacts from cell 10 so downstream cells produce stable numbers
# regardless of MCMC re-runs. This cell is the standalone entry point for cells 18+.
import shutil, os, json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import matplotlib.patheffects as pe
from IPython.display import display as ipy_display

# ── CINEPLEX BRAND COLOURS ────────────────────────────────────────────────
CPX_NAVY  = "#002B5C"
CPX_RED   = "#D5001C"
CPX_GOLD  = "#C4A747"
CPX_LGREY = "#BFBFBF"
CPX_MGREY = "#808080"
CPX_DGREY = "#404040"
WHITE     = "#FFFFFF"

# ── LOAD SAVED ARTIFACTS ─────────────────────────────────────────────────────
ARTIFACT_PATH = f"{OUTDIR}/lbe_v5e"

with open(f"{ARTIFACT_PATH}_artifacts.json", "r") as f:
    _artifacts = json.load(f)

_arrays = {}
for brand in ["TRR", "PDM"]:
    _arrays[brand] = pd.read_parquet(f"{ARTIFACT_PATH}_{brand.lower()}_arrays.parquet")

results = {}
for brand in ["TRR", "PDM"]:
    arr = _arrays[brand]
    art = _artifacts[brand]
    results[brand] = {
        "pred_rpt":      arr["pred_rpt"].values,
        "pred_txn":      arr["pred_txn"].values,
        "log_pred_rpt":  arr["log_pred_rpt"].values,
        "log_pred_txn":  arr["log_pred_txn"].values,
        "m_aw_rpt":      arr["m_aw_rpt"].values,
        "m_co_rpt":      arr["m_co_rpt"].values,
        "m_aw_txn":      arr["m_aw_txn"].values,
        "m_co_txn":      arr["m_co_txn"].values,
        "adstock_aw_rpt": art["adstock_aw_rpt"],
        "adstock_co_rpt": art["adstock_co_rpt"],
        "adstock_aw_txn": art["adstock_aw_txn"],
        "adstock_co_txn": art["adstock_co_txn"],
        "anchor_aw":      art["anchor_aw"],
        "anchor_co":      art["anchor_co"],
        "roi_aw":         art["roi_aw"],
        "roi_co":         art["roi_co"],
        "spend_aw":       art["spend_aw"],
        "spend_co":       art["spend_co"],
        "aw_incr_rev":    art["aw_incr_rev"],
        "co_incr_rev":    art["co_incr_rev"],
    }

print(f"✓ Artifacts loaded from {ARTIFACT_PATH}_*.parquet / .json")
print(f"  Arrays pinned — downstream cells will not shift on re-runs.")

# ── Forecast & output ────────────────────────────────────────────────────────────
FORECAST_2026 = 148_546_568          # full-year 2026 revenue forecast (H1 actual + H2 projected)
FORECAST_2027 = 143_049_761

# Legacy int-year list (kept for backward compat with existing downstream cells)
PLOT_YEARS = [2023, 2024, 2025]

# Canonical ordered list of periods for new/updated plots
PLOT_PERIODS      = ["2023", "2024", "2025", "2026 H1", "2026 Forecast"]
ACTUAL_PERIODS    = ["2023", "2024", "2025", "2026 H1"]   # periods with spend + ROI
FORECAST_PERIODS  = ["2026 Forecast"]                     # revenue-only reference bars

OUTPUT_DIR    = OUTDIR

def get_figsize(ratio="16:9", base=16):
    w, h = [int(x) for x in ratio.split(":")]
    return (base, base * h / w)

def save_fig(fig, fname):
    os.makedirs(OUTPUT_DIR, exist_ok=True)
    fig.savefig(os.path.join(OUTPUT_DIR, fname), dpi=150, bbox_inches="tight", facecolor=WHITE)
    print(f"✓ Saved: {fname}")

# ── Shared dataframe (used by every plot cell) ───────────────────────────────────
df_all = d.copy()
df_all["year"]              = df_all["week_date"].dt.year
df_all["LBE_Revenue"]       = df_all["TRR_Rev"] + df_all["PDM_Rev"]
df_all["Awareness_TRR"]     = df_all["TRR_Awareness_spend"]
df_all["Consideration_TRR"] = df_all["TRR_Consideration_spend"]
df_all["Awareness_PDM"]     = df_all["PDM_Awareness_spend"]
df_all["Consideration_PDM"] = df_all["PDM_Consideration_spend"]
df_all["aw_spend_proxy"]    = df_all["TRR_Awareness_spend"] + df_all["PDM_Awareness_spend"]
df_all["co_spend_proxy"]    = df_all["TRR_Consideration_spend"] + df_all["PDM_Consideration_spend"]
df_all["Awareness"]         = df_all["aw_spend_proxy"]
df_all["Consideration"]     = df_all["co_spend_proxy"]

# ── V5e ROI — cross-brand average and per-brand ───────────────────────────────────
# ROI is model-scalar; same value applies to every actual-period bar.
_roi_aw_avg = (results["TRR"]["roi_aw"] + results["PDM"]["roi_aw"]) / 2
_roi_co_avg = (results["TRR"]["roi_co"] + results["PDM"]["roi_co"]) / 2

roi_yr = {
    "Awareness":     {p: _roi_aw_avg for p in ACTUAL_PERIODS},
    "Consideration": {p: _roi_co_avg for p in ACTUAL_PERIODS},
}
brand_roi = {
    b: {"Awareness": results[b]["roi_aw"], "Consideration": results[b]["roi_co"]}
    for b in ["TRR", "PDM"]
}

# ── Pre-aggregated annual spend (actuals only — 2023, 2024, 2025, 2026 H1) ───────
_spend_actual = (
    df_all[df_all["year"].isin([2023, 2024, 2025, 2026])]
    .groupby("year")[["aw_spend_proxy", "co_spend_proxy"]]
    .sum()
    .rename(columns={"aw_spend_proxy": "Awareness", "co_spend_proxy": "Consideration"})
)
_spend_actual.index = [str(y) if y != 2026 else "2026 H1" for y in _spend_actual.index]
_spend_actual["Total"] = _spend_actual["Awareness"] + _spend_actual["Consideration"]
annual_spend   = _spend_actual.reindex(ACTUAL_PERIODS)
annual_spend_M = annual_spend / 1e6

# ── Annual revenue (actuals + 2026 full-year forecast as separate bar) ───────────
_rev_actual = (
    df_all[df_all["year"].isin([2023, 2024, 2025, 2026])]
    .groupby("year")["LBE_Revenue"].sum()
    .rename("Revenue")
    .to_frame()
)
_rev_actual.index = [str(y) if y != 2026 else "2026 H1" for y in _rev_actual.index]
annual_revenue = _rev_actual.reindex(ACTUAL_PERIODS)
annual_revenue.loc["2026 Forecast"] = FORECAST_2026
annual_revenue_M = annual_revenue / 1e6

print(f"✓ Plot config ready  |  FORECAST_2026 ${FORECAST_2026:,.0f}")
print(f"  TRR — Awareness {results['TRR']['roi_aw']:.2f}×  Consideration {results['TRR']['roi_co']:.2f}×")
print(f"  PDM — Awareness {results['PDM']['roi_aw']:.2f}×  Consideration {results['PDM']['roi_co']:.2f}×")
print(f"  Periods: {PLOT_PERIODS}")

In [0]:
# ── LBE V5e ANNUAL REVENUE DECOMPOSITION BY FUNNEL + 2026 H1 + 2026/27 FORECAST ──
# Depends on: Plots Config cell (FORECAST_2026, df_all, CPX_* colors, get_figsize)

# ── V5e counterfactual decomp: weekly incremental revenue per funnel ──────────
def _weekly_incr(r):
    """Remove media log-contribution from both RPT and TXN sub-models."""
    wk_aw = (
        (r["pred_rpt"] - np.exp(r["log_pred_rpt"] - r["m_aw_rpt"])) * r["pred_txn"]
        + r["pred_rpt"] * (r["pred_txn"] - np.exp(r["log_pred_txn"] - r["m_aw_txn"]))
    )
    wk_co = (
        (r["pred_rpt"] - np.exp(r["log_pred_rpt"] - r["m_co_rpt"])) * r["pred_txn"]
        + r["pred_rpt"] * (r["pred_txn"] - np.exp(r["log_pred_txn"] - r["m_co_txn"]))
    )
    return wk_aw, wk_co

wk_aw_total = np.zeros(len(d))
wk_co_total = np.zeros(len(d))
for brand in ["TRR", "PDM"]:
    aw, co = _weekly_incr(results[brand])
    wk_aw_total += aw
    wk_co_total += co

decomp_df = d[["week_date"]].copy()
decomp_df["year"]                = decomp_df["week_date"].dt.year
decomp_df["total_rev"]           = d["TRR_Rev"] + d["PDM_Rev"]
decomp_df["media_Awareness"]     = wk_aw_total
decomp_df["media_Consideration"] = wk_co_total

# ── Aggregate Annual by Funnel (actuals: 2023–2025 + 2026 H1) ─────────────────
agg_cols = {"total_rev": "sum", "media_Awareness": "sum", "media_Consideration": "sum"}
annual = decomp_df[decomp_df["year"].isin([2023, 2024, 2025, 2026])].groupby("year").agg(agg_cols).round(0)
years_avail_int    = sorted(annual.index.tolist())
period_labels      = [str(y) if y != 2026 else "2026 H1" for y in years_avail_int]
actual_vals        = annual.loc[years_avail_int, "total_rev"].values
awareness_vals     = annual.loc[years_avail_int, "media_Awareness"].values
consideration_vals = annual.loc[years_avail_int, "media_Consideration"].values
baseline_vals      = actual_vals - awareness_vals - consideration_vals

# ── Reference: 2025 mix (used for forecast decomposition — not last-actual) ───
ref_idx        = years_avail_int.index(2025)
ref_total      = actual_vals[ref_idx]
ref_media      = awareness_vals[ref_idx] + consideration_vals[ref_idx]
aw_share       = awareness_vals[ref_idx] / ref_media
csd_share      = consideration_vals[ref_idx] / ref_media
media_ratio    = ref_media / ref_total

# ── Assemble bar lists: actuals + forecasts ───────────────────────────────────
all_labels    = list(period_labels)
all_actual    = list(actual_vals)
all_baseline  = list(baseline_vals)
all_awareness = list(awareness_vals)
all_consid    = list(consideration_vals)
is_forecast   = [False] * len(period_labels)

if FORECAST_2026 is not None:
    media_2026 = FORECAST_2026 * media_ratio
    all_labels.append("2026 Forecast")
    all_actual.append(FORECAST_2026)
    all_baseline.append(FORECAST_2026 - media_2026)
    all_awareness.append(media_2026 * aw_share)
    all_consid.append(media_2026 * csd_share)
    is_forecast.append(True)
    show_forecast = True

    if FORECAST_2027 is not None:
        media_2027 = FORECAST_2027 * media_ratio
        all_labels.append("2027 Forecast")
        all_actual.append(FORECAST_2027)
        all_baseline.append(FORECAST_2027 - media_2027)
        all_awareness.append(media_2027 * aw_share)
        all_consid.append(media_2027 * csd_share)
        is_forecast.append(True)
else:
    show_forecast = False

n_actual = sum(1 for f in is_forecast if not f)

# ── Helper ────────────────────────────────────────────────────────────────────
def bar_label(ax, x_pos, bottom, height, pct, color, min_height=0.5):
    if abs(height) / 1e6 < min_height:
        return
    light_colors = {CPX_LGREY, CPX_GOLD}
    txt_color = "black" if color in light_colors else "white"
    ax.text(
        x_pos, (bottom + height / 2) / 1e6,
        f"{pct:.0f}%",
        ha="center", va="center",
        fontsize=12, fontweight="bold", color=txt_color,
        path_effects=[pe.withStroke(
            linewidth=2,
            foreground="white" if txt_color == "black" else "black"
        )]
    )

# ── Plot ──────────────────────────────────────────────────────────────────────
fig, ax = plt.subplots(figsize=get_figsize("16:9"))
fig.patch.set_facecolor(WHITE)

x     = np.arange(len(all_labels))
w     = 0.5
alpha = [0.5 if f else 0.9 for f in is_forecast]

for i, label in enumerate(all_labels):
    a      = alpha[i]
    b_base = all_baseline[i]
    b_aw   = all_awareness[i]
    b_csd  = all_consid[i]
    total  = all_actual[i]

    ax.bar(x[i], b_base / 1e6,
           width=w, color=CPX_LGREY, alpha=a,
           edgecolor=WHITE, linewidth=0.5,
           label="Baseline (non-media)" if i == 0 else "_nolegend_")
    ax.bar(x[i], b_aw / 1e6,
           width=w, bottom=b_base / 1e6,
           color=CPX_NAVY, alpha=a,
           edgecolor=WHITE, linewidth=0.5,
           label="Awareness" if i == 0 else "_nolegend_")
    ax.bar(x[i], b_csd / 1e6,
           width=w, bottom=(b_base + b_aw) / 1e6,
           color=CPX_RED, alpha=a,
           edgecolor=WHITE, linewidth=0.5,
           label="Consideration" if i == 0 else "_nolegend_")

    bar_label(ax, x[i], 0,             b_base, b_base / total * 100, CPX_LGREY)
    bar_label(ax, x[i], b_base,        b_aw,   b_aw   / total * 100, CPX_NAVY, 1.5)
    bar_label(ax, x[i], b_base + b_aw, b_csd,  b_csd  / total * 100, CPX_RED,  1.5)

    top_label = f"${total/1e6:.2f}M" + (" (F)" if is_forecast[i] else "")
    ax.text(x[i], total / 1e6 + 0.3, top_label,
            ha="center", va="bottom",
            fontsize=12, fontweight="bold",
            color=CPX_RED if is_forecast[i] else CPX_DGREY)

# Actual scatter dots
ax.scatter(x[:n_actual], np.array(all_actual[:n_actual]) / 1e6,
           color="black", zorder=5, s=80,
           marker="D", label="Actual Revenue")

# Forecast marker and divider
if show_forecast:
    fc_x = x[n_actual:]
    fc_y = [all_actual[i] / 1e6 for i in range(n_actual, len(all_labels))]
    ax.scatter(fc_x, fc_y,
               color=CPX_RED, zorder=5, s=80,
               marker="D", label="Forecast")
    ax.axvline(x=n_actual - 0.5, color=CPX_MGREY, linewidth=1, linestyle="--")
    ax.text(n_actual - 0.47, ax.get_ylim()[0] + 2, "Forecast →",
            fontsize=12, color=CPX_MGREY, fontstyle="italic")

ax.set_xticks(x)
ax.set_xticklabels(all_labels, fontsize=14, fontweight="bold", rotation=15, ha="right")
plt.setp(ax.get_yticklabels(), fontsize=16, fontweight="bold")
ax.set_ylabel("Annual Revenue ($M)", fontsize=16, fontweight="bold")
ax.set_xlabel("Period", fontsize=16, fontweight="bold")
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda v, _: f"${v:.0f}M"))
title_suffix = f"2023–{all_labels[-1]}" if show_forecast else "2023–2026 H1"
ax.set_title(f"LBE Annual Revenue Decomposition by Funnel ({title_suffix})", loc="center",
             fontsize=14, fontweight="bold", color=CPX_NAVY)
ax.legend(loc="upper center", bbox_to_anchor=(0.5, -0.12), ncol=3, fontsize=16, frameon=False)
ax.grid(False)
ax.spines["top"].set_visible(False); ax.spines["right"].set_visible(False)

plt.tight_layout()
save_fig(fig, "lbe_v5e_annual_rev_decomp.png")
display(fig)
plt.close(fig)

# ── Summary Table ─────────────────────────────────────────────────────────────
tbl_rows = []
for i, lbl in enumerate(all_labels):
    total = all_actual[i]
    media_total = all_awareness[i] + all_consid[i]
    tbl_rows.append({
        "Period": f"{lbl}{' (F)' if is_forecast[i] else ''}",
        "Baseline (M)": round(all_baseline[i] / 1e6, 2),
        "Awareness (M)": round(all_awareness[i] / 1e6, 2),
        "Consideration (M)": round(all_consid[i] / 1e6, 2),
        "Media Total (M)": round(media_total / 1e6, 2),
        "Total (M)": round(total / 1e6, 2),
        "Media %": f"{media_total / total * 100:.1f}%",
    })
print(f"\n{'═'*70}\n  LBE Annual Media Contribution Summary\n{'═'*70}")
display(pd.DataFrame(tbl_rows))

In [0]:
# ━━ LBE FUNNEL EFFICIENCY — MECHANISM-SPECIFIC ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
# Awareness → Volume Play (Cost per Incremental Visit)
# Consideration → Value Play (ROAS via spend-per-visit uplift)

# Parallel lists: int for filtering, string for display
period_years_int = [2023, 2024, 2025, 2026]
period_labels    = ACTUAL_PERIODS   # ["2023", "2024", "2025", "2026 H1"]

# ── Decompose by PRIMARY MECHANISM per funnel ─────────────────────────────────
incr_txn_aw     = np.zeros(len(d))
rev_via_txn_aw  = np.zeros(len(d))
rev_via_rpt_co  = np.zeros(len(d))

for brand in ["TRR", "PDM"]:
    r = results[brand]
    txn_uplift_aw      = r["pred_txn"] - np.exp(r["log_pred_txn"] - r["m_aw_txn"])
    incr_txn_aw       += txn_uplift_aw
    rev_via_txn_aw    += r["pred_rpt"] * txn_uplift_aw

    rpt_uplift_co      = r["pred_rpt"] - np.exp(r["log_pred_rpt"] - r["m_co_rpt"])
    rev_via_rpt_co    += rpt_uplift_co * r["pred_txn"]

mech_df = d[["week_date"]].copy()
mech_df["year"]            = mech_df["week_date"].dt.year
mech_df["incr_txn_aw"]     = incr_txn_aw
mech_df["rev_via_txn_aw"]  = rev_via_txn_aw
mech_df["rev_via_rpt_co"]  = rev_via_rpt_co

# ── Build efficiency rows ─────────────────────────────────────────────────────
eff_rows = []
for yr_int, yr_label in zip(period_years_int, period_labels):
    yr_spend  = df_all[df_all["year"] == yr_int]
    yr_mech   = mech_df[mech_df["year"] == yr_int]
    yr_decomp = decomp_df[decomp_df["year"] == yr_int]

    # Awareness
    aw_spend     = yr_spend["Awareness"].sum()
    aw_rev_total = yr_decomp["media_Awareness"].sum()
    aw_txns      = yr_mech["incr_txn_aw"].sum()
    eff_rows.append({
        "Period": yr_label, "Funnel": "Awareness",
        "Spend_M": aw_spend / 1e6,
        "Contribution_M": aw_rev_total / 1e6,
        "Incr_Txn": aw_txns,
        "Txn_per_Dollar": aw_txns / aw_spend if aw_spend > 0 else 0,
        "Cost_per_Visit": aw_spend / aw_txns if aw_txns > 0 else 0,
    })

    # Consideration
    co_spend     = yr_spend["Consideration"].sum()
    co_rev_total = yr_decomp["media_Consideration"].sum()
    co_rev_rpt   = yr_mech["rev_via_rpt_co"].sum()
    eff_rows.append({
        "Period": yr_label, "Funnel": "Consideration",
        "Spend_M": co_spend / 1e6,
        "Contribution_M": co_rev_total / 1e6,
        "Incr_Txn": 0,
        "Txn_per_Dollar": 0,
        "Cost_per_Visit": 0,
        "Rev_per_Dollar_RPT": co_rev_rpt / co_spend if co_spend > 0 else 0,
    })

eff_df = pd.DataFrame(eff_rows)

# ── Funnel colours & styling ──────────────────────────────────────────────────
funnel_colors_eff = {"Awareness": CPX_NAVY, "Consideration": CPX_RED}
funnel_markers    = {"Awareness": "o", "Consideration": "s"}

# ── Figure: 1×2 subplots — reader-friendly metrics ───────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(16, 8), sharey=False)
fig.patch.set_facecolor(WHITE)

for idx, funnel in enumerate(["Awareness", "Consideration"]):
    ax  = axes[idx]
    ax2 = ax.twinx()
    fdata = eff_df[eff_df["Funnel"] == funnel].reset_index(drop=True)
    color = funnel_colors_eff[funnel]
    x = np.arange(len(period_labels))
    w = 0.30

    bars_sp = ax.bar(x - w/2, fdata["Spend_M"], width=w,
                     color=color, alpha=0.30, edgecolor=color, linewidth=1.5,
                     label="Media Spend")
    ct_label = "Revenue from Extra Visits" if funnel == "Awareness" else "Revenue from Higher Spend/Visit"
    bars_ct = ax.bar(x + w/2, fdata["Contribution_M"], width=w,
                     color=color, alpha=0.85, edgecolor=WHITE, linewidth=0.5,
                     label=ct_label)

    for bar in bars_sp:
        h = bar.get_height()
        ax.text(bar.get_x() + bar.get_width()/2, h + 0.03,
                f"${h:.2f}M", ha="center", va="bottom",
                fontsize=11, color=color, fontweight="bold")

    for bar in bars_ct:
        h = bar.get_height()
        ax.text(bar.get_x() + bar.get_width()/2, h + 0.03,
                f"${h:.2f}M", ha="center", va="bottom",
                fontsize=11, color="black", fontweight="bold")

    if funnel == "Awareness":
        eff_vals = fdata["Cost_per_Visit"].values
        eff_label_fmt = lambda v: f"${v:.0f}"
        eff_name = "Cost / Visit"
        ax2_ylabel = "Cost per Incremental Visit ($)"
        ax2.invert_yaxis()
    else:
        eff_vals = fdata["Rev_per_Dollar_RPT"].values
        eff_label_fmt = lambda v: f"{v:.1f}×"
        eff_name = "ROAS (upsell)"
        ax2_ylabel = "ROAS — Return per $1 Spent (×)"

    ax2.plot(x, eff_vals, color="black", linewidth=3, zorder=5,
             marker=funnel_markers[funnel], markersize=13,
             markerfacecolor=WHITE, markeredgecolor="black", markeredgewidth=2.5)

    for i, val in enumerate(eff_vals):
        offset_y = -18 if (funnel == "Awareness") else 14
        ax2.annotate(
            eff_label_fmt(val),
            xy=(x[i], val), xytext=(0, offset_y),
            textcoords="offset points", ha="center",
            fontsize=14, fontweight="bold", color="black",
            path_effects=[pe.withStroke(linewidth=4, foreground=WHITE)]
        )

    ax.set_xticks(x)
    ax.set_xticklabels(period_labels, fontsize=13, fontweight="bold", rotation=10, ha="right")
    ax.set_ylabel("Dollars ($M)", fontsize=13, fontweight="bold")
    ax2.set_ylabel(ax2_ylabel, fontsize=12, fontweight="bold", color="black")

    ax.set_title(funnel, fontsize=14, fontweight="bold", color=color, pad=12)

    ax.set_ylim(0, max(fdata["Contribution_M"].max(), fdata["Spend_M"].max()) * 1.45)
    if funnel == "Awareness":
        pad = max(eff_vals) * 0.3
        ax2.set_ylim(max(eff_vals) + pad, min(eff_vals) - pad)
    else:
        ax2.set_ylim(0, max(eff_vals) * 1.5)
    ax.grid(False)
    ax2.grid(False)
    plt.setp(ax.get_yticklabels(), fontsize=12, fontweight="bold")
    plt.setp(ax2.get_yticklabels(), fontsize=12, fontweight="bold")
    ax.spines["top"].set_visible(False)
    ax2.spines["top"].set_visible(False)

    ax.legend(loc="upper left", fontsize=12, frameon=False)
    ax2.plot([], [], color="black", linewidth=3, marker=funnel_markers[funnel],
             markerfacecolor=WHITE, markeredgecolor="black", markersize=9,
             label=eff_name)
    ax2.legend(loc="upper right", fontsize=12, frameon=False)

    if funnel == "Awareness":
        for i, row in fdata.iterrows():
            top = max(row["Spend_M"], row["Contribution_M"])
            ax.text(x[i], top * 1.18, f"+{row['Incr_Txn']:,.0f} visits",
                    ha="center", fontsize=10, color=CPX_NAVY, fontstyle="italic")

fig.suptitle("LBE Media Efficiency by Funnel Mechanism (2023–2026 H1)",
             fontsize=16, fontweight="bold", color=CPX_NAVY, y=0.99)
plt.tight_layout(rect=[0, 0.02, 1, 0.94])
save_fig(fig, "lbe_funnel_efficiency_by_year.png")
display(fig)
plt.close(fig)

# ── Summary Table — Wide Pivot (Period × Funnel) ─────────────────────────────
metrics = {}
for yr_label in period_labels:
    aw = eff_df[(eff_df["Period"] == yr_label) & (eff_df["Funnel"] == "Awareness")].iloc[0]
    co = eff_df[(eff_df["Period"] == yr_label) & (eff_df["Funnel"] == "Consideration")].iloc[0]
    metrics[(yr_label, "Awareness")] = {
        "Spend_M": aw["Spend_M"],
        "Contribution_M": aw["Contribution_M"],
        "Efficiency": aw["Cost_per_Visit"],
    }
    metrics[(yr_label, "Consideration")] = {
        "Spend_M": co["Spend_M"],
        "Contribution_M": co["Contribution_M"],
        "Efficiency": co["Rev_per_Dollar_RPT"],
    }

periods_cols = period_labels
funnels_cols = ["Awareness", "Consideration"]
multi_cols = pd.MultiIndex.from_product([periods_cols, funnels_cols], names=["Period", "Funnel"])

spend_row   = [metrics[(p, f)]["Spend_M"] for p in periods_cols for f in funnels_cols]
contrib_row = [metrics[(p, f)]["Contribution_M"] for p in periods_cols for f in funnels_cols]
eff_row     = [metrics[(p, f)]["Efficiency"] for p in periods_cols for f in funnels_cols]

# YoY Δ per funnel (period-over-period on efficiency; 2025→2026H1 is partial-year, flag it)
yoy_row = []
for f in funnels_cols:
    prev_val = None
    for p in periods_cols:
        curr_val = metrics[(p, f)]["Efficiency"]
        yoy_row.append(None if prev_val is None else (curr_val - prev_val) / prev_val * 100)
        prev_val = curr_val
yoy_ordered = []
for p_idx, p in enumerate(periods_cols):
    for f_idx, f in enumerate(funnels_cols):
        yoy_ordered.append(yoy_row[f_idx * len(periods_cols) + p_idx])

wide_df = pd.DataFrame(
    [spend_row, contrib_row, eff_row, yoy_ordered],
    index=["Spend ($M)", "Contribution ($M)", "Efficiency", "YoY Eff Δ"],
    columns=multi_cols
)

fmt_df = wide_df.copy().astype(object)
for p in periods_cols:
    for f in funnels_cols:
        fmt_df.loc["Spend ($M)", (p, f)]        = f"{wide_df.loc['Spend ($M)', (p, f)]:.2f}"
        fmt_df.loc["Contribution ($M)", (p, f)] = f"{wide_df.loc['Contribution ($M)', (p, f)]:.2f}"
        eff_val = wide_df.loc["Efficiency", (p, f)]
        if f == "Awareness":
            fmt_df.loc["Efficiency", (p, f)] = f"${eff_val:.2f}"
        else:
            fmt_df.loc["Efficiency", (p, f)] = f"{eff_val:.2f}"
        yoy_val = wide_df.loc["YoY Eff Δ", (p, f)]
        if yoy_val is None or pd.isna(yoy_val):
            fmt_df.loc["YoY Eff Δ", (p, f)] = "—"
        else:
            # Flag the partial-year comparison
            suffix = " (H1 vs full yr)" if p == "2026 H1" else ""
            fmt_df.loc["YoY Eff Δ", (p, f)] = f"{yoy_val:+.1f}%{suffix}"

fmt_df = fmt_df.rename(index={"Efficiency": "Cost/Visit | ROAS"})

def color_yoy(val):
    if val == "—":
        return ""
    try:
        v = float(val.replace("%", "").replace("+", "").split()[0])
        return f"color: {'#d32f2f' if v > 0 else '#2e7d32'}" if "$" not in str(val) else ""
    except:
        return ""

print(f"{'═'*74}")
print(f"  LBE Funnel Efficiency Summary (2023–2026 H1)")
print(f"{'═'*74}")
print(f"  Awareness: Cost per Incremental Visit (lower = better)")
print(f"  Consideration: ROAS via spend-per-visit uplift (higher = better)")
print(f"  Note: 2026 H1 spend/contribution bars reflect partial year (~half of full).")
print(f"{'─'*74}")

flat_df = fmt_df.copy()
flat_df.columns = [f"{p} | {f}" for p, f in flat_df.columns]
flat_df.insert(0, "Metric", flat_df.index)
flat_df = flat_df.reset_index(drop=True)
display(flat_df)

In [0]:
# ━━ LBE FUNNEL ADSTOCK DECAY CURVES — CARRY-OVER & PLANNING LEAD TIME ━━━━━━━━━━━
# Shows how each funnel's media impact decays week-over-week after activation
# Uses fitted alpha values from V5e model (averaged across TRR + PDM)

# ── Extract fitted adstock alphas from results dict ───────────────────────────
# Awareness primary mechanism = volume (TXN model)
# Consideration primary mechanism = value (RPT model)
alpha_aw_txn = np.mean([results[b]["adstock_aw_txn"] for b in ["TRR", "PDM"]])
alpha_aw_rpt = np.mean([results[b]["adstock_aw_rpt"] for b in ["TRR", "PDM"]])
alpha_co_rpt = np.mean([results[b]["adstock_co_rpt"] for b in ["TRR", "PDM"]])
alpha_co_txn = np.mean([results[b]["adstock_co_txn"] for b in ["TRR", "PDM"]])

# Primary mechanism alpha for headline
alpha_aw = alpha_aw_txn  # awareness drives volume
alpha_co = alpha_co_rpt  # consideration drives value

# Derive carry-over window (weeks until <10% remaining)
def weeks_to_threshold(alpha, threshold=0.10):
    return int(np.ceil(np.log(threshold) / np.log(alpha))) if alpha > 0 else 0

aw_carry = weeks_to_threshold(alpha_aw)
co_carry = weeks_to_threshold(alpha_co)

decay_params = {
    "Awareness": {
        "alpha": alpha_aw,
        "color": CPX_NAVY,
        "carryover": f"{aw_carry-1}–{aw_carry} wks",
        "lead": f"{aw_carry-1}–{aw_carry} wks before",
    },
    "Consideration": {
        "alpha": alpha_co,
        "color": CPX_RED,
        "carryover": f"{co_carry-1}–{co_carry} wks",
        "lead": f"{co_carry-1}–{co_carry} wks before",
    },
}

print(f"Fitted adstock alphas (mean of TRR + PDM):")
print(f"  Awareness  → TXN model: {alpha_aw_txn:.3f}  |  RPT model: {alpha_aw_rpt:.3f}  → primary: {alpha_aw:.3f}")
print(f"  Consideration → RPT model: {alpha_co_rpt:.3f}  |  TXN model: {alpha_co_txn:.3f}  → primary: {alpha_co:.3f}")

# ── Plot ───────────────────────────────────────────────────────────────────
weeks = np.arange(0, 9, 0.1)
weeks_int = np.arange(0, 9)

fig, ax = plt.subplots(figsize=(14, 7))
fig.patch.set_facecolor(WHITE)

for funnel, params in decay_params.items():
    alpha = params["alpha"]
    color = params["color"]

    # Decay curve: effect remaining = alpha^t (impulse response)
    decay_smooth = alpha ** weeks * 100
    decay_int    = alpha ** weeks_int * 100

    # Main curve
    ax.plot(weeks, decay_smooth, color=color, linewidth=3, label=funnel, zorder=3)
    ax.scatter(weeks_int, decay_int, color=color, s=60, zorder=4,
              edgecolors=WHITE, linewidths=1.2)

    # Shade the carry-over window (weeks until <10%)
    shade_end = min(weeks_to_threshold(alpha, 0.10), 8)
    shade_x = weeks[weeks <= shade_end]
    shade_y = alpha ** shade_x * 100
    ax.fill_between(shade_x, 0, shade_y, color=color, alpha=0.08)

    # Annotate the carry-over window on the curve (stagger positions to avoid overlap)
    if funnel == "Awareness":
        annot_x = 3.5
        annot_y = alpha ** annot_x * 100
        txt_offset = (1.2, 12)
    else:
        annot_x = 0.8
        annot_y = alpha ** annot_x * 100
        txt_offset = (1.5, 18)

    ax.annotate(
        f"α = {alpha:.2f}\n{params['carryover']} carry-over\nPlan: {params['lead']}",
        xy=(annot_x, annot_y),
        xytext=(annot_x + txt_offset[0], annot_y + txt_offset[1]),
        fontsize=10, fontweight="bold", color=color,
        arrowprops=dict(arrowstyle="->", color=color, lw=1.5),
        bbox=dict(boxstyle="round,pad=0.4", facecolor=WHITE, edgecolor=color, alpha=0.9),
    )

# Reference lines
ax.axhline(50, color=CPX_LGREY, linewidth=1, linestyle="--", zorder=1)
ax.text(8.1, 50, "50%", fontsize=10, fontweight="bold", color=CPX_DGREY, va="center")
ax.axhline(10, color=CPX_LGREY, linewidth=1, linestyle=":", zorder=1)
ax.text(8.1, 10, "10%", fontsize=10, fontweight="bold", color=CPX_DGREY, va="center")

# Formatting
ax.set_xlabel("Weeks After Media Activation", fontsize=14, fontweight="bold", labelpad=10)
ax.set_ylabel("Remaining Effect (%)", fontsize=14, fontweight="bold", labelpad=10)
ax.set_xlim(-0.2, 8.5)
ax.set_ylim(0, 105)
ax.set_xticks(range(9))
ax.set_xticklabels([f"Wk {i}" for i in range(9)], fontsize=12, fontweight="bold")
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda v, _: f"{v:.0f}%"))
plt.setp(ax.get_yticklabels(), fontsize=12, fontweight="bold")
ax.grid(False)
ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)
ax.legend(loc="upper right", fontsize=12, frameon=True, framealpha=0.9,
          edgecolor=CPX_LGREY)

ax.set_title("LBE — Adstock Decay by Funnel (Fitted from V5e Model)",
             fontsize=16, fontweight="bold", color=CPX_NAVY, pad=15)

plt.tight_layout()
save_fig(fig, "lbe_funnel_adstock_decay_curves.png")
display(fig)
plt.close(fig)

# ── Summary Table ────────────────────────────────────────────────────────
print(f"\n{'═'*74}")
print(f"  LBE Adstock Decay Parameters (V5e Fitted)")
print(f"{'═'*74}")
adstock_tbl = pd.DataFrame([
    {"Funnel": "Awareness", "Primary α": f"{alpha_aw:.3f}",
     "Mechanism": "Volume (Txn)", "Half-life (wks)": f"{np.log(0.5)/np.log(alpha_aw):.1f}",
     "90% Decay (wks)": f"{weeks_to_threshold(alpha_aw)}",
     "Planning Lead": decay_params['Awareness']['lead']},
    {"Funnel": "Consideration", "Primary α": f"{alpha_co:.3f}",
     "Mechanism": "Value (RPT)", "Half-life (wks)": f"{np.log(0.5)/np.log(alpha_co):.1f}",
     "90% Decay (wks)": f"{weeks_to_threshold(alpha_co)}",
     "Planning Lead": decay_params['Consideration']['lead']},
])
display(adstock_tbl)

In [0]:
# ── LBE Annual Spend Overview ─────────────────────────────────────────────────
# Uses df_all and annual_spend_M from Plots Config cell.

# ── Table display ─────────────────────────────────────────────────────────────
tbl = annual_spend_M.copy()
tbl.index = tbl.index.astype(str)
tbl.loc["3-Year Total"] = tbl.sum()

ipy_display(
    tbl.style
    .format("${:.2f}M")
    .set_caption("LBE Annual Media Spend by Funnel (2023–2025)")
    .set_table_styles([
        {"selector": "caption",
         "props": [("font-size", "13px"), ("font-weight", "bold"),
                   ("color", CPX_NAVY), ("text-align", "left")]},
        {"selector": "th",
         "props": [("background-color", CPX_NAVY), ("color", WHITE),
                   ("font-weight", "bold"), ("text-align", "center")]},
        {"selector": "tr:last-child",
         "props": [("font-weight", "bold"), ("background-color", "#F2F2F2")]},
    ])
)
# ── Chart ─────────────────────────────────────────────────────────────────────
years   = ACTUAL_PERIODS   # ["2023", "2024", "2025", "2026 H1"]
aw_vals = annual_spend_M.loc[years, "Awareness"].values
co_vals = annual_spend_M.loc[years, "Consideration"].values
x = np.arange(len(years))
width = 0.55

fig, ax = plt.subplots(figsize=(11, 9))
fig.patch.set_facecolor(WHITE)

b1 = ax.bar(x, aw_vals, width, label="Awareness", color=CPX_NAVY)
b2 = ax.bar(x, co_vals, width, label="Consideration", color=CPX_RED, bottom=aw_vals)

def label_segments(bars, bottoms):
    for i, bar in enumerate(bars):
        h = bar.get_height()
        b = bottoms[i]
        if h >= 0.1:
            ax.text(
                bar.get_x() + bar.get_width() / 2,
                b + h / 2,
                f"${h:.2f}M",
                ha="center", va="center",
                fontsize=14, fontweight="bold", color=WHITE
            )

label_segments(b1, [0] * len(years))
label_segments(b2, aw_vals)

totals = aw_vals + co_vals
for i, total in enumerate(totals):
    ax.text(i, total + 0.06, f"${total:.2f}M",
            ha="center", va="bottom",
            fontsize=14, fontweight="bold", color=CPX_NAVY)

ax.set_xticks(x)
ax.set_xticklabels(years, fontsize=13, fontweight="bold", rotation=10, ha="right")
plt.setp(ax.get_yticklabels(), fontsize=12, fontweight="bold")
ax.set_ylabel("Annual Media Spend ($M)", fontsize=14, fontweight="bold")
ax.set_xlabel("Period", fontsize=14, fontweight="bold")
ax.set_ylim(0, totals.max() * 1.15)
ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda v, _: f"${v:.1f}M"))
ax.set_title("LBE MMM — Annual Media Spend by Funnel\n2023–2026 H1",
             fontsize=14, fontweight="bold", color=CPX_NAVY)
ax.legend(loc="upper left", frameon=False, fontsize=12)
ax.grid(False)
ax.spines["top"].set_visible(False); ax.spines["right"].set_visible(False)

plt.tight_layout()
save_fig(fig, "lbe_b0_annual_spend_by_funnel.png")
display(fig); plt.close(fig)

In [0]:
# ── LBE SPEND BY CHANNEL — STACKED BY FUNNEL — ONE PLOT PER YEAR ─────────────
channel_funnel_cols = {
    ("TRR", "Awareness")    : "Awareness_TRR",
    ("TRR", "Consideration"): "Consideration_TRR",
    ("PDM", "Awareness")    : "Awareness_PDM",
    ("PDM", "Consideration"): "Consideration_PDM",
}

funnels  = ["Awareness", "Consideration"]
channels = ["TRR", "PDM"]
funnel_colors = {"Awareness": CPX_NAVY, "Consideration": CPX_RED}

# Parallel lists: int for filtering df_all, string for display/filename
period_years_int = [2023, 2024, 2025, 2026]
period_labels    = ACTUAL_PERIODS   # ["2023", "2024", "2025", "2026 H1"]

rows = []
for (channel, funnel), col in channel_funnel_cols.items():
    for yr_int, yr_label in zip(period_years_int, period_labels):
        spend = df_all.loc[df_all["year"] == yr_int, col].sum()
        rows.append({"Channel": channel, "Funnel": funnel, "Period": yr_label, "Spend_M": spend / 1e6})

spend_df = pd.DataFrame(rows)
spend_df = spend_df[spend_df["Spend_M"] > 0.001].copy()

for yr_label in period_labels:
    ydata = spend_df[spend_df["Period"] == yr_label].copy()
    ch_totals = ydata.groupby("Channel")["Spend_M"].sum().sort_values(ascending=False)
    ch_order  = ch_totals.index.tolist()
    x         = np.arange(len(ch_order))

    fig, ax = plt.subplots(figsize=get_figsize("16:9"))
    fig.patch.set_facecolor(WHITE)
    bottoms = np.zeros(len(ch_order))

    for funnel in funnels:
        heights = []
        for ch in ch_order:
            val = ydata.loc[(ydata["Funnel"] == funnel) & (ydata["Channel"] == ch), "Spend_M"]
            heights.append(float(val.values[0]) if len(val) > 0 else 0.0)
        bars = ax.bar(x, heights, bottom=bottoms, label=funnel,
                      color=funnel_colors[funnel], width=0.55, edgecolor=WHITE, linewidth=0.5)
        for i, bar in enumerate(bars):
            h = bar.get_height()
            if h > 0.03:
                ax.text(bar.get_x() + bar.get_width() / 2, bottoms[i] + h / 2,
                        f"${h:.2f}M", ha="center", va="center",
                        fontsize=14, fontweight="bold", color=WHITE)
        bottoms += np.array(heights)

    for i, total in enumerate(bottoms):
        if total > 0:
            ax.text(x[i], total + 0.01, f"${total:.2f}M",
                    ha="center", va="bottom", fontsize=14, fontweight="bold", color=CPX_NAVY)

    ax.set_xticks(x)
    ax.set_xticklabels(ch_order, rotation=0, ha="center", fontsize=14, fontweight="bold")
    plt.setp(ax.get_yticklabels(), fontsize=12, fontweight="bold")
    ax.set_ylabel("Annual Spend ($M)", fontsize=14, fontweight="bold")
    ax.set_xlabel("Channel", fontsize=14, fontweight="bold")
    ax.set_ylim(0, bottoms.max() * 1.15)
    ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda v, _: f"${v:.2f}M"))
    ax.set_title(f"LBE Annual Media Spend by Channel & Funnel — {yr_label}",
                 fontsize=14, fontweight="bold", color=CPX_NAVY)
    ax.legend(title="Funnel", fontsize=12, title_fontsize=12, loc="upper right", frameon=False)
    ax.grid(False)
    ax.spines["top"].set_visible(False); ax.spines["right"].set_visible(False)
    plt.tight_layout()
    # Safe filename: "2026 H1" → "2026_h1"
    fname_period = yr_label.lower().replace(" ", "_")
    save_fig(fig, f"lbe_spend_channel_funnel_{fname_period}.png")
    display(fig); plt.close(fig)

In [0]:
# ── LBE SPEND BY CHANNEL & FUNNEL — ALL YEARS COMBINED ────────────────────────
import matplotlib.ticker as mticker

channel_funnel_cols = {
    ("TRR", "Awareness")    : "Awareness_TRR",
    ("TRR", "Consideration"): "Consideration_TRR",
    ("PDM", "Awareness")    : "Awareness_PDM",
    ("PDM", "Consideration"): "Consideration_PDM",
}

funnels  = ["Awareness", "Consideration"]
channels = ["TRR", "PDM"]
years    = [2023, 2024, 2025, 2026]
funnel_colors = {"Awareness": CPX_NAVY, "Consideration": CPX_RED}

rows = []
for (channel, funnel), col in channel_funnel_cols.items():
    for year in years:
        spend = df_all.loc[df_all["year"] == year, col].sum()
        rows.append({"Channel": channel, "Funnel": funnel, "Year": year, "Spend_M": spend / 1e6})

spend_df_combined = pd.DataFrame(rows)

n_years    = len(years)
n_channels = len(channels)
bar_width  = 0.25
group_gap  = 0.15

fig, ax = plt.subplots(figsize=(11, 9))
fig.patch.set_facecolor(WHITE)
all_totals = []

for ci, channel in enumerate(channels):
    offset = (ci - (n_channels - 1) / 2) * (bar_width + group_gap)
    x_positions = np.arange(n_years) + offset
    bottoms = np.zeros(n_years)

    for funnel in funnels:
        heights = []
        for year in years:
            val = spend_df_combined.loc[
                (spend_df_combined["Channel"] == channel) &
                (spend_df_combined["Funnel"] == funnel) &
                (spend_df_combined["Year"] == year), "Spend_M"]
            heights.append(float(val.values[0]) if len(val) > 0 else 0.0)
        heights = np.array(heights)
        bars = ax.bar(x_positions, heights, width=bar_width, bottom=bottoms,
                      color=funnel_colors[funnel], edgecolor=WHITE, linewidth=0.5,
                      label=funnel if ci == 0 else "")
        for i, bar in enumerate(bars):
            h = bar.get_height()
            if h > 0.03:
                ax.text(bar.get_x() + bar.get_width() / 2, bottoms[i] + h / 2,
                        f"${h:.2f}M", ha="center", va="center",
                        fontsize=10, fontweight="bold", color=WHITE)
        bottoms += heights

    for i, total in enumerate(bottoms):
        if total > 0:
            ax.text(x_positions[i], total + 0.04, f"${total:.2f}M",
                    ha="center", va="bottom", fontsize=14, fontweight="bold", color=CPX_NAVY)

    all_totals.append((x_positions.copy(), bottoms.copy(), channel))

    for i, xp in enumerate(x_positions):
        ax.text(xp, -0.03, channel, ha="center", va="top",
                fontsize=12, fontweight="bold", color=CPX_DGREY,
                transform=ax.get_xaxis_transform())

max_height = max(t.max() for _, t, _ in all_totals)
label_y = max_height + 0.55
for yi, yr_label in enumerate(ACTUAL_PERIODS):
    ax.text(yi, label_y, yr_label, ha="center", va="bottom",
            fontsize=12, fontweight="bold", color="black")

ax.set_xticks(np.arange(n_years))
ax.set_xticklabels([""] * n_years)
plt.setp(ax.get_yticklabels(), fontsize=12, fontweight="bold")
ax.set_ylabel("Annual Spend ($M)", fontsize=12, fontweight="bold")
ax.set_xlabel("")
ax.set_ylim(0, spend_df_combined.groupby(["Channel", "Year"])["Spend_M"].sum().max() * 1.5)
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda v, _: f"${v:.2f}M"))
ax.set_title("LBE Annual Media Spend by Channel & Funnel (2023–2026 H1)",
             fontsize=14, fontweight="bold", color=CPX_NAVY)
ax.legend(loc="upper right", title="Funnel", fontsize=12, title_fontsize=12, frameon=False)
ax.grid(False)
ax.spines["top"].set_visible(False); ax.spines["right"].set_visible(False)
plt.tight_layout()
save_fig(fig, "lbe_spend_channel_funnel_combined_2026h1.png")
display(fig); plt.close(fig)

# ── TABLE: LBE SPEND BY CHANNEL & FUNNEL ──────────────────────────────────────
tbl = (spend_df_combined
       .pivot_table(index=["Channel", "Funnel"], columns="Year", values="Spend_M")
       .reindex(index=[("TRR", "Awareness"), ("TRR", "Consideration"),
                       ("PDM", "Awareness"), ("PDM", "Consideration")]))

# Channel subtotals
for ch in ["TRR", "PDM"]:
    tbl.loc[(ch, "Total"), :] = tbl.loc[ch].sum()

# Grand total row
tbl.loc[("All", "Total"), :] = tbl.xs("Total", level=1).sum()

# Row total column
tbl["Total"] = tbl.sum(axis=1)

tbl = tbl.round(2).applymap(lambda v: f"${v:,.2f}M")
tbl.columns = [str(c) if c != "Total" else "Total" for c in tbl.columns]
tbl.index.names = ["Channel", "Funnel"]

display(tbl)

In [0]:
# ── TABLE: LBE SPEND BY FUNNEL — YoY (TRR + PDM combined) ─────────────────────
tbl = (spend_df_combined
       .groupby(["Funnel", "Year"])["Spend_M"].sum()
       .unstack("Year")
       .reindex(["Awareness", "Consideration"]))

tbl.loc["Total"] = tbl.sum()
tbl["Total"] = tbl.sum(axis=1)

tbl = tbl.round(2).applymap(lambda v: f"${v:,.2f}M")
tbl.columns = [str(c) if c != "Total" else "Total" for c in tbl.columns]
tbl.index.name = "Funnel"

display(tbl)

In [0]:
# ── TABLE: LBE SPEND BY YEAR & BRAND (Awareness / Consideration / Total) ──────
tbl = (spend_df_combined
       .pivot_table(index=["Year", "Channel"], columns="Funnel", values="Spend_M")
       .reindex(columns=["Awareness", "Consideration"]))
tbl["Total Spend"] = tbl["Awareness"] + tbl["Consideration"]
tbl = tbl.rename(columns={"Awareness": "Awareness Spend", "Consideration": "Consideration Spend"})
tbl.index.names = ["Year", "Brand"]

def fmt_spend(v):
    if pd.isna(v):
        return ""
    return f"${v:.2f}M" if v >= 1 else f"${v*1000:.0f}K"

tbl_fmt = tbl.applymap(fmt_spend)

styled = (
    tbl_fmt.style
    .set_table_styles([
        {"selector": "th", "props": [("background-color", "black"), ("color", WHITE),
                                      ("font-weight", "bold"), ("text-align", "center"),
                                      ("padding", "8px")]},
        {"selector": "td", "props": [("text-align", "center"), ("padding", "8px"),
                                      ("font-weight", "bold")]},
    ])
    .set_properties(subset=["Awareness Spend"], **{"background-color": CPX_NAVY, "color": WHITE})
    .set_properties(subset=["Consideration Spend"], **{"background-color": CPX_RED, "color": WHITE})
)
display(styled)

In [0]:
# ── TIER-LEVEL SPEND & MEDIA-ATTRIBUTED REVENUE ───────────────────────────────
import matplotlib.patches as mpatches

FUNNELS      = ["Awareness", "Consideration"]
BRANDS       = ["TRR", "PDM"]
FUNNEL_COLOR = {"Awareness": CPX_NAVY, "Consideration": CPX_RED}
brand_tiers  = {"TRR": ["HPL", "LPL", "NL"], "PDM": ["OL", "NL"]}

# Parallel lists: int for filtering, string for display
YEARS        = [2023, 2024, 2025, 2026]
YEAR_LABELS  = {2023: "2023", 2024: "2024", 2025: "2025", 2026: "2026 H1"}
YEAR_COLORS  = {2023: CPX_DGREY, 2024: CPX_NAVY, 2025: CPX_RED, 2026: CPX_GOLD}

# ── Step 1: Annual spend per (brand, funnel, tier, year) — raw tier columns ──
eff_spend = {}
for brand, tiers in brand_tiers.items():
    for funnel in FUNNELS:
        for tier in tiers:
            col = f"{funnel}_{brand}_{tier}"
            if col not in d.columns:
                continue
            for yr in YEARS:
                eff_spend[(brand, funnel, tier, yr)] = float(
                    d.loc[d["week_date"].dt.year == yr, col].sum()
                )

# ── Step 2: Year-specific counterfactual incremental revenue ─────────────────
def _brand_weekly_incr(r):
    """Remove media log-contribution from both RPT and TXN sub-models."""
    wk_aw = (
        (r["pred_rpt"] - np.exp(r["log_pred_rpt"] - r["m_aw_rpt"])) * r["pred_txn"]
        + r["pred_rpt"] * (r["pred_txn"] - np.exp(r["log_pred_txn"] - r["m_aw_txn"]))
    )
    wk_co = (
        (r["pred_rpt"] - np.exp(r["log_pred_rpt"] - r["m_co_rpt"])) * r["pred_txn"]
        + r["pred_rpt"] * (r["pred_txn"] - np.exp(r["log_pred_txn"] - r["m_co_txn"]))
    )
    return wk_aw, wk_co

brand_funnel_yr_incr = {}
for brand in BRANDS:
    wk_aw, wk_co = _brand_weekly_incr(results[brand])
    decomp_tmp = d[["week_date"]].copy()
    decomp_tmp["year"] = decomp_tmp["week_date"].dt.year
    decomp_tmp["incr_Awareness"] = wk_aw
    decomp_tmp["incr_Consideration"] = wk_co
    for yr in YEARS:
        mask = decomp_tmp["year"] == yr
        brand_funnel_yr_incr[(brand, "Awareness", yr)] = decomp_tmp.loc[mask, "incr_Awareness"].sum()
        brand_funnel_yr_incr[(brand, "Consideration", yr)] = decomp_tmp.loc[mask, "incr_Consideration"].sum()

tier_rev = {}
for brand, tiers in brand_tiers.items():
    for funnel in FUNNELS:
        for yr in YEARS:
            yr_incr = brand_funnel_yr_incr[(brand, funnel, yr)]
            yr_total_eff = max(
                sum(eff_spend.get((brand, funnel, t, yr), 0.0) for t in tiers),
                1e-9
            )
            for tier in tiers:
                w = eff_spend.get((brand, funnel, tier, yr), 0.0) / yr_total_eff
                tier_rev[(brand, funnel, tier, yr)] = yr_incr * w

print("Year-specific counterfactual ROI by brand/funnel:")
for brand in BRANDS:
    for funnel in FUNNELS:
        vals = []
        for yr in YEARS:
            s = sum(eff_spend.get((brand, funnel, t, yr), 0.0) for t in brand_tiers[brand])
            r_val = brand_funnel_yr_incr[(brand, funnel, yr)]
            roi = r_val / s if s > 0 else 0
            vals.append(f"{YEAR_LABELS[yr]}={roi:.2f}x")
        print(f"  {brand} {funnel}: {', '.join(vals)}")

# ── Accessors ($M) ────────────────────────────────────────────────────────────
def sp(brand, funnel, tier, yr):
    return eff_spend.get((brand, funnel, tier, yr), 0.0) / 1e6

def rv(brand, funnel, tier, yr):
    return tier_rev.get((brand, funnel, tier, yr), 0.0) / 1e6

fmt_M = mticker.FuncFormatter(lambda v, _: f"${v:.2f}M")

# Shared x-axis layout: TRR tiers then PDM tiers
all_brand_tiers = [(b, t) for b in BRANDS for t in brand_tiers[b]]
n_bt       = len(all_brand_tiers)
trr_div    = len(brand_tiers["TRR"]) - 0.5
trr_center = (len(brand_tiers["TRR"]) - 1) / 2
pdm_center = len(brand_tiers["TRR"]) + (len(brand_tiers["PDM"]) - 1) / 2

def add_brand_dividers(ax, y_top):
    ax.axvline(trr_div, color="#CCCCCC", lw=1.2, linestyle="--")
    for bx, blabel in [(trr_center, "TRR"), (pdm_center, "PDM")]:
        ax.text(bx, y_top * 0.97, blabel, ha="center", va="top",
                fontsize=11, fontweight="bold", color="black")

def style_ax(ax, y_top):
    ax.set_ylim(0, y_top)
    ax.yaxis.set_major_formatter(fmt_M)
    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)
    plt.setp(ax.get_yticklabels(), fontsize=12, fontweight="bold")
    ax.grid(False)

# ═══════════════════════════════════════════════════════════════════════════════
# CHART 1 — Spend by Tier, both brands, one chart per year, funnel stacked
# ═══════════════════════════════════════════════════════════════════════════════
for yr in YEARS:
    fig, ax = plt.subplots(figsize=(11, 7))
    fig.patch.set_facecolor(WHITE)

    x       = np.arange(n_bt)
    bar_w   = 0.5
    bottoms = np.zeros(n_bt)

    for funnel in FUNNELS:
        heights = np.array([sp(b, funnel, t, yr) for b, t in all_brand_tiers])
        ax.bar(x, heights, bar_w, bottom=bottoms,
               color=FUNNEL_COLOR[funnel], alpha=0.9,
               edgecolor=WHITE, lw=0.4, label=funnel)
        bottoms += heights

    for i, total in enumerate(bottoms):
        if total > 0:
            ax.annotate(f"${total:.2f}M", xy=(x[i], total),
                        xytext=(0, 4), textcoords="offset points",
                        ha="center", fontsize=12, color=CPX_DGREY, fontweight="bold")

    y_top = bottoms.max() * 1.3
    style_ax(ax, y_top)
    add_brand_dividers(ax, y_top)
    ax.set_xticks(x)
    ax.set_xticklabels([t for _, t in all_brand_tiers], fontsize=14, fontweight="bold")
    ax.set_ylabel("Media Spend ($M)", fontsize=14, fontweight="bold")
    ax.set_title(f"LBE Spend by Tier — {YEAR_LABELS[yr]}",
                 fontsize=14, fontweight="bold", color=CPX_NAVY)
    ax.legend(fontsize=12, loc="upper left", frameon=False)
    plt.tight_layout()
    fname_yr = YEAR_LABELS[yr].lower().replace(" ", "_")
    save_fig(fig, f"lbe_tier_spend_{fname_yr}.png")
    display(fig)
    plt.close(fig)

    tbl_rows = []
    for b, t in all_brand_tiers:
        row = {"Brand": b, "Tier": t}
        for funnel in FUNNELS:
            row[f"{funnel} ($M)"] = round(sp(b, funnel, t, yr), 3)
        row["Total ($M)"] = round(sum(sp(b, f, t, yr) for f in FUNNELS), 3)
        tbl_rows.append(row)
    print(f"\n{'═'*55}\n  Spend by Tier — {YEAR_LABELS[yr]}\n{'═'*55}")
    ipy_display(pd.DataFrame(tbl_rows))

# ═══════════════════════════════════════════════════════════════════════════════
# CHART 2 — Spend vs Revenue by Tier, one chart per year
# ═══════════════════════════════════════════════════════════════════════════════
bar_w = 0.3
gap   = 0.08
x_sp  = np.arange(n_bt) - (bar_w + gap) / 2
x_rv  = np.arange(n_bt) + (bar_w + gap) / 2

for yr in YEARS:
    fig, ax = plt.subplots(figsize=(11, 7))
    fig.patch.set_facecolor(WHITE)

    bottoms_s = np.zeros(n_bt)
    bottoms_r = np.zeros(n_bt)

    for funnel in FUNNELS:
        sp_vals = np.array([sp(b, funnel, t, yr) for b, t in all_brand_tiers])
        rv_vals = np.array([rv(b, funnel, t, yr) for b, t in all_brand_tiers])

        ax.bar(x_sp, sp_vals, bar_w, bottom=bottoms_s,
               color=FUNNEL_COLOR[funnel], alpha=0.9,
               edgecolor=WHITE, lw=0.4)
        ax.bar(x_rv, rv_vals, bar_w, bottom=bottoms_r,
               color=FUNNEL_COLOR[funnel], alpha=0.5,
               edgecolor=WHITE, lw=0.4, hatch="///")

        bottoms_s += sp_vals
        bottoms_r += rv_vals

    for i in range(n_bt):
        if bottoms_s[i] > 0:
            ax.annotate(f"${bottoms_s[i]:.2f}M",
                        xy=(x_sp[i], bottoms_s[i]),
                        xytext=(0, 4), textcoords="offset points",
                        ha="center", fontsize=12, color=CPX_DGREY, fontweight="bold")
        if bottoms_r[i] > 0:
            ax.annotate(f"${bottoms_r[i]:.2f}M",
                        xy=(x_rv[i], bottoms_r[i]),
                        xytext=(0, 4), textcoords="offset points",
                        ha="center", fontsize=12, color=CPX_DGREY, fontweight="bold")

    y_top = max(bottoms_s.max(), bottoms_r.max(), 0.01) * 1.3
    style_ax(ax, y_top)
    add_brand_dividers(ax, y_top)

    ax.set_xticks(np.arange(n_bt))
    ax.set_xticklabels([t for _, t in all_brand_tiers], fontsize=14, fontweight="bold")
    ax.set_ylabel("Media Spend ($M)", fontsize=14, fontweight="bold")

    funnel_handles = [mpatches.Patch(color=FUNNEL_COLOR[f], label=f) for f in FUNNELS]
    type_handles   = [
        mpatches.Patch(facecolor="#999999", alpha=0.9, label="Spend"),
        mpatches.Patch(facecolor="#999999", alpha=0.5, hatch="///", label="Rev Contrib"),
    ]
    ax.legend(handles=funnel_handles + type_handles,
              fontsize=12, loc="upper left", frameon=False)

    ax.set_title(f"LBE Spend vs Media-Attributed Revenue by Tier — {YEAR_LABELS[yr]}",
                 fontsize=14, fontweight="bold", color=CPX_NAVY)
    plt.tight_layout()
    fname_yr = YEAR_LABELS[yr].lower().replace(" ", "_")
    save_fig(fig, f"lbe_tier_spend_vs_rev_{fname_yr}.png")
    display(fig)
    plt.close(fig)

    tbl_rows = []
    for b, t in all_brand_tiers:
        row = {"Brand": b, "Tier": t}
        for funnel in FUNNELS:
            row[f"{funnel} Spend ($M)"] = round(sp(b, funnel, t, yr), 3)
            row[f"{funnel} Rev ($M)"]   = round(rv(b, funnel, t, yr), 3)
        row["Total Spend ($M)"] = round(sum(sp(b, f, t, yr) for f in FUNNELS), 3)
        row["Total Rev ($M)"]   = round(sum(rv(b, f, t, yr) for f in FUNNELS), 3)
        tbl_rows.append(row)
    rate_row = {"Brand": "ROI", "Tier": ""}
    for funnel in FUNNELS:
        s = sum(sp(b, f, t, yr) for b, t in all_brand_tiers for f in [funnel])
        r = sum(rv(b, f, t, yr) for b, t in all_brand_tiers for f in [funnel])
        rate_row[f"{funnel} Spend ($M)"] = ""
        rate_row[f"{funnel} Rev ($M)"]   = f"{r/s:.2f}×" if s > 0 else "—"
    rate_row["Total Spend ($M)"] = ""
    rate_row["Total Rev ($M)"]   = ""
    tbl_rows.append(rate_row)
    print(f"\n{'═'*55}\n  Spend vs Revenue by Tier — {YEAR_LABELS[yr]}\n{'═'*55}")
    ipy_display(pd.DataFrame(tbl_rows))

# ═══════════════════════════════════════════════════════════════════════════════
# CHART 3 — Spend vs Revenue by Tier, one chart per brand
# ═══════════════════════════════════════════════════════════════════════════════
bar_w    = 0.11
gap      = 0.05
pair_w   = bar_w * 2 + gap
tier_gap = 0.08
year_gap = 0.25

for brand in BRANDS:
    tiers   = brand_tiers[brand]
    n_tiers = len(tiers)
    n_years = len(YEARS)

    year_w = n_tiers * pair_w + (n_tiers - 1) * tier_gap

    fig, ax = plt.subplots(figsize=(max(11, n_years * (year_w + year_gap) * 5 + 2), 7))
    fig.patch.set_facecolor(WHITE)

    year_centers = []

    for yi, yr in enumerate(YEARS):
        base_x = yi * (year_w + year_gap)
        year_centers.append(base_x + year_w / 2)

        for ti, tier in enumerate(tiers):
            x_sp_pos = base_x + ti * (pair_w + tier_gap)
            x_rv_pos = x_sp_pos + bar_w + gap

            bottoms_s = 0.0
            bottoms_r = 0.0
            for funnel in FUNNELS:
                sp_val = sp(brand, funnel, tier, yr)
                rv_val = rv(brand, funnel, tier, yr)

                ax.bar(x_sp_pos, sp_val, bar_w, bottom=bottoms_s,
                       color=FUNNEL_COLOR[funnel], alpha=0.9,
                       edgecolor=WHITE, lw=0.3)
                ax.bar(x_rv_pos, rv_val, bar_w, bottom=bottoms_r,
                       color=FUNNEL_COLOR[funnel], alpha=0.5,
                       edgecolor=WHITE, lw=0.3, hatch="///")

                bottoms_s += sp_val
                bottoms_r += rv_val

            ax.text((x_sp_pos + x_rv_pos) / 2, -0.012, tier,
                    ha="center", va="top", fontsize=14,
                    color=CPX_DGREY, fontweight="bold",
                    transform=ax.get_xaxis_transform())

            if bottoms_s > 0:
                ax.annotate(f"${bottoms_s:.2f}M",
                            xy=(x_sp_pos, bottoms_s),
                            xytext=(0, 3), textcoords="offset points",
                            ha="center", fontsize=14, color=CPX_DGREY, fontweight="bold")
            if bottoms_r > 0:
                ax.annotate(f"${bottoms_r:.2f}M",
                            xy=(x_rv_pos, bottoms_r),
                            xytext=(0, 3), textcoords="offset points",
                            ha="center", fontsize=14, color=CPX_DGREY, fontweight="bold")

        if yi < n_years - 1:
            div_x = base_x + year_w + year_gap / 2
            ax.axvline(div_x, color="#CCCCCC", lw=1.2, linestyle="--")

    ax.set_xticks([])
    all_sp = [sum(sp(brand, f, t, yr) for f in FUNNELS) for t in tiers for yr in YEARS]
    all_rv = [sum(rv(brand, f, t, yr) for f in FUNNELS) for t in tiers for yr in YEARS]
    y_top = max(max(all_sp), max(all_rv), 0.01) * 1.35
    ax.set_ylim(0, y_top)

    for yi, yr in enumerate(YEARS):
        ax.text(year_centers[yi], y_top * 0.82, YEAR_LABELS[yr],
                ha="center", va="top", fontsize=14,
                fontweight="bold", color=YEAR_COLORS[yr])

    ax.set_xlim(-0.2, n_years * (year_w + year_gap) - year_gap + 0.2)
    ax.yaxis.set_major_formatter(fmt_M)
    ax.set_ylabel("Spend & Revenue ($M)", fontsize=14, fontweight="bold", color=CPX_DGREY)
    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)
    plt.setp(ax.get_yticklabels(), fontsize=14, fontweight="bold")
    ax.set_axisbelow(True)
    ax.grid(axis="y", alpha=0.2, color="#CCCCCC")

    funnel_handles = [mpatches.Patch(color=FUNNEL_COLOR[f], label=f) for f in FUNNELS]
    year_handles   = [mpatches.Patch(facecolor=YEAR_COLORS[yr], label=YEAR_LABELS[yr]) for yr in YEARS]
    type_handles   = [
        mpatches.Patch(facecolor="#999999", alpha=0.9, label="Spend"),
        mpatches.Patch(facecolor="#999999", alpha=0.5, hatch="///", label="Rev Contrib"),
    ]
    ax.legend(handles=funnel_handles + year_handles + type_handles,
              fontsize=14, loc="upper right", framealpha=0.9,
              edgecolor="#CCCCCC", ncol=3)

    ax.set_title(f"{brand} — Spend vs Revenue by Year & Tier ($M)",
                 fontsize=14, fontweight="bold", color=CPX_NAVY)
    plt.tight_layout()
    save_fig(fig, f"lbe_tier_{brand.lower()}_spend_vs_rev_by_year.png")
    display(fig)
    plt.close(fig)

    for yr in YEARS:
        tbl_rows = []
        for tier in tiers:
            row = {"Tier": tier}
            for funnel in FUNNELS:
                row[f"{funnel} Spend ($M)"] = round(sp(brand, funnel, tier, yr), 3)
                row[f"{funnel} Rev ($M)"]   = round(rv(brand, funnel, tier, yr), 3)
            row["Total Spend ($M)"] = round(sum(sp(brand, f, tier, yr) for f in FUNNELS), 3)
            row["Total Rev ($M)"]   = round(sum(rv(brand, f, tier, yr) for f in FUNNELS), 3)
            tbl_rows.append(row)
        rate_row = {"Tier": "ROI"}
        for funnel in FUNNELS:
            s = sum(sp(brand, funnel, t, yr) for t in tiers)
            r = sum(rv(brand, funnel, t, yr) for t in tiers)
            rate_row[f"{funnel} Spend ($M)"] = ""
            rate_row[f"{funnel} Rev ($M)"]   = f"{r/s:.2f}×" if s > 0 else "—"
        rate_row["Total Spend ($M)"] = ""
        rate_row["Total Rev ($M)"]   = ""
        tbl_rows.append(rate_row)
        print(f"\n{'═'*55}\n  {brand} — Spend vs Revenue by Tier — {YEAR_LABELS[yr]}\n{'═'*55}")
        ipy_display(pd.DataFrame(tbl_rows))

In [0]:
# ── Per-Year Spend by Brand (TRR vs PDM) ─────────────────────────────────────

period_years_int = [2023, 2024, 2025, 2026]
period_labels    = ACTUAL_PERIODS   # ["2023", "2024", "2025", "2026 H1"]

brand_spend = {}
for brand in ["TRR", "PDM"]:
    aw_col = f"{brand}_Awareness_spend"
    co_col = f"{brand}_Consideration_spend"
    totals = []
    for yr in period_years_int:
        s = float(df_all.loc[df_all["year"] == yr, [aw_col, co_col]].sum().sum())
        totals.append(s / 1e6)
    brand_spend[brand] = np.array(totals)

# ── Chart ────────────────────────────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(11, 6))
fig.patch.set_facecolor(WHITE)

x = np.arange(len(period_labels))
w = 0.38

bars_trr = ax.bar(x - w/2, brand_spend["TRR"], w, color=CPX_NAVY, label="TRR")
bars_pdm = ax.bar(x + w/2, brand_spend["PDM"], w, color=CPX_RED,  label="PDM")

for bars in (bars_trr, bars_pdm):
    for bar in bars:
        h = bar.get_height()
        if h > 0:
            ax.text(bar.get_x() + bar.get_width()/2, h + 0.03, f"${h:.2f}M",
                    ha="center", va="bottom", fontsize=12, fontweight="bold",
                    color=CPX_DGREY)

ax.set_xticks(x)
ax.set_xticklabels(period_labels, fontsize=13, fontweight="bold", rotation=10, ha="right")
plt.setp(ax.get_yticklabels(), fontsize=12, fontweight="bold")
ax.set_ylabel("Total Media Spend ($M)", fontsize=13, fontweight="bold")
ax.set_xlabel("Period", fontsize=13, fontweight="bold")
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda v, _: f"${v:.1f}M"))
ax.set_ylim(0, max(brand_spend["TRR"].max(), brand_spend["PDM"].max()) * 1.20)
ax.set_title("LBE Media Spend by Brand & Year",
             fontsize=14, fontweight="bold", color=CPX_NAVY)
ax.legend(fontsize=12, loc="upper right", frameon=False)
ax.spines["top"].set_visible(False); ax.spines["right"].set_visible(False)
ax.grid(False)

plt.tight_layout()
save_fig(fig, "lbe_spend_by_brand_year.png")
display(fig); plt.close(fig)

# ── Table ────────────────────────────────────────────────────────────────────
tbl = pd.DataFrame({
    "Period": period_labels,
    "TRR ($M)":   [round(v, 2) for v in brand_spend["TRR"]],
    "PDM ($M)":   [round(v, 2) for v in brand_spend["PDM"]],
    "Total ($M)": [round(t + p, 2) for t, p in zip(brand_spend["TRR"], brand_spend["PDM"])],
    "TRR share":  [f"{t/(t+p)*100:.0f}%" if (t+p) > 0 else "—"
                   for t, p in zip(brand_spend["TRR"], brand_spend["PDM"])],
})
ipy_display(tbl)

In [0]:
# ── DATA TABLES: LBE Spend by Channel & Funnel — One Table Per Year ───────────
year_labels = {2023: "2023", 2024: "2024", 2025: "2025", 2026: "2026 H1"}
for year in [2023, 2024, 2025, 2026]:
    pivot = (
        spend_df_combined[spend_df_combined["Year"] == year]
        .pivot_table(index="Channel", columns="Funnel", values="Spend_M", aggfunc="sum", fill_value=0)
        .reindex(columns=["Awareness", "Consideration"])
        .fillna(0)
    )
    pivot = pivot * 100
    pivot["Total"] = pivot.sum(axis=1)
    pivot = pivot.sort_values("Total", ascending=False).reset_index()
    pivot.columns.name = None
    pivot[["Awareness", "Consideration", "Total"]] = pivot[["Awareness", "Consideration", "Total"]].round(2)

    print(f"\n{'═'*60}")
    print(f"  LBE Annual Media Spend by Channel & Funnel — {year_labels[year]}")
    print(f"  (values in $10K)")
    print(f"{'═'*60}")
    display(pivot)

In [0]:
# ── LBE REVENUE BREAKDOWN BY YEAR ───────────────────────────────────────────

# ── Build period-labeled revenue frame ────────────────────────────────────────
# Actuals: 2023, 2024, 2025, 2026 H1
_rev_actual = (
    df_all[df_all["year"].isin([2023, 2024, 2025, 2026])]
    .groupby("year")["LBE_Revenue"].sum()
    .reset_index()
)
_rev_actual["Period"] = _rev_actual["year"].map(lambda y: str(y) if y != 2026 else "2026 H1")
_rev_actual["Revenue_M"] = _rev_actual["LBE_Revenue"] / 1e6
_rev_actual["is_forecast"] = False

# Forecasts (separate bars)
forecast_rows = []
if FORECAST_2026 is not None:
    forecast_rows.append({
        "Period": "2026 Forecast", "LBE_Revenue": FORECAST_2026,
        "Revenue_M": FORECAST_2026 / 1e6, "is_forecast": True,
    })
if FORECAST_2027 is not None:
    forecast_rows.append({
        "Period": "2027 Forecast", "LBE_Revenue": FORECAST_2027,
        "Revenue_M": FORECAST_2027 / 1e6, "is_forecast": True,
    })

rev_df_lbe = pd.concat(
    [_rev_actual[["Period", "LBE_Revenue", "Revenue_M", "is_forecast"]],
     pd.DataFrame(forecast_rows)],
    ignore_index=True
)

# ── YoY logic (custom anchors, not naive pct_change) ──────────────────────────
# Anchors:
#   2024, 2025           → prior full year
#   2026 H1              → 2025 (partial)
#   2026 Forecast        → 2025 (skip H1 as anchor — apples-to-apples full year)
#   2027 Forecast        → 2026 Forecast
rev_2025 = float(rev_df_lbe.loc[rev_df_lbe["Period"] == "2025", "Revenue_M"].values[0])
rev_2024 = float(rev_df_lbe.loc[rev_df_lbe["Period"] == "2024", "Revenue_M"].values[0])
rev_2023 = float(rev_df_lbe.loc[rev_df_lbe["Period"] == "2023", "Revenue_M"].values[0])

yoy_values = []
yoy_labels = []
for _, row in rev_df_lbe.iterrows():
    p = row["Period"]; v = row["Revenue_M"]
    if p == "2023":
        yoy, lbl = None, ""
    elif p == "2024":
        yoy = (v - rev_2023) / rev_2023 * 100
        lbl = f"{yoy:+.1f}% YoY"
    elif p == "2025":
        yoy = (v - rev_2024) / rev_2024 * 100
        lbl = f"{yoy:+.1f}% YoY"
    elif p == "2026 H1":
        yoy = (v - rev_2025) / rev_2025 * 100
        lbl = f"{yoy:+.1f}% YoY (partial)"
    elif p == "2026 Forecast":
        yoy = (v - rev_2025) / rev_2025 * 100
        lbl = f"{yoy:+.1f}% YoY (F)"
    elif p == "2027 Forecast":
        rev_2026F = float(rev_df_lbe.loc[rev_df_lbe["Period"] == "2026 Forecast", "Revenue_M"].values[0])
        yoy = (v - rev_2026F) / rev_2026F * 100
        lbl = f"{yoy:+.1f}% YoY (F)"
    else:
        yoy, lbl = None, ""
    yoy_values.append(yoy)
    yoy_labels.append(lbl)

rev_df_lbe["YoY_pct"] = yoy_values
rev_df_lbe["YoY_label"] = yoy_labels

# ── Plot ──────────────────────────────────────────────────────────────────────
fig, ax = plt.subplots(figsize=get_figsize("16:9"))
fig.patch.set_facecolor(WHITE)
x     = np.arange(len(rev_df_lbe))
width = 0.5
bar_alphas = [0.5 if fc else 0.9 for fc in rev_df_lbe["is_forecast"]]

for i_b in range(len(x)):
    ax.bar(x[i_b], rev_df_lbe["Revenue_M"].iloc[i_b], width=width,
           color="black", alpha=bar_alphas[i_b], edgecolor=WHITE, linewidth=0.5)
bars = ax.containers[0] if len(ax.containers) == 1 else [p for c in ax.containers for p in c]

for i, (bar, row) in enumerate(zip(bars, rev_df_lbe.itertuples())):
    h = bar.get_height()
    ax.text(bar.get_x() + bar.get_width() / 2, h + 1.0, f"${h:.2f}M",
            ha="center", va="bottom", fontsize=14, fontweight="bold", color=CPX_NAVY)
    if row.YoY_label:
        color = CPX_RED if (row.YoY_pct is not None and row.YoY_pct < 0) else "black"
        ax.text(bar.get_x() + bar.get_width() / 2, h + 5.0, row.YoY_label,
                ha="center", va="bottom", fontsize=13, fontweight="bold", color=color)

ax.set_xticks(x)
ax.set_xticklabels(rev_df_lbe["Period"], fontsize=13, fontweight="bold", rotation=10, ha="right")
plt.setp(ax.get_yticklabels(), fontsize=14, fontweight="bold")
ax.set_ylabel("Total Annual Revenue ($M)", fontsize=14, fontweight="bold")
ax.set_xlabel("Period", fontsize=14, fontweight="bold")
ax.set_ylim(0, rev_df_lbe["Revenue_M"].max() * 1.30)
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda v, _: f"${v:.0f}M"))
last_period = rev_df_lbe["Period"].iloc[-1]
ax.set_title(f"LBE Total Annual Revenue (2023–{last_period})",
             fontsize=14, fontweight="bold", color=CPX_NAVY)
ax.grid(False)
ax.spines["top"].set_visible(False); ax.spines["right"].set_visible(False)
plt.tight_layout()
save_fig(fig, "lbe_revenue_by_year.png")
display(fig); plt.close(fig)

# ── Table ────────────────────────────────────────────────────────────────────
table_lbe = rev_df_lbe[["Period", "Revenue_M", "YoY_label"]].copy()
table_lbe.columns = ["Period", "Total Revenue ($M)", "YoY Change"]
table_lbe["Total Revenue ($M)"] = table_lbe["Total Revenue ($M)"].round(2)
table_lbe["YoY Change"] = table_lbe["YoY Change"].replace("", "—")
table_lbe = table_lbe.set_index("Period")

display(
    table_lbe
    .style
    .format({"Total Revenue ($M)": "${:.2f}M"}, na_rep="—")
    .set_caption(f"LBE Total Annual Revenue (2023–{last_period})")
    .set_properties(**{"text-align": "center"})
    .set_table_styles([
        {"selector": "caption",
         "props": [("font-size", "13px"), ("font-weight", "bold"),
                   ("color", CPX_NAVY), ("text-align", "left")]},
        {"selector": "th",
         "props": [("background-color", CPX_NAVY), ("color", WHITE),
                   ("font-weight", "bold"), ("text-align", "center")]},
    ])
)

In [0]:
# ── LBE PER-CHANNEL ANNUAL SPEND & REVENUE CONTRIBUTION ─────────────────────
# One chart per channel (TRR, PDM). Uses V5e per-brand ROI from results dict.
import matplotlib.patches as mpatches

FUNNEL_COLOR = {"Awareness": CPX_NAVY, "Consideration": CPX_RED}
FUNNELS_LBE  = ["Awareness", "Consideration"]
CHANNELS     = ["TRR", "PDM"]

# Parallel lists: int for filtering df_all, string for display
period_years_int = [2023, 2024, 2025, 2026]
period_labels    = ACTUAL_PERIODS   # ["2023", "2024", "2025", "2026 H1"]

# ── Per-brand ROI from V5e results ──────────────────────────────────────────
chan_roi = {
    ("TRR", "Awareness"):     brand_roi["TRR"]["Awareness"],
    ("TRR", "Consideration"): brand_roi["TRR"]["Consideration"],
    ("PDM", "Awareness"):     brand_roi["PDM"]["Awareness"],
    ("PDM", "Consideration"): brand_roi["PDM"]["Consideration"],
}

chan_funnel_col = {
    ("TRR", "Awareness"):     "Awareness_TRR",
    ("TRR", "Consideration"): "Consideration_TRR",
    ("PDM", "Awareness"):     "Awareness_PDM",
    ("PDM", "Consideration"): "Consideration_PDM",
}

n_yrs = len(period_labels)
bar_w = 0.35
gap   = 0.15
group = bar_w * 2 + gap + 0.2

for channel in CHANNELS:
    spend_by_f = {}
    contrib_by_f = {}
    for funnel in FUNNELS_LBE:
        col = chan_funnel_col[(channel, funnel)]
        roi = chan_roi[(channel, funnel)]
        s_arr, c_arr = [], []
        for yr_int in period_years_int:
            spend = df_all.loc[df_all["year"] == yr_int, col].sum() / 1e6
            s_arr.append(spend)
            c_arr.append(spend * roi)
        spend_by_f[funnel]   = np.array(s_arr)
        contrib_by_f[funnel] = np.array(c_arr)

    total_spend_yr   = sum(spend_by_f[f] for f in FUNNELS_LBE)
    total_contrib_yr = sum(contrib_by_f[f] for f in FUNNELS_LBE)

    x_spend   = np.array([i * group for i in range(n_yrs)])
    x_contrib = x_spend + bar_w + gap

    fig, ax = plt.subplots(figsize=(11, 9))
    bottoms_s = np.zeros(n_yrs)
    for f in FUNNELS_LBE:
        ax.bar(x_spend, spend_by_f[f], bar_w, bottom=bottoms_s,
               color=FUNNEL_COLOR[f], alpha=0.9, edgecolor=WHITE, lw=0.4)
        bottoms_s += spend_by_f[f]

    bottoms_c = np.zeros(n_yrs)
    for f in FUNNELS_LBE:
        ax.bar(x_contrib, contrib_by_f[f], bar_w, bottom=bottoms_c,
               color=FUNNEL_COLOR[f], alpha=0.55, edgecolor=WHITE, lw=0.4, hatch="///")
        bottoms_c += contrib_by_f[f]

    for i in range(n_yrs):
        if bottoms_s[i] > 0:
            ax.annotate(f"${bottoms_s[i]:.2f}M",
                        xy=(x_spend[i], bottoms_s[i]), xytext=(0, 4), textcoords="offset points",
                        ha="center", fontsize=8, color=CPX_DGREY, fontweight="bold")
        if bottoms_c[i] > 0:
            ax.annotate(f"${bottoms_c[i]:.2f}M",
                        xy=(x_contrib[i], bottoms_c[i]), xytext=(0, 4), textcoords="offset points",
                        ha="center", fontsize=11, color=CPX_DGREY, fontweight="bold")

    x_mid = (x_spend + x_contrib) / 2
    ax.set_xticks(x_mid)
    ax.set_xticklabels(period_labels, fontsize=13, fontweight="bold", rotation=10, ha="right")
    ax.set_xlim(-0.3, x_contrib[-1] + bar_w + 0.3)

    max_val = max(bottoms_s.max(), bottoms_c.max())
    ax.set_ylabel("$M", fontsize=14, fontweight="bold")
    ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda v, _: f"${v:.1f}M"))
    ax.set_ylim(bottom=0, top=max_val * 1.25)
    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)
    plt.setp(ax.get_yticklabels(), fontsize=12, fontweight="bold")
    ax.grid(False)

    handles = [mpatches.Patch(color=FUNNEL_COLOR[f], label=f) for f in FUNNELS_LBE]
    handles += [
        mpatches.Patch(facecolor="#999999", alpha=0.9, label="Spend"),
        mpatches.Patch(facecolor="#999999", alpha=0.55, hatch="///", label="Revenue Contrib"),
    ]
    ax.legend(handles=handles, fontsize=12, loc="upper right", frameon=False)
    ax.set_title(f"{channel} — Annual Spend vs Revenue Contribution",
                 fontsize=14, fontweight="bold", color=CPX_NAVY, pad=10)

    plt.tight_layout()
    save_fig(fig, f"lbe_channel_{channel.lower()}_spend_vs_contrib.png")
    display(fig)
    plt.close(fig)

    tbl_rows = []
    for yr_idx, yr_label in enumerate(period_labels):
        row = {"Period": yr_label}
        for f in FUNNELS_LBE:
            row[f"{f} Spend ($K)"] = f"{spend_by_f[f][yr_idx] * 1000:.0f}K"
            row[f"{f} Contrib ($M)"] = round(contrib_by_f[f][yr_idx], 3)
        row["Total Spend ($K)"] = f"{total_spend_yr[yr_idx] * 1000:.0f}K"
        row["Total Contrib ($M)"] = round(total_contrib_yr[yr_idx], 3)
        tbl_rows.append(row)
    rate_row = {"Period": "ROI"}
    for f in FUNNELS_LBE:
        rate_row[f"{f} Spend ($K)"] = ""
        rate_row[f"{f} Contrib ($M)"] = f"{chan_roi[(channel, f)]:.2f}×"
    rate_row["Total Spend ($K)"] = ""
    rate_row["Total Contrib ($M)"] = ""
    tbl_rows.append(rate_row)

    print(f"\n{'═'*60}")
    print(f"  {channel} — Spend & Revenue Contribution")
    print(f"{'═'*60}")
    ipy_display(pd.DataFrame(tbl_rows))

In [0]:
# ── LBE DIMINISHING RETURNS & MARGINAL RETURN CURVES ───────────────────────
# Built on V5e dual-target model with sat_half FIXED at per-brand data anchors.
# Varies total budget at 2025 allocation shares, computes combined Hill saturation
# response and marginal return.
from matplotlib.lines import Line2D

print("Generating aggregate response and marginal return curves...")

# ── V5e Hill saturation response — uses per-brand anchors ────────────────────
def hill(x, half):
    """Hill saturation: x / (x + half). `half` is the anchor (per brand+channel)."""
    return x / (x + half)

def steady_state_adstock(raw, alpha, L=ADSTOCK_L):
    """Constant weekly spend at steady state → geometric-sum adstock value.
    For raw normalized weekly = w: adstocked = w * (1 − α^L) / (1 − α)."""
    if alpha >= 1.0:
        return raw * L
    return raw * (1 - alpha**L) / (1 - alpha)

def v5e_media_rev(brand, aw_raw_norm, co_raw_norm):
    r = results[brand]
    anchor_aw = r["anchor_aw"]
    anchor_co = r["anchor_co"]
    alpha_aw  = (r["adstock_aw_rpt"] + r["adstock_aw_txn"]) / 2
    alpha_co  = (r["adstock_co_rpt"] + r["adstock_co_txn"]) / 2

    aw_ad_new = steady_state_adstock(aw_raw_norm, alpha_aw)
    co_ad_new = steady_state_adstock(co_raw_norm, alpha_co)

    m_s_aw = max(float(d[f"{brand}_Awareness_spend"].max()), 1.0)
    m_s_co = max(float(d[f"{brand}_Consideration_spend"].max()), 1.0)
    aw_train_raw = (d[f"{brand}_Awareness_spend"] / m_s_aw).values.astype(float)
    co_train_raw = (d[f"{brand}_Consideration_spend"] / m_s_co).values.astype(float)
    aw_ad_ref = adstock_np(aw_train_raw, alpha_aw).mean()
    co_ad_ref = adstock_np(co_train_raw, alpha_co).mean()

    aw_scale = hill(aw_ad_new, anchor_aw) / max(hill(aw_ad_ref, anchor_aw), 1e-9)
    co_scale = hill(co_ad_new, anchor_co) / max(hill(co_ad_ref, anchor_co), 1e-9)

    return r["aw_incr_rev"] * aw_scale + r["co_incr_rev"] * co_scale

# ── 2025 budget allocation shares ────────────────────────────────────────────
df_2025 = df_all[df_all["year"] == 2025]
aw_weekly_2025    = df_2025["Awareness"].mean()
co_weekly_2025    = df_2025["Consideration"].mean()
total_weekly_2025 = aw_weekly_2025 + co_weekly_2025
aw_share = aw_weekly_2025 / total_weekly_2025
co_share = co_weekly_2025 / total_weekly_2025
total_annual_2025 = total_weekly_2025 * 52

m_s_TRR = np.array([d["TRR_Awareness_spend"].max(), d["TRR_Consideration_spend"].max()])
m_s_PDM = np.array([d["PDM_Awareness_spend"].max(), d["PDM_Consideration_spend"].max()])
m_s_TRR[m_s_TRR == 0] = 1.0
m_s_PDM[m_s_PDM == 0] = 1.0

def combined_rev(annual_budget):
    weekly = annual_budget / 52
    aw_wk = weekly * aw_share
    co_wk = weekly * co_share
    return (v5e_media_rev("TRR", aw_wk / m_s_TRR[0], co_wk / m_s_TRR[1]) +
            v5e_media_rev("PDM", aw_wk / m_s_PDM[0], co_wk / m_s_PDM[1]))

# ── X axis + curve ────────────────────────────────────────────────────────────
x_annual   = np.linspace(52 * 100, total_annual_2025 * 10, 500)
curve_mean = np.array([combined_rev(b) for b in x_annual])
marginal_mean = np.gradient(curve_mean, x_annual)

# ── Break-even: where marginal return crosses $1 (going downward) ────────────
inflection_idx = None
for i in range(1, len(marginal_mean)):
    if marginal_mean[i - 1] >= 1.0 and marginal_mean[i] < 1.0:
        inflection_idx = i
        break

if inflection_idx is not None:
    x0, x1 = x_annual[inflection_idx - 1], x_annual[inflection_idx]
    m0, m1 = marginal_mean[inflection_idx - 1], marginal_mean[inflection_idx]
    roi1_spend = x0 + (1.0 - m0) * (x1 - x0) / (m1 - m0)
    roi1_rev   = float(np.interp(roi1_spend, x_annual, curve_mean))
else:
    roi1_spend = None
    roi1_rev   = None
    print("  ⚠ Marginal return does not cross $1 in the plotted range.")

# ── Period markers — includes 2026 H1 ─────────────────────────────────────────
period_years_int = [2023, 2024, 2025, 2026]
period_label_map = {2023: "2023", 2024: "2024", 2025: "2025", 2026: "2026 H1"}

year_points = {}
for yr_int in period_years_int:
    df_yr = df_all[df_all["year"] == yr_int]
    annual_budget = df_yr["Awareness"].sum() + df_yr["Consideration"].sum()
    year_points[yr_int] = {
        "label":        period_label_map[yr_int],
        "annual_spend": annual_budget,
        "annual_rev":   combined_rev(annual_budget),
    }

YEAR_STYLES = {
    2023: {"color": CPX_DGREY,  "linestyle": "--", "linewidth": 1.2, "marker": "o"},
    2024: {"color": CPX_NAVY,   "linestyle": "-.", "linewidth": 1.2, "marker": "s"},
    2025: {"color": CPX_RED,    "linestyle": "-",  "linewidth": 1.5, "marker": "D"},
    2026: {"color": CPX_GOLD,   "linestyle": ":",  "linewidth": 1.5, "marker": "v"},
}

# ── PLOT 1: Total Response Curve ──────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(16, 9))
fig.patch.set_facecolor(WHITE); ax.set_facecolor(WHITE)
ax.plot(x_annual / 1e6, curve_mean / 1e6, color=CPX_NAVY, linewidth=2.5)

# Break-even: gold dashed vline + gold star
if roi1_spend is not None:
    ax.axvline(roi1_spend / 1e6, color=CPX_GOLD, linewidth=1.8, linestyle="--", alpha=0.9, zorder=5)
    ax.scatter([roi1_spend / 1e6], [roi1_rev / 1e6], color=CPX_GOLD, s=120, zorder=6,
               marker="*", edgecolors=WHITE, linewidths=0.8)
    ax.annotate(f"Efficiency Breakeven\nMarginal Return = $1 per $1\n${roi1_spend/1e6:.1f}M / yr",
                xy=(roi1_spend / 1e6, roi1_rev / 1e6), xytext=(-12, 38), textcoords="offset points",
                fontsize=10, color=CPX_GOLD, fontweight="bold", ha="right",
                arrowprops=dict(arrowstyle="->", color=CPX_GOLD, lw=0.8),
                bbox=dict(boxstyle="round,pad=0.3", fc=WHITE, alpha=0.9, ec="none"))

# Year markers — vline + scatter + label-only annotation
for yr_int, style in YEAR_STYLES.items():
    pts = year_points[yr_int]
    sm  = pts["annual_spend"] / 1e6
    rm  = pts["annual_rev"] / 1e6
    ax.axvline(sm, color=style["color"], linestyle=style["linestyle"],
               linewidth=style["linewidth"], alpha=0.85, zorder=4)
    ax.scatter([sm], [rm], color=style["color"], marker=style["marker"], s=100, zorder=7,
               edgecolors=WHITE, linewidths=1.0)
    ax.annotate(pts["label"],
                xy=(sm, rm), xytext=(6, 6), textcoords="offset points",
                fontsize=11, color=style["color"], fontweight="bold")

ax.set_title("LBE MMM — Total Media Response Curve (V5e)\n"
             "Steady-state adstock  |  Budget shares held at 2025 mix",
             fontsize=14, fontweight="bold", color="black", pad=16)
ax.set_xlabel("Annual Media Budget ($M)", fontsize=14, fontweight="bold")
ax.set_ylabel("Annual Media Revenue Contribution ($M)", fontsize=14, fontweight="bold")
ax.xaxis.set_major_formatter(plt.FuncFormatter(lambda v, _: f"${v:.0f}M"))
ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda v, _: f"${v:.1f}M"))
plt.setp(ax.get_xticklabels(), fontsize=12, fontweight="bold")
plt.setp(ax.get_yticklabels(), fontsize=12, fontweight="bold")
ax.grid(False)

legend_handles = [
    Line2D([0], [0], color=s["color"], marker=s["marker"], linestyle=s["linestyle"],
           linewidth=s["linewidth"], markersize=7,
           label=f"{year_points[yr]['label']} (${year_points[yr]['annual_spend']/1e6:.1f}M annual)")
    for yr, s in YEAR_STYLES.items()
]
ax.legend(handles=legend_handles, loc="lower right", fontsize=13, frameon=True,
          facecolor=WHITE, edgecolor="#CCCCCC", framealpha=0.95, markerscale=1.6,
          borderpad=1.0, labelspacing=0.8, handlelength=2.8,
          title="Annual Spend Reference", title_fontsize=12)
ax.spines["top"].set_visible(False); ax.spines["right"].set_visible(False)
fig.text(0.02, 0.01,
         "* Budget distributed at 2025 spend shares. Hill uses per-brand anchor midpoints; "
         "steady-state adstock applied. 2026 H1 shown at partial-year spend total.",
         fontsize=7.5, color=CPX_DGREY, style="italic")
plt.tight_layout(rect=[0, 0.03, 1, 1])
save_fig(fig, "lbe_total_media_response_curve.png")
display(fig); plt.close(fig)

# ── PLOT 2: Marginal Return Curve ─────────────────────────────────────────────
fig2, ax2 = plt.subplots(figsize=(16, 9))
fig2.patch.set_facecolor(WHITE); ax2.set_facecolor(WHITE)
ax2.plot(x_annual / 1e6, marginal_mean, color=CPX_NAVY, linewidth=2.5)

# Red inefficient-spending zone (marginal < $1)
ax2.axhspan(0, 1, alpha=0.12, color=CPX_RED, zorder=0)
ax2.axhline(1, color=CPX_RED, lw=1.8, alpha=0.85, zorder=6)
ax2.text(x_annual[-1] / 1e6 * 0.60, 0.5,
         "Inefficient Spending Zone\nMarginal Return < $1 per $1",
         fontsize=10, color=CPX_RED, fontweight="bold", alpha=0.9, va="center", ha="center",
         bbox=dict(boxstyle="round,pad=0.35", fc=WHITE, alpha=0.8, ec=CPX_RED, lw=0.6))

# Gold ROI=1 crossover
if roi1_spend is not None:
    ax2.vlines(roi1_spend / 1e6, ymin=0, ymax=1.0, color=CPX_GOLD,
               linewidth=2.2, linestyle="--", alpha=0.9, zorder=5)
    ax2.scatter([roi1_spend / 1e6], [1.0], color=CPX_GOLD, s=140, zorder=8,
                marker="*", edgecolors=WHITE, linewidths=0.8)
    ax2.annotate(f"${roi1_spend/1e6:.1f}M", xy=(roi1_spend / 1e6, 0), xytext=(5, 6),
                 textcoords="offset points", fontsize=10, color=CPX_GOLD, fontweight="bold", ha="left")

# Year markers — vline + scatter + label-only annotation
for yr_int, style in YEAR_STYLES.items():
    pts = year_points[yr_int]
    sm  = pts["annual_spend"] / 1e6
    idx = np.argmin(np.abs(x_annual - pts["annual_spend"]))
    ym  = marginal_mean[idx]
    ax2.axvline(sm, color=style["color"], linestyle=style["linestyle"],
                linewidth=2.2, alpha=0.85, zorder=4)
    ax2.scatter([sm], [ym], color=style["color"], marker=style["marker"], s=100, zorder=7,
                edgecolors=WHITE, linewidths=1.0)
    ax2.annotate(pts["label"],
                 xy=(sm, ym), xytext=(6, 6), textcoords="offset points",
                 fontsize=11, color=style["color"], fontweight="bold")

ax2.set_title("LBE MMM — Marginal Return per $1 Spent (V5e)\n"
              "Incremental revenue per additional dollar",
              fontsize=14, fontweight="bold", color="black", pad=16)
ax2.set_xlabel("Annual Media Budget ($M)", fontsize=14, fontweight="bold")
ax2.set_ylabel("Incremental Revenue per $1", fontsize=14, fontweight="bold")
ax2.xaxis.set_major_formatter(plt.FuncFormatter(lambda v, _: f"${v:.0f}M"))
plt.setp(ax2.get_xticklabels(), fontsize=12, fontweight="bold")
plt.setp(ax2.get_yticklabels(), fontsize=12, fontweight="bold")
ax2.set_ylim(bottom=0); ax2.grid(False)

legend_handles_m = [
    Line2D([0], [0], color=s["color"], marker=s["marker"], linestyle=s["linestyle"],
           linewidth=s["linewidth"], markersize=7,
           label=f"{year_points[yr]['label']} (${year_points[yr]['annual_spend']/1e6:.1f}M annual)")
    for yr, s in YEAR_STYLES.items()
]
ax2.legend(handles=legend_handles_m, loc="upper right", fontsize=13, frameon=True,
           facecolor=WHITE, edgecolor="#CCCCCC", framealpha=0.95, markerscale=1.6,
           borderpad=1.0, labelspacing=0.8, handlelength=2.8,
           title="Annual Spend Reference", title_fontsize=12)
ax2.spines["top"].set_visible(False); ax2.spines["right"].set_visible(False)
fig2.text(0.02, 0.01,
          "* Budget distributed at 2025 spend shares. Hill uses per-brand anchor midpoints; "
          "steady-state adstock applied. 2026 H1 shown at partial-year spend total.",
          fontsize=7.5, color=CPX_DGREY, style="italic")
plt.tight_layout(rect=[0, 0.03, 1, 1])
save_fig(fig2, "lbe_marginal_return_curve.png")
display(fig2); plt.close(fig2)

print(f"\n✓ Response + marginal curves complete.")
print(f"  2025 annual spend:      ${total_annual_2025/1e6:.2f}M")
print(f"  2026 H1 spend:          ${year_points[2026]['annual_spend']/1e6:.2f}M (partial year)")
if roi1_spend is not None:
    print(f"  Efficiency breakeven:   ${roi1_spend/1e6:.1f}M ({roi1_spend/total_annual_2025:.1f}x 2025)")

In [0]:
# ── TIER-LEVEL SPEND & MEDIA-ATTRIBUTED REVENUE ───────────────────────────────
import matplotlib.patches as mpatches

FUNNELS      = ["Awareness", "Consideration"]
BRANDS       = ["TRR", "PDM"]
FUNNEL_COLOR = {"Awareness": CPX_NAVY, "Consideration": CPX_RED}
brand_tiers  = {"TRR": ["HPL", "LPL", "NL"], "PDM": ["OL", "NL"]}

# Parallel lists: int for filtering, string for display
YEARS        = [2023, 2024, 2025, 2026]
YEAR_LABELS  = {2023: "2023", 2024: "2024", 2025: "2025", 2026: "2026 H1"}
YEAR_COLORS  = {2023: CPX_DGREY, 2024: CPX_NAVY, 2025: CPX_RED, 2026: CPX_GOLD}

# ── Step 1: Annual spend per (brand, funnel, tier, year) — raw tier columns ──
eff_spend = {}
for brand, tiers in brand_tiers.items():
    for funnel in FUNNELS:
        for tier in tiers:
            col = f"{funnel}_{brand}_{tier}"
            if col not in d.columns:
                continue
            for yr in YEARS:
                eff_spend[(brand, funnel, tier, yr)] = float(
                    d.loc[d["week_date"].dt.year == yr, col].sum()
                )

# ── Step 2: Year-specific counterfactual incremental revenue ─────────────────
def _brand_weekly_incr(r):
    """Remove media log-contribution from both RPT and TXN sub-models."""
    wk_aw = (
        (r["pred_rpt"] - np.exp(r["log_pred_rpt"] - r["m_aw_rpt"])) * r["pred_txn"]
        + r["pred_rpt"] * (r["pred_txn"] - np.exp(r["log_pred_txn"] - r["m_aw_txn"]))
    )
    wk_co = (
        (r["pred_rpt"] - np.exp(r["log_pred_rpt"] - r["m_co_rpt"])) * r["pred_txn"]
        + r["pred_rpt"] * (r["pred_txn"] - np.exp(r["log_pred_txn"] - r["m_co_txn"]))
    )
    return wk_aw, wk_co

brand_funnel_yr_incr = {}
for brand in BRANDS:
    wk_aw, wk_co = _brand_weekly_incr(results[brand])
    decomp_tmp = d[["week_date"]].copy()
    decomp_tmp["year"] = decomp_tmp["week_date"].dt.year
    decomp_tmp["incr_Awareness"] = wk_aw
    decomp_tmp["incr_Consideration"] = wk_co
    for yr in YEARS:
        mask = decomp_tmp["year"] == yr
        brand_funnel_yr_incr[(brand, "Awareness", yr)] = decomp_tmp.loc[mask, "incr_Awareness"].sum()
        brand_funnel_yr_incr[(brand, "Consideration", yr)] = decomp_tmp.loc[mask, "incr_Consideration"].sum()

# Allocate year-specific incr revenue to tiers by within-year spend proportions
tier_rev = {}
for brand, tiers in brand_tiers.items():
    for funnel in FUNNELS:
        for yr in YEARS:
            yr_incr = brand_funnel_yr_incr[(brand, funnel, yr)]
            yr_total_eff = max(
                sum(eff_spend.get((brand, funnel, t, yr), 0.0) for t in tiers),
                1e-9
            )
            for tier in tiers:
                w = eff_spend.get((brand, funnel, tier, yr), 0.0) / yr_total_eff
                tier_rev[(brand, funnel, tier, yr)] = yr_incr * w

print("Year-specific counterfactual ROI by brand/funnel:")
for brand in BRANDS:
    for funnel in FUNNELS:
        vals = []
        for yr in YEARS:
            s = sum(eff_spend.get((brand, funnel, t, yr), 0.0) for t in brand_tiers[brand])
            r_val = brand_funnel_yr_incr[(brand, funnel, yr)]
            roi = r_val / s if s > 0 else 0
            vals.append(f"{YEAR_LABELS[yr]}={roi:.2f}x")
        print(f"  {brand} {funnel}: {', '.join(vals)}")

# ── Accessors ($M) ────────────────────────────────────────────────────────────
def sp(brand, funnel, tier, yr):
    return eff_spend.get((brand, funnel, tier, yr), 0.0) / 1e6

def rv(brand, funnel, tier, yr):
    return tier_rev.get((brand, funnel, tier, yr), 0.0) / 1e6

fmt_M = mticker.FuncFormatter(lambda v, _: f"${v:.2f}M")

# Shared x-axis layout: TRR tiers then PDM tiers
all_brand_tiers = [(b, t) for b in BRANDS for t in brand_tiers[b]]
n_bt       = len(all_brand_tiers)
trr_div    = len(brand_tiers["TRR"]) - 0.5
trr_center = (len(brand_tiers["TRR"]) - 1) / 2
pdm_center = len(brand_tiers["TRR"]) + (len(brand_tiers["PDM"]) - 1) / 2

def add_brand_dividers(ax, y_top):
    ax.axvline(trr_div, color="#CCCCCC", lw=1.2, linestyle="--")
    for bx, blabel in [(trr_center, "TRR"), (pdm_center, "PDM")]:
        ax.text(bx, y_top * 0.97, blabel, ha="center", va="top",
                fontsize=11, fontweight="bold", color="black")

def style_ax(ax, y_top):
    ax.set_ylim(0, y_top)
    ax.yaxis.set_major_formatter(fmt_M)
    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)
    plt.setp(ax.get_yticklabels(), fontsize=12, fontweight="bold")
    ax.grid(False)

# ═══════════════════════════════════════════════════════════════════════════════
# CHART 1 — Spend by Tier, both brands, one chart per year, funnel stacked
# ═══════════════════════════════════════════════════════════════════════════════
for yr in YEARS:
    fig, ax = plt.subplots(figsize=(11, 7))
    fig.patch.set_facecolor(WHITE)

    x       = np.arange(n_bt)
    bar_w   = 0.5
    bottoms = np.zeros(n_bt)

    for funnel in FUNNELS:
        heights = np.array([sp(b, funnel, t, yr) for b, t in all_brand_tiers])
        ax.bar(x, heights, bar_w, bottom=bottoms,
               color=FUNNEL_COLOR[funnel], alpha=0.9,
               edgecolor=WHITE, lw=0.4, label=funnel)
        bottoms += heights

    for i, total in enumerate(bottoms):
        if total > 0:
            ax.annotate(f"${total:.2f}M", xy=(x[i], total),
                        xytext=(0, 4), textcoords="offset points",
                        ha="center", fontsize=12, color=CPX_DGREY, fontweight="bold")

    y_top = bottoms.max() * 1.3
    style_ax(ax, y_top)
    add_brand_dividers(ax, y_top)
    ax.set_xticks(x)
    ax.set_xticklabels([t for _, t in all_brand_tiers], fontsize=14, fontweight="bold")
    ax.set_ylabel("Media Spend ($M)", fontsize=14, fontweight="bold")
    ax.set_title(f"LBE Spend by Tier — {YEAR_LABELS[yr]}",
                 fontsize=14, fontweight="bold", color=CPX_NAVY)
    ax.legend(fontsize=12, loc="upper left", frameon=False)
    plt.tight_layout()
    fname_yr = YEAR_LABELS[yr].lower().replace(" ", "_")
    save_fig(fig, f"lbe_tier_spend_{fname_yr}.png")
    display(fig)
    plt.close(fig)

    tbl_rows = []
    for b, t in all_brand_tiers:
        row = {"Brand": b, "Tier": t}
        for funnel in FUNNELS:
            row[f"{funnel} ($M)"] = round(sp(b, funnel, t, yr), 3)
        row["Total ($M)"] = round(sum(sp(b, f, t, yr) for f in FUNNELS), 3)
        tbl_rows.append(row)
    print(f"\n{'═'*55}\n  Spend by Tier — {YEAR_LABELS[yr]}\n{'═'*55}")
    ipy_display(pd.DataFrame(tbl_rows))


# ═══════════════════════════════════════════════════════════════════════════════
# CHART 2 — Spend vs Revenue by Tier, one chart per year
# ═══════════════════════════════════════════════════════════════════════════════
bar_w = 0.3
gap   = 0.08
x_sp  = np.arange(n_bt) - (bar_w + gap) / 2
x_rv  = np.arange(n_bt) + (bar_w + gap) / 2

for yr in YEARS:
    fig, ax = plt.subplots(figsize=(11, 7))
    fig.patch.set_facecolor(WHITE)

    bottoms_s = np.zeros(n_bt)
    bottoms_r = np.zeros(n_bt)

    for funnel in FUNNELS:
        sp_vals = np.array([sp(b, funnel, t, yr) for b, t in all_brand_tiers])
        rv_vals = np.array([rv(b, funnel, t, yr) for b, t in all_brand_tiers])

        ax.bar(x_sp, sp_vals, bar_w, bottom=bottoms_s,
               color=FUNNEL_COLOR[funnel], alpha=0.9,
               edgecolor=WHITE, lw=0.4)
        ax.bar(x_rv, rv_vals, bar_w, bottom=bottoms_r,
               color=FUNNEL_COLOR[funnel], alpha=0.5,
               edgecolor=WHITE, lw=0.4, hatch="///")

        bottoms_s += sp_vals
        bottoms_r += rv_vals

    for i in range(n_bt):
        if bottoms_s[i] > 0:
            ax.annotate(f"${bottoms_s[i]:.2f}M",
                        xy=(x_sp[i], bottoms_s[i]),
                        xytext=(0, 4), textcoords="offset points",
                        ha="center", fontsize=12, color=CPX_DGREY, fontweight="bold")
        if bottoms_r[i] > 0:
            ax.annotate(f"${bottoms_r[i]:.2f}M",
                        xy=(x_rv[i], bottoms_r[i]),
                        xytext=(0, 4), textcoords="offset points",
                        ha="center", fontsize=12, color=CPX_DGREY, fontweight="bold")

    y_top = max(bottoms_s.max(), bottoms_r.max(), 0.01) * 1.3
    style_ax(ax, y_top)
    add_brand_dividers(ax, y_top)

    ax.set_xticks(np.arange(n_bt))
    ax.set_xticklabels([t for _, t in all_brand_tiers], fontsize=14, fontweight="bold")
    ax.set_ylabel("Media Spend ($M)", fontsize=14, fontweight="bold")

    funnel_handles = [mpatches.Patch(color=FUNNEL_COLOR[f], label=f) for f in FUNNELS]
    type_handles   = [
        mpatches.Patch(facecolor="#999999", alpha=0.9, label="Spend"),
        mpatches.Patch(facecolor="#999999", alpha=0.5, hatch="///", label="Rev Contrib"),
    ]
    ax.legend(handles=funnel_handles + type_handles,
              fontsize=12, loc="upper left", frameon=False)

    ax.set_title(f"LBE Spend vs Media-Attributed Revenue by Tier — {YEAR_LABELS[yr]}",
                 fontsize=14, fontweight="bold", color=CPX_NAVY)
    plt.tight_layout()
    fname_yr = YEAR_LABELS[yr].lower().replace(" ", "_")
    save_fig(fig, f"lbe_tier_spend_vs_rev_{fname_yr}.png")
    display(fig)
    plt.close(fig)

    tbl_rows = []
    for b, t in all_brand_tiers:
        row = {"Brand": b, "Tier": t}
        for funnel in FUNNELS:
            row[f"{funnel} Spend ($M)"] = round(sp(b, funnel, t, yr), 3)
            row[f"{funnel} Rev ($M)"]   = round(rv(b, funnel, t, yr), 3)
        row["Total Spend ($M)"] = round(sum(sp(b, f, t, yr) for f in FUNNELS), 3)
        row["Total Rev ($M)"]   = round(sum(rv(b, f, t, yr) for f in FUNNELS), 3)
        tbl_rows.append(row)
    rate_row = {"Brand": "ROI", "Tier": ""}
    for funnel in FUNNELS:
        s = sum(sp(b, f, t, yr) for b, t in all_brand_tiers for f in [funnel])
        r = sum(rv(b, f, t, yr) for b, t in all_brand_tiers for f in [funnel])
        rate_row[f"{funnel} Spend ($M)"] = ""
        rate_row[f"{funnel} Rev ($M)"]   = f"{r/s:.2f}×" if s > 0 else "—"
    rate_row["Total Spend ($M)"] = ""
    rate_row["Total Rev ($M)"]   = ""
    tbl_rows.append(rate_row)
    print(f"\n{'═'*55}\n  Spend vs Revenue by Tier — {YEAR_LABELS[yr]}\n{'═'*55}")
    ipy_display(pd.DataFrame(tbl_rows))

# ═══════════════════════════════════════════════════════════════════════════════
# CHART 3 — Spend vs Revenue by Tier, one chart per brand
# ═══════════════════════════════════════════════════════════════════════════════
bar_w    = 0.11
gap      = 0.05
pair_w   = bar_w * 2 + gap
tier_gap = 0.08
year_gap = 0.25

for brand in BRANDS:
    tiers   = brand_tiers[brand]
    n_tiers = len(tiers)
    n_years = len(YEARS)

    year_w = n_tiers * pair_w + (n_tiers - 1) * tier_gap

    fig, ax = plt.subplots(figsize=(max(11, n_years * (year_w + year_gap) * 5 + 2), 7))
    fig.patch.set_facecolor(WHITE)

    year_centers = []

    for yi, yr in enumerate(YEARS):
        base_x = yi * (year_w + year_gap)
        year_centers.append(base_x + year_w / 2)

        for ti, tier in enumerate(tiers):
            x_sp_pos = base_x + ti * (pair_w + tier_gap)
            x_rv_pos = x_sp_pos + bar_w + gap

            bottoms_s = 0.0
            bottoms_r = 0.0
            for funnel in FUNNELS:
                sp_val = sp(brand, funnel, tier, yr)
                rv_val = rv(brand, funnel, tier, yr)

                ax.bar(x_sp_pos, sp_val, bar_w, bottom=bottoms_s,
                       color=FUNNEL_COLOR[funnel], alpha=0.9,
                       edgecolor=WHITE, lw=0.3)
                ax.bar(x_rv_pos, rv_val, bar_w, bottom=bottoms_r,
                       color=FUNNEL_COLOR[funnel], alpha=0.5,
                       edgecolor=WHITE, lw=0.3, hatch="///")

                bottoms_s += sp_val
                bottoms_r += rv_val

            ax.text((x_sp_pos + x_rv_pos) / 2, -0.012, tier,
                    ha="center", va="top", fontsize=14,
                    color=CPX_DGREY, fontweight="bold",
                    transform=ax.get_xaxis_transform())

            if bottoms_s > 0:
                ax.annotate(f"${bottoms_s:.2f}M",
                            xy=(x_sp_pos, bottoms_s),
                            xytext=(0, 3), textcoords="offset points",
                            ha="center", fontsize=14, color=CPX_DGREY, fontweight="bold")
            if bottoms_r > 0:
                ax.annotate(f"${bottoms_r:.2f}M",
                            xy=(x_rv_pos, bottoms_r),
                            xytext=(0, 3), textcoords="offset points",
                            ha="center", fontsize=14, color=CPX_DGREY, fontweight="bold")

        if yi < n_years - 1:
            div_x = base_x + year_w + year_gap / 2
            ax.axvline(div_x, color="#CCCCCC", lw=1.2, linestyle="--")

    ax.set_xticks([])
    all_sp = [sum(sp(brand, f, t, yr) for f in FUNNELS) for t in tiers for yr in YEARS]
    all_rv = [sum(rv(brand, f, t, yr) for f in FUNNELS) for t in tiers for yr in YEARS]
    y_top = max(max(all_sp), max(all_rv), 0.01) * 1.35
    ax.set_ylim(0, y_top)

    for yi, yr in enumerate(YEARS):
        ax.text(year_centers[yi], y_top * 0.82, YEAR_LABELS[yr],
                ha="center", va="top", fontsize=14,
                fontweight="bold", color=YEAR_COLORS[yr])

    ax.set_xlim(-0.2, n_years * (year_w + year_gap) - year_gap + 0.2)
    ax.yaxis.set_major_formatter(fmt_M)
    ax.set_ylabel("Spend & Revenue ($M)", fontsize=14, fontweight="bold", color=CPX_DGREY)
    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)
    plt.setp(ax.get_yticklabels(), fontsize=14, fontweight="bold")
    ax.set_axisbelow(True)
    ax.grid(axis="y", alpha=0.2, color="#CCCCCC")

    funnel_handles = [mpatches.Patch(color=FUNNEL_COLOR[f], label=f) for f in FUNNELS]
    year_handles   = [mpatches.Patch(facecolor=YEAR_COLORS[yr], label=YEAR_LABELS[yr]) for yr in YEARS]
    type_handles   = [
        mpatches.Patch(facecolor="#999999", alpha=0.9, label="Spend"),
        mpatches.Patch(facecolor="#999999", alpha=0.5, hatch="///", label="Rev Contrib"),
    ]
    ax.legend(handles=funnel_handles + year_handles + type_handles,
              fontsize=14, loc="upper right", framealpha=0.9,
              edgecolor="#CCCCCC", ncol=3)

    ax.set_title(f"{brand} — Spend vs Revenue by Year & Tier ($M)",
                 fontsize=14, fontweight="bold", color=CPX_NAVY)
    plt.tight_layout()
    save_fig(fig, f"lbe_tier_{brand.lower()}_spend_vs_rev_by_year.png")
    display(fig)
    plt.close(fig)

    for yr in YEARS:
        tbl_rows = []
        for tier in tiers:
            row = {"Tier": tier}
            for funnel in FUNNELS:
                row[f"{funnel} Spend ($M)"] = round(sp(brand, funnel, tier, yr), 3)
                row[f"{funnel} Rev ($M)"]   = round(rv(brand, funnel, tier, yr), 3)
            row["Total Spend ($M)"] = round(sum(sp(brand, f, tier, yr) for f in FUNNELS), 3)
            row["Total Rev ($M)"]   = round(sum(rv(brand, f, tier, yr) for f in FUNNELS), 3)
            tbl_rows.append(row)
        rate_row = {"Tier": "ROI"}
        for funnel in FUNNELS:
            s = sum(sp(brand, funnel, t, yr) for t in tiers)
            r = sum(rv(brand, funnel, t, yr) for t in tiers)
            rate_row[f"{funnel} Spend ($M)"] = ""
            rate_row[f"{funnel} Rev ($M)"]   = f"{r/s:.2f}×" if s > 0 else "—"
        rate_row["Total Spend ($M)"] = ""
        rate_row["Total Rev ($M)"]   = ""
        tbl_rows.append(rate_row)
        print(f"\n{'═'*55}\n  {brand} — Spend vs Revenue by Tier — {YEAR_LABELS[yr]}\n{'═'*55}")
        ipy_display(pd.DataFrame(tbl_rows))

In [0]:
# ═══════════════════════════════════════════════════════════════════════════════
# TIER ROI COMPARISON BY YEAR — One chart per location tier
# X-axis = Years (2023, 2024, 2025, 2026 H1)
# Bars = Spend (stacked by funnel), ROI curve on secondary y-axis
# ═══════════════════════════════════════════════════════════════════════════════
import matplotlib.patches as mpatches
import matplotlib.lines as mlines

for brand in BRANDS:
    tiers = brand_tiers[brand]
    for tier in tiers:
        x_base  = np.arange(len(YEARS))
        bar_w   = 0.55

        fig, ax1 = plt.subplots(figsize=(11, 7))
        fig.patch.set_facecolor(WHITE)

        # ── Stacked bars: Spend by funnel per year ─────────────────────────
        bottoms = np.zeros(len(YEARS))
        for funnel in FUNNELS:
            vals = np.array([sp(brand, funnel, tier, yr) for yr in YEARS])
            ax1.bar(x_base, vals, bar_w, bottom=bottoms,
                    color=FUNNEL_COLOR[funnel], alpha=0.85,
                    edgecolor=WHITE, lw=0.5, label=funnel)
            bottoms += vals

        for i, total in enumerate(bottoms):
            if total > 0:
                ax1.annotate(f"${total:.2f}M",
                             xy=(x_base[i], total),
                             xytext=(0, 5), textcoords="offset points",
                             ha="center", fontsize=14, color=CPX_DGREY, fontweight="bold")

        ax1.set_ylabel("Media Spend ($M)", fontsize=14, fontweight="bold", color=CPX_DGREY)
        ax1.set_xticks(x_base)
        ax1.set_xticklabels([YEAR_LABELS[yr] for yr in YEARS],
                            fontsize=13, fontweight="bold", rotation=10, ha="right")
        plt.setp(ax1.get_yticklabels(), fontsize=14, fontweight="bold")
        ax1.yaxis.set_major_formatter(mticker.FuncFormatter(lambda v, _: f"${v:.2f}M"))
        ax1.spines["top"].set_visible(False)
        ax1.set_axisbelow(True)
        ax1.grid(axis="y", alpha=0.15, color="#CCCCCC")
        ax1.set_ylim(0, bottoms.max() * 1.4)

        # ── ROI curve on secondary y-axis ─────────────────────────────────
        ax2 = ax1.twinx()
        roi_vals = []
        for yr in YEARS:
            t_spend = sum(sp(brand, f, tier, yr) for f in FUNNELS)
            t_rev   = sum(rv(brand, f, tier, yr) for f in FUNNELS)
            roi_vals.append(t_rev / t_spend if t_spend > 0 else 0.0)

        ax2.plot(x_base, roi_vals,
                 color="black", linewidth=3, marker="D",
                 markersize=12, markeredgecolor=WHITE, markeredgewidth=2,
                 zorder=5, label="ROI")

        for i, roi in enumerate(roi_vals):
            if roi > 0:
                ax2.annotate(f"{roi:.2f}×",
                             xy=(x_base[i], roi),
                             xytext=(0, 12), textcoords="offset points",
                             ha="center", fontsize=14, fontweight="bold",
                             color="black")

        ax2.set_ylabel("ROI (Rev / Spend)", fontsize=14, fontweight="bold", color=CPX_DGREY)
        plt.setp(ax2.get_yticklabels(), fontsize=14, fontweight="bold")
        ax2.spines["top"].set_visible(False)
        if max(roi_vals) > 0:
            ax2.set_ylim(0, max(roi_vals) * 2.2)
        else:
            ax2.set_ylim(0, 5)

        # ── Legend ─────────────────────────────────────────────────────────
        funnel_handles = [mpatches.Patch(color=FUNNEL_COLOR[f], label=f"{f} Spend")
                          for f in FUNNELS]
        roi_handle = mlines.Line2D([], [], color="black", linewidth=3,
                                   marker="D", markersize=10, label="ROI")
        ax1.legend(handles=funnel_handles + [roi_handle],
                   fontsize=14, loc="upper left", framealpha=0.9,
                   edgecolor="#CCCCCC")

        ax1.set_title(f"{brand} {tier} — Spend & ROI by Year",
                      fontsize=14, fontweight="bold", color=CPX_NAVY)
        plt.tight_layout()
        save_fig(fig, f"lbe_tier_roi_{brand.lower()}_{tier.lower()}_by_year.png")
        display(fig)
        plt.close(fig)

        # ── Summary table ───────────────────────────────────────────────────
        tbl_rows = []
        for yr in YEARS:
            t_spend = sum(sp(brand, f, tier, yr) for f in FUNNELS)
            t_rev   = sum(rv(brand, f, tier, yr) for f in FUNNELS)
            roi     = t_rev / t_spend if t_spend > 0 else 0.0
            tbl_rows.append({
                "Year": YEAR_LABELS[yr],
                "Awareness Spend ($M)": round(sp(brand, "Awareness", tier, yr), 3),
                "Consideration Spend ($M)": round(sp(brand, "Consideration", tier, yr), 3),
                "Total Spend ($M)": round(t_spend, 3),
                "Total Rev ($M)": round(t_rev, 3),
                "ROI": f"{roi:.2f}×" if t_spend > 0 else "—",
            })
        print(f"\n{'═'*60}\n  {brand} {tier} — Spend & ROI by Year\n{'═'*60}")
        ipy_display(pd.DataFrame(tbl_rows))

In [0]:
# ═══════════════════════════════════════════════════════════════════════════════
# TIER ROI COMPARISON BY YEAR — One chart per location tier
# X-axis = Years (2023, 2024, 2025, 2026 H1)
# Bars = Spend (stacked by funnel), ROI curve on secondary y-axis
# H1 ROI line added so every year is also comparable H1-vs-H1 (2026 is H1-only)
# ═══════════════════════════════════════════════════════════════════════════════
import matplotlib.patches as mpatches
import matplotlib.lines as mlines

# ── H1-only spend/rev per (brand, funnel, tier, year) — mirrors eff_spend/tier_rev
#    from cell 28, filtered to Jan–Jun, so we can plot an H1-only ROI line ──────
h1_eff_spend = {}
for brand, tiers in brand_tiers.items():
    for funnel in FUNNELS:
        for tier in tiers:
            col = f"{funnel}_{brand}_{tier}"
            if col not in d.columns:
                continue
            for yr in YEARS:
                mask = (d["week_date"].dt.year == yr) & (d["week_date"].dt.month <= 6)
                h1_eff_spend[(brand, funnel, tier, yr)] = float(d.loc[mask, col].sum())

h1_brand_funnel_yr_incr = {}
for brand in BRANDS:
    wk_aw, wk_co = _brand_weekly_incr(results[brand])
    decomp_tmp = d[["week_date"]].copy()
    decomp_tmp["year"] = decomp_tmp["week_date"].dt.year
    decomp_tmp["h1"] = decomp_tmp["week_date"].dt.month <= 6
    decomp_tmp["incr_Awareness"] = wk_aw
    decomp_tmp["incr_Consideration"] = wk_co
    for yr in YEARS:
        mask = (decomp_tmp["year"] == yr) & decomp_tmp["h1"]
        h1_brand_funnel_yr_incr[(brand, "Awareness", yr)] = decomp_tmp.loc[mask, "incr_Awareness"].sum()
        h1_brand_funnel_yr_incr[(brand, "Consideration", yr)] = decomp_tmp.loc[mask, "incr_Consideration"].sum()

h1_tier_rev = {}
for brand, tiers in brand_tiers.items():
    for funnel in FUNNELS:
        for yr in YEARS:
            yr_incr = h1_brand_funnel_yr_incr[(brand, funnel, yr)]
            yr_total_eff = max(
                sum(h1_eff_spend.get((brand, funnel, t, yr), 0.0) for t in tiers),
                1e-9
            )
            for tier in tiers:
                w = h1_eff_spend.get((brand, funnel, tier, yr), 0.0) / yr_total_eff
                h1_tier_rev[(brand, funnel, tier, yr)] = yr_incr * w

def sp_h1(brand, funnel, tier, yr):
    return h1_eff_spend.get((brand, funnel, tier, yr), 0.0) / 1e6

def rv_h1(brand, funnel, tier, yr):
    return h1_tier_rev.get((brand, funnel, tier, yr), 0.0) / 1e6

for brand in BRANDS:
    tiers = brand_tiers[brand]
    for tier in tiers:
        x_base  = np.arange(len(YEARS))
        bar_w   = 0.55

        fig, ax1 = plt.subplots(figsize=(11, 7))
        fig.patch.set_facecolor(WHITE)

        # ── Stacked bars: Spend by funnel per year ─────────────────────────
        bottoms = np.zeros(len(YEARS))
        for funnel in FUNNELS:
            vals = np.array([sp(brand, funnel, tier, yr) for yr in YEARS])
            ax1.bar(x_base, vals, bar_w, bottom=bottoms,
                    color=FUNNEL_COLOR[funnel], alpha=0.85,
                    edgecolor=WHITE, lw=0.5, label=funnel)
            bottoms += vals

        for i, total in enumerate(bottoms):
            if total > 0:
                ax1.annotate(f"${total:.2f}M",
                             xy=(x_base[i], total),
                             xytext=(0, 5), textcoords="offset points",
                             ha="center", fontsize=14, color=CPX_DGREY, fontweight="bold")

        ax1.set_ylabel("Media Spend ($M)", fontsize=14, fontweight="bold", color=CPX_DGREY)
        ax1.set_xticks(x_base)
        ax1.set_xticklabels([YEAR_LABELS[yr] for yr in YEARS],
                            fontsize=13, fontweight="bold", rotation=10, ha="right")
        plt.setp(ax1.get_yticklabels(), fontsize=14, fontweight="bold")
        ax1.yaxis.set_major_formatter(mticker.FuncFormatter(lambda v, _: f"${v:.2f}M"))
        ax1.spines["top"].set_visible(False)
        ax1.set_axisbelow(True)
        ax1.grid(axis="y", alpha=0.15, color="#CCCCCC")
        ax1.set_ylim(0, bottoms.max() * 1.4)

        # ── ROI curve on secondary y-axis (full-year) ──────────────────────
        ax2 = ax1.twinx()
        roi_vals = []
        for yr in YEARS:
            t_spend = sum(sp(brand, f, tier, yr) for f in FUNNELS)
            t_rev   = sum(rv(brand, f, tier, yr) for f in FUNNELS)
            roi_vals.append(t_rev / t_spend if t_spend > 0 else 0.0)

        ax2.plot(x_base, roi_vals,
                 color="black", linewidth=3, marker="D",
                 markersize=12, markeredgecolor=WHITE, markeredgewidth=2,
                 zorder=5, label="ROI")

        for i, roi in enumerate(roi_vals):
            if roi > 0:
                ax2.annotate(f"{roi:.2f}×",
                             xy=(x_base[i], roi),
                             xytext=(0, 12), textcoords="offset points",
                             ha="center", fontsize=14, fontweight="bold",
                             color="black")

        # ── H1 ROI curve — same-period comparison across all years ─────────
        # 2026 is already H1-only, so its H1 value equals its full-year value.
        h1_roi_vals = []
        for yr in YEARS:
            if yr == max(YEARS):
                h1_roi_vals.append(roi_vals[YEARS.index(yr)])
                continue
            t_spend_h1 = sum(sp_h1(brand, f, tier, yr) for f in FUNNELS)
            t_rev_h1   = sum(rv_h1(brand, f, tier, yr) for f in FUNNELS)
            h1_roi_vals.append(t_rev_h1 / t_spend_h1 if t_spend_h1 > 0 else 0.0)

        ax2.plot(x_base, h1_roi_vals,
                 color=CPX_GOLD, linewidth=2.5, linestyle="--", marker="o",
                 markersize=10, markeredgecolor=WHITE, markeredgewidth=1.5,
                 zorder=4, label="H1 ROI")

        for i, roi in enumerate(h1_roi_vals):
            if roi > 0 and i != len(YEARS) - 1:  # last year's H1 == full-year, already labeled
                ax2.annotate(f"{roi:.2f}×",
                             xy=(x_base[i], roi),
                             xytext=(0, -18), textcoords="offset points",
                             ha="center", fontsize=12, fontweight="bold",
                             color=CPX_GOLD)

        ax2.set_ylabel("ROI (Rev / Spend)", fontsize=14, fontweight="bold", color=CPX_DGREY)
        plt.setp(ax2.get_yticklabels(), fontsize=14, fontweight="bold")
        ax2.spines["top"].set_visible(False)
        y2_vals = [v for v in roi_vals + h1_roi_vals if v > 0]
        if y2_vals:
            y2_min, y2_max = min(y2_vals), max(y2_vals)
            pad = (y2_max - y2_min) * 0.4 if y2_max > y2_min else y2_max * 0.3
            ax2.set_ylim(max(0, y2_min - pad), y2_max + pad)
        else:
            ax2.set_ylim(0, 5)

        # ── Legend ─────────────────────────────────────────────────────────
        funnel_handles = [mpatches.Patch(color=FUNNEL_COLOR[f], label=f"{f} Spend")
                          for f in FUNNELS]
        roi_handle = mlines.Line2D([], [], color="black", linewidth=3,
                                   marker="D", markersize=10, label="ROI")
        h1_roi_handle = mlines.Line2D([], [], color=CPX_GOLD, linewidth=2.5, linestyle="--",
                                      marker="o", markersize=8, label="H1 ROI")
        ax1.legend(handles=funnel_handles + [roi_handle, h1_roi_handle],
                   fontsize=14, loc="upper left", framealpha=0.9,
                   edgecolor="#CCCCCC")

        ax1.set_title(f"{brand} {tier} — Spend & ROI by Year",
                      fontsize=14, fontweight="bold", color=CPX_NAVY)
        plt.tight_layout()
        save_fig(fig, f"lbe_tier_roi_{brand.lower()}_{tier.lower()}_by_year.png")
        display(fig)
        plt.close(fig)

        # ── Summary table ───────────────────────────────────────────────────
        tbl_rows = []
        for i, yr in enumerate(YEARS):
            t_spend = sum(sp(brand, f, tier, yr) for f in FUNNELS)
            t_rev   = sum(rv(brand, f, tier, yr) for f in FUNNELS)
            roi     = t_rev / t_spend if t_spend > 0 else 0.0
            tbl_rows.append({
                "Year": YEAR_LABELS[yr],
                "Awareness Spend ($M)": round(sp(brand, "Awareness", tier, yr), 3),
                "Consideration Spend ($M)": round(sp(brand, "Consideration", tier, yr), 3),
                "Total Spend ($M)": round(t_spend, 3),
                "Total Rev ($M)": round(t_rev, 3),
                "ROI": f"{roi:.2f}×" if t_spend > 0 else "—",
                "H1 ROI": f"{h1_roi_vals[i]:.2f}×" if h1_roi_vals[i] > 0 else "—",
            })
        print(f"\n{'═'*60}\n  {brand} {tier} — Spend & ROI by Year\n{'═'*60}")
        ipy_display(pd.DataFrame(tbl_rows))

In [0]:
# ═══════════════════════════════════════════════════════════════════════════════
# WHY TWO MODELS? — Revenue Mechanism Decomposition
# Shows stakeholders how each funnel drives revenue through different paths:
#   Awareness  → gets people in the door (VOLUME = more transactions)
#   Consideration → gets them to spend more (YIELD = higher rev per visit)
# Revenue = Transactions × Rev/Transaction, so both paths lead to $$.
# ═══════════════════════════════════════════════════════════════════════════════

# ── Decompose each funnel's revenue into Volume vs Yield paths ───────────────
# Volume path: RPT × ΔTXN (more visits at current spend-per-visit)
# Yield path:  ΔRPT × TXN (higher spend per visit with current traffic)
decomp = {}
for brand in BRANDS:
    r = results[brand]
    pred_rpt = r["pred_rpt"]
    pred_txn = r["pred_txn"]
    log_pred_rpt = r["log_pred_rpt"]
    log_pred_txn = r["log_pred_txn"]

    # Awareness decomposition
    aw_volume = (pred_rpt * (pred_txn - np.exp(log_pred_txn - r["m_aw_txn"]))).sum()
    aw_yield  = ((pred_rpt - np.exp(log_pred_rpt - r["m_aw_rpt"])) * pred_txn).sum()
    aw_total  = aw_volume + aw_yield

    # Consideration decomposition
    co_volume = (pred_rpt * (pred_txn - np.exp(log_pred_txn - r["m_co_txn"]))).sum()
    co_yield  = ((pred_rpt - np.exp(log_pred_rpt - r["m_co_rpt"])) * pred_txn).sum()
    co_total  = co_volume + co_yield

    decomp[brand] = {
        "Awareness":     {"Volume": aw_volume, "Yield": aw_yield, "Total": aw_total},
        "Consideration": {"Volume": co_volume, "Yield": co_yield, "Total": co_total},
    }

# ═══ FIGURE 1: Mechanism split — horizontal stacked bars (stakeholder-friendly) ═══
fig, axes = plt.subplots(1, 2, figsize=(16, 7))
fig.patch.set_facecolor(WHITE)

for ci, brand in enumerate(BRANDS):
    ax = axes[ci]
    funnels = ["Awareness", "Consideration"]
    y_pos   = np.arange(len(funnels))
    bar_h   = 0.5

    vol_pcts = []
    yld_pcts = []
    for funnel in funnels:
        total = decomp[brand][funnel]["Total"]
        vol_pct = decomp[brand][funnel]["Volume"] / total * 100 if total > 0 else 0
        yld_pct = decomp[brand][funnel]["Yield"] / total * 100 if total > 0 else 0
        vol_pcts.append(vol_pct)
        yld_pcts.append(yld_pct)

    bars_vol = ax.barh(y_pos, vol_pcts, bar_h, color=CPX_NAVY, label="Volume Path\n(more visits)")
    bars_yld = ax.barh(y_pos, yld_pcts, bar_h, left=vol_pcts, color=CPX_RED, label="Yield Path\n(higher spend/visit)")

    for i in range(len(funnels)):
        if vol_pcts[i] > 15:
            ax.text(vol_pcts[i] / 2, y_pos[i], f"{vol_pcts[i]:.0f}%\nVolume",
                    ha="center", va="center", fontsize=14, fontweight="bold", color=WHITE)
        if yld_pcts[i] > 15:
            ax.text(vol_pcts[i] + yld_pcts[i] / 2, y_pos[i], f"{yld_pcts[i]:.0f}%\nYield",
                    ha="center", va="center", fontsize=14, fontweight="bold", color=WHITE)

    ax.set_yticks(y_pos)
    ax.set_yticklabels(funnels, fontsize=14, fontweight="bold")
    ax.set_xlabel("% of Revenue Contribution", fontsize=14, fontweight="bold")
    ax.set_xlim(0, 100)
    plt.setp(ax.get_xticklabels(), fontsize=14, fontweight="bold")
    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)
    ax.set_title(f"{brand}", fontsize=14, fontweight="bold", color=CPX_NAVY)

handles, labels = axes[0].get_legend_handles_labels()
fig.legend(handles, labels, fontsize=14, loc="lower center", ncol=2,
           framealpha=0.9, edgecolor="#CCCCCC", bbox_to_anchor=(0.5, -0.08))

fig.suptitle("How Each Funnel Drives Revenue — Two Different Mechanisms\n"
             "Revenue = Transactions × Rev per Transaction",
             fontsize=14, fontweight="bold", color=CPX_NAVY, y=1.0)
plt.tight_layout()
plt.subplots_adjust(bottom=0.15)
save_fig(fig, "lbe_why_two_models_mechanism_split.png")
display(fig)
plt.close(fig)

# ═══ FIGURE 2: Dollar contribution by mechanism (absolute $M) ═════════════════
fig2, ax2 = plt.subplots(figsize=(12, 7))
fig2.patch.set_facecolor(WHITE)

labels = []
vol_vals = []
yld_vals = []
for brand in BRANDS:
    for funnel in ["Awareness", "Consideration"]:
        labels.append(f"{brand}\n{funnel}")
        vol_vals.append(decomp[brand][funnel]["Volume"] / 1e6)
        yld_vals.append(decomp[brand][funnel]["Yield"] / 1e6)

x = np.arange(len(labels))
bar_w = 0.55

ax2.bar(x, vol_vals, bar_w, color=CPX_NAVY, label="Volume Path (RPT × ΔTxn)")
ax2.bar(x, yld_vals, bar_w, bottom=vol_vals, color=CPX_RED, label="Yield Path (ΔRPT × Txn)")

for i in range(len(labels)):
    total = vol_vals[i] + yld_vals[i]
    ax2.annotate(f"${total:.1f}M", xy=(x[i], total),
                 xytext=(0, 5), textcoords="offset points",
                 ha="center", fontsize=14, fontweight="bold", color=CPX_DGREY)
    pct_vol = vol_vals[i] / total * 100
    pct_yld = yld_vals[i] / total * 100
    if vol_vals[i] > 0.3:
        ax2.text(x[i], vol_vals[i] / 2, f"{pct_vol:.0f}%",
                 ha="center", va="center", fontsize=14, fontweight="bold", color=WHITE)
    if yld_vals[i] > 0.3:
        ax2.text(x[i], vol_vals[i] + yld_vals[i] / 2, f"{pct_yld:.0f}%",
                 ha="center", va="center", fontsize=14, fontweight="bold", color=WHITE)

ax2.set_xticks(x)
ax2.set_xticklabels(labels, fontsize=14, fontweight="bold")
ax2.set_ylabel("Media-Attributed Revenue ($M)", fontsize=14, fontweight="bold")
plt.setp(ax2.get_yticklabels(), fontsize=14, fontweight="bold")
ax2.spines["top"].set_visible(False)
ax2.spines["right"].set_visible(False)
ax2.legend(fontsize=14, loc="upper right", framealpha=0.9)
ax2.set_title("Revenue Attribution by Mechanism — Why We Need Two Models",
              fontsize=14, fontweight="bold", color=CPX_NAVY)
ax2.grid(axis="y", alpha=0.15)
plt.tight_layout()
save_fig(fig2, "lbe_why_two_models_dollar_split.png")
display(fig2)
plt.close(fig2)

# ── Summary table for stakeholders ───────────────────────────────────────────
print("\n" + "═"*75)
print("  WHY TWO MODELS? — Summary")
print("═"*75)
rows = []
for brand in BRANDS:
    for funnel in ["Awareness", "Consideration"]:
        d_item = decomp[brand][funnel]
        total = d_item["Total"]
        rows.append({
            "Brand": brand,
            "Funnel": funnel,
            "Total Revenue ($M)": f"${total/1e6:.1f}M",
            "Via Volume (more visits)": f"{d_item['Volume']/total*100:.0f}%" if total > 0 else "—",
            "Via Yield (higher spend)": f"{d_item['Yield']/total*100:.0f}%" if total > 0 else "—",
            "Primary Target": "Transactions" if d_item['Volume'] > d_item['Yield'] else "Rev/Transaction",
        })
ipy_display(pd.DataFrame(rows))

print("\n\U0001F4A1 Key Takeaway for Stakeholders:")
print("   Awareness and Consideration grow revenue through DIFFERENT mechanisms.")
print("   A single-model approach conflates these — modeling log(Revenue) directly")
print("   fails validation (val R² < 0 for both brands) while the dual-target model")
print("   generalizes cleanly. The dual-target model separates:")
print("     • Model A: log(Rev/Txn) — how media lifts YIELD (spend per visit)")
print("     • Model B: log(Txn)     — how media lifts VOLUME (foot traffic)")
print("   Then multiplies them back: Revenue = Rev/Txn × Transactions")

In [0]:
# ═══════════════════════════════════════════════════════════════════════════════
# LOG-SPACE PARTIAL CORRELATIONS
# The V5e model estimates: log(Txn) = baseline + β_aw*f(aw) + controls
#                          log(RPT) = baseline + β_co*f(co) + controls
# So the correct correlation space is log-transformed outcomes.
# This matches what the Bayesian model actually sees.
# ═══════════════════════════════════════════════════════════════════════════════
from scipy.stats import pearsonr

# ── Ensure is_long_weekend is numeric (defensive; data cell may not have run) ─
if d["is_long_weekend"].dtype == object:
    d["is_long_weekend"] = (d["is_long_weekend"].astype(str) == "Y").astype(int)

# ── Helpers (rebuilt inline) ─────────────────────────────────────────────────
def _residualize(y, X):
    """Return residuals of y regressed on X via OLS. Adds intercept column."""
    X_aug = np.column_stack([np.ones(len(X)), X])
    beta, *_ = np.linalg.lstsq(X_aug, y, rcond=None)
    return y - X_aug @ beta

def _scatter_partial(ax, x, y, title, xlab, ylab, pt_color, line_color):
    """Partial-r scatter with OLS fit line and r annotation."""
    ax.scatter(x, y, s=30, color=pt_color, alpha=0.5, edgecolors="none")
    slope, intercept = np.polyfit(x, y, 1)
    xs = np.linspace(x.min(), x.max(), 100)
    ax.plot(xs, slope * xs + intercept, color=line_color, linewidth=2)
    r, p = pearsonr(x, y)
    ax.annotate(f"r = {r:+.3f}\np = {p:.3g}\nn = {len(x)}",
                xy=(0.03, 0.97), xycoords="axes fraction",
                ha="left", va="top", fontsize=11, fontweight="bold",
                bbox=dict(boxstyle="round,pad=0.3", fc=WHITE, alpha=0.85, ec="none"))
    ax.set_title(title, fontsize=12, fontweight="bold", color=CPX_NAVY)
    ax.set_xlabel(xlab, fontsize=11, fontweight="bold")
    ax.set_ylabel(ylab, fontsize=11, fontweight="bold")
    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)

# ── Standardized control matrix (same as model's Xz) ─────────────────────────
_X_raw = d[CONTROLS].values.astype(float)
_mu    = _X_raw.mean(0)
_sd    = _X_raw.std(0); _sd[_sd == 0] = 1.0
X_ctrl = np.nan_to_num((_X_raw - _mu) / _sd)

# ── Pre-compute LINEAR-space series (needed for comparison table) ────────────
_series = {}
for brand in BRANDS:
    aw_raw = d[f"{brand}_Awareness_spend"].values.astype(float)
    co_raw = d[f"{brand}_Consideration_spend"].values.astype(float)
    txn    = d[f"{brand}_Txns"].values.astype(float)
    rpt    = d[f"{brand}_Rev"].values / np.maximum(txn, 1)

    aw_ads = adstock_np(aw_raw, results[brand]["adstock_aw_txn"])
    co_ads = adstock_np(co_raw, results[brand]["adstock_co_rpt"])

    _series[brand] = {
        "aw_resid":  _residualize(aw_ads, X_ctrl),
        "co_resid":  _residualize(co_ads, X_ctrl),
        "txn_resid": _residualize(txn,    X_ctrl),
        "rpt_resid": _residualize(rpt,    X_ctrl),
    }

# ── Pre-compute LOG-SPACE residualized series for both brands ────────────────
_series_log = {}
for brand in BRANDS:
    aw_raw = d[f"{brand}_Awareness_spend"].values.astype(float)
    co_raw = d[f"{brand}_Consideration_spend"].values.astype(float)
    txn    = d[f"{brand}_Txns"].values.astype(float)
    rpt    = d[f"{brand}_Rev"].values / np.maximum(txn, 1)

    log_txn = np.log(np.maximum(txn, 1))
    log_rpt = np.log(np.maximum(rpt, 1))

    aw_ads = adstock_np(aw_raw, results[brand]["adstock_aw_txn"])
    co_ads = adstock_np(co_raw, results[brand]["adstock_co_rpt"])

    log_aw = np.log1p(aw_ads)
    log_co = np.log1p(co_ads)

    _series_log[brand] = {
        "aw_resid":  _residualize(log_aw,  X_ctrl),
        "co_resid":  _residualize(log_co,  X_ctrl),
        "txn_resid": _residualize(log_txn, X_ctrl),
        "rpt_resid": _residualize(log_rpt, X_ctrl),
    }

# ═══ FIGURE 1: AWARENESS in log-space ══════════════════════════════════════════
fig1, axes1 = plt.subplots(2, 2, figsize=(14, 10))
fig1.patch.set_facecolor(WHITE)

for ci, brand in enumerate(BRANDS):
    s = _series_log[brand]
    _scatter_partial(axes1[0, ci], s["aw_resid"], s["txn_resid"],
                     f"{brand} — log(Awareness) → log(Transactions)",
                     "log(Awareness) residualized", "log(Txn) residualized",
                     CPX_NAVY, CPX_RED)
    _scatter_partial(axes1[1, ci], s["aw_resid"], s["rpt_resid"],
                     f"{brand} — log(Awareness) → log(Rev/Txn)",
                     "log(Awareness) residualized", "log(RPT) residualized",
                     CPX_LGREY, CPX_DGREY)

fig1.suptitle("AWARENESS — Log-Space Partial Correlations (matching V5e model specification)\n"
              "Top: expected target log(Txn)  |  Bottom: non-target log(RPT)",
              fontsize=14, fontweight="bold", color=CPX_NAVY, y=1.02)
plt.tight_layout()
save_fig(fig1, "lbe_awareness_logspace_partial_correlation.png")
display(fig1)
plt.close(fig1)

# ═══ FIGURE 2: CONSIDERATION in log-space ═══════════════════════════════════════
fig2, axes2 = plt.subplots(2, 2, figsize=(14, 10))
fig2.patch.set_facecolor(WHITE)

for ci, brand in enumerate(BRANDS):
    s = _series_log[brand]
    _scatter_partial(axes2[0, ci], s["co_resid"], s["rpt_resid"],
                     f"{brand} — log(Consideration) → log(Rev/Txn)",
                     "log(Consideration) residualized", "log(RPT) residualized",
                     CPX_RED, CPX_NAVY)
    _scatter_partial(axes2[1, ci], s["co_resid"], s["txn_resid"],
                     f"{brand} — log(Consideration) → log(Transactions)",
                     "log(Consideration) residualized", "log(Txn) residualized",
                     CPX_LGREY, CPX_DGREY)

fig2.suptitle("CONSIDERATION — Log-Space Partial Correlations (matching V5e model specification)\n"
              "Top: expected target log(RPT)  |  Bottom: non-target log(Txn)",
              fontsize=14, fontweight="bold", color=CPX_NAVY, y=1.02)
plt.tight_layout()
save_fig(fig2, "lbe_consideration_logspace_partial_correlation.png")
display(fig2)
plt.close(fig2)

# ── Summary comparison table ────────────────────────────────────────────────────
print("\n" + "═"*75)
print("  COMPARISON: Linear-space vs Log-space partial r")
print("═"*75)
rows = []
for brand in BRANDS:
    s_lin = _series[brand]
    s_log = _series_log[brand]
    r_lin, _ = pearsonr(s_lin["aw_resid"], s_lin["txn_resid"])
    r_log, _ = pearsonr(s_log["aw_resid"], s_log["txn_resid"])
    rows.append({"Brand": brand, "Relationship": "Awareness → Txn",
                 "Linear r": f"{r_lin:.3f}", "Log-space r": f"{r_log:.3f}", "Expected": "Strong +"})
    r_lin, _ = pearsonr(s_lin["aw_resid"], s_lin["rpt_resid"])
    r_log, _ = pearsonr(s_log["aw_resid"], s_log["rpt_resid"])
    rows.append({"Brand": brand, "Relationship": "Awareness → RPT",
                 "Linear r": f"{r_lin:.3f}", "Log-space r": f"{r_log:.3f}", "Expected": "Weak / 0"})
    r_lin, _ = pearsonr(s_lin["co_resid"], s_lin["rpt_resid"])
    r_log, _ = pearsonr(s_log["co_resid"], s_log["rpt_resid"])
    rows.append({"Brand": brand, "Relationship": "Consideration → RPT",
                 "Linear r": f"{r_lin:.3f}", "Log-space r": f"{r_log:.3f}", "Expected": "Strong +"})
    r_lin, _ = pearsonr(s_lin["co_resid"], s_lin["txn_resid"])
    r_log, _ = pearsonr(s_log["co_resid"], s_log["txn_resid"])
    rows.append({"Brand": brand, "Relationship": "Consideration → Txn",
                 "Linear r": f"{r_lin:.3f}", "Log-space r": f"{r_log:.3f}", "Expected": "Weak / 0"})

ipy_display(pd.DataFrame(rows))

print("\nKey insight: The V5e model operates in log-space, so log(media) ↔ log(outcome)")
print("captures the multiplicative relationship the Bayesian model estimates.")
print("Linear-space partial r underestimates the signal because it misses the")
print("diminishing-returns (concave) structure the model exploits.")

In [0]:
# ━━ LBE BUDGET SCENARIO — DEPENDENCIES ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
# Derives an LBE-level single-target response model from V5e dual-target results.
# Uses Hill saturation with per-brand anchors + finite-L steady-state adstock,
# matching the fitted V5e architecture. Calibrated so that at 2025 observed spend,
# the model reproduces the 2025-year-specific media contribution from decomp_df
# (i.e. matches the annual revenue decomposition chart).

# ── 1. df_lbe_all: weekly total Awareness & Consideration spend ───────────────
df_lbe_all = df_all[["week_date", "Awareness", "Consideration"]].copy()

# ── 2. Normalization maxes (weekly) ───────────────────────────────────────────
lbe_aw_max = float(df_lbe_all["Awareness"].max())
lbe_co_max = float(df_lbe_all["Consideration"].max())

# ── 3. Log-revenue statistics (LBE = TRR + PDM) ──────────────────────────────
_lbe_weekly_rev = d["TRR_Rev"] + d["PDM_Rev"]
_log_rev = np.log(_lbe_weekly_rev)
lbe_y_mean = float(_log_rev.mean())
lbe_y_std  = float(_log_rev.std())

# ── 4. Revenue-weighted brand aggregation of per-brand params ─────────────────
_trr_w = results["TRR"]["aw_incr_rev"] + results["TRR"]["co_incr_rev"]
_pdm_w = results["PDM"]["aw_incr_rev"] + results["PDM"]["co_incr_rev"]
_w_trr = _trr_w / (_trr_w + _pdm_w)
_w_pdm = _pdm_w / (_trr_w + _pdm_w)

_aw_decay = float(
    _w_trr * (results["TRR"]["adstock_aw_rpt"] + results["TRR"]["adstock_aw_txn"]) / 2
    + _w_pdm * (results["PDM"]["adstock_aw_rpt"] + results["PDM"]["adstock_aw_txn"]) / 2
)
_co_decay = float(
    _w_trr * (results["TRR"]["adstock_co_rpt"] + results["TRR"]["adstock_co_txn"]) / 2
    + _w_pdm * (results["PDM"]["adstock_co_rpt"] + results["PDM"]["adstock_co_txn"]) / 2
)

_aw_anchor = float(_w_trr * results["TRR"]["anchor_aw"] + _w_pdm * results["PDM"]["anchor_aw"])
_co_anchor = float(_w_trr * results["TRR"]["anchor_co"] + _w_pdm * results["PDM"]["anchor_co"])

# ── 5. Finite-L steady-state adstock + Hill helpers ───────────────────────────
def _ss_adstock(w_norm, alpha, L=ADSTOCK_L):
    if alpha >= 1.0:
        return w_norm * L
    return w_norm * (1 - alpha**L) / (1 - alpha)

def _hill(x, half):
    return x / (x + half)

# ── 6. Calibration target: 2025-only weekly counterfactual (from decomp_df) ──
# This matches the annual revenue decomposition chart exactly.
if "decomp_df" not in dir():
    # Fallback: rebuild the weekly counterfactual (only if cell 16 hasn't run)
    _wk_aw_tmp = np.zeros(len(d)); _wk_co_tmp = np.zeros(len(d))
    for _b in ["TRR", "PDM"]:
        _r = results[_b]
        _wk_aw_tmp += (
            (_r["pred_rpt"] - np.exp(_r["log_pred_rpt"] - _r["m_aw_rpt"])) * _r["pred_txn"]
            + _r["pred_rpt"] * (_r["pred_txn"] - np.exp(_r["log_pred_txn"] - _r["m_aw_txn"]))
        )
        _wk_co_tmp += (
            (_r["pred_rpt"] - np.exp(_r["log_pred_rpt"] - _r["m_co_rpt"])) * _r["pred_txn"]
            + _r["pred_rpt"] * (_r["pred_txn"] - np.exp(_r["log_pred_txn"] - _r["m_co_txn"]))
        )
    decomp_df = d[["week_date"]].copy()
    decomp_df["year"] = decomp_df["week_date"].dt.year
    decomp_df["media_Awareness"]     = _wk_aw_tmp
    decomp_df["media_Consideration"] = _wk_co_tmp

_dec_2025 = decomp_df[decomp_df["year"] == 2025]
_total_aw_incr_2025 = float(_dec_2025["media_Awareness"].sum())
_total_co_incr_2025 = float(_dec_2025["media_Consideration"].sum())
_total_media_2025   = _total_aw_incr_2025 + _total_co_incr_2025

# ── 7. Calibrate beta so model reproduces 2025 media contribution at 2025 spend
_df_2025 = df_lbe_all[df_lbe_all["week_date"].dt.year == 2025]
_aw_weekly_2025 = float(_df_2025["Awareness"].mean())
_co_weekly_2025 = float(_df_2025["Consideration"].mean())

_ads_aw_2025 = _ss_adstock(_aw_weekly_2025 / lbe_aw_max, _aw_decay)
_ads_co_2025 = _ss_adstock(_co_weekly_2025 / lbe_co_max, _co_decay)
_sat_aw_2025 = _hill(_ads_aw_2025, _aw_anchor)
_sat_co_2025 = _hill(_ads_co_2025, _co_anchor)

_baseline_rev_annual = np.exp(lbe_y_mean) * 52
_mu_total = np.log(1 + _total_media_2025 / _baseline_rev_annual) / lbe_y_std
_mu_aw    = _mu_total * (_total_aw_incr_2025 / _total_media_2025)
_mu_co    = _mu_total * (_total_co_incr_2025 / _total_media_2025)

_beta_aw = _mu_aw / _sat_aw_2025
_beta_co = _mu_co / _sat_co_2025

# ── 8. Construct lbe_draws (point-estimate posterior, repeated DRAWS times) ───
_n = DRAWS if "DRAWS" in dir() else 1000
lbe_draws = {
    "aw_decay":   np.full(_n, _aw_decay),
    "co_decay":   np.full(_n, _co_decay),
    "aw_anchor":  np.full(_n, _aw_anchor),
    "co_anchor":  np.full(_n, _co_anchor),
    "beta_aw":    np.full(_n, _beta_aw),
    "beta_co":    np.full(_n, _beta_co),
}

# ── 9. Helper for downstream cells: compute annual LBE media revenue ─────────
def lbe_media_rev(aw_annual, co_annual):
    """Given annual Aw & Co budget ($ total), return annual LBE incremental revenue ($).
    Uses steady-state weekly, finite-L adstock, Hill saturation with LBE-level anchors.
    Calibrated to 2025 year-specific decomp."""
    w_aw = aw_annual / 52.0
    w_co = co_annual / 52.0
    ads_aw = _ss_adstock(w_aw / lbe_aw_max, _aw_decay)
    ads_co = _ss_adstock(w_co / lbe_co_max, _co_decay)
    sat_aw = _hill(ads_aw, _aw_anchor)
    sat_co = _hill(ads_co, _co_anchor)
    mu     = _beta_aw * sat_aw + _beta_co * sat_co
    weekly_rev_with_media = np.exp(lbe_y_mean + mu * lbe_y_std)
    weekly_rev_baseline   = np.exp(lbe_y_mean)
    return (weekly_rev_with_media - weekly_rev_baseline) * 52

print("LBE Budget Scenario — calibrated to 2025 year-specific decomp")
print(f"  lbe_aw_max:       {lbe_aw_max:>12,.0f}")
print(f"  lbe_co_max:       {lbe_co_max:>12,.0f}")
print(f"  lbe_y_mean:       {lbe_y_mean:>12.6f}")
print(f"  lbe_y_std:        {lbe_y_std:>12.6f}")
print(f"  aw_decay:         {_aw_decay:>12.4f}   (finite-L={ADSTOCK_L})")
print(f"  co_decay:         {_co_decay:>12.4f}   (finite-L={ADSTOCK_L})")
print(f"  aw_anchor:        {_aw_anchor:>12.4f}   (rev-weighted TRR+PDM)")
print(f"  co_anchor:        {_co_anchor:>12.4f}   (rev-weighted TRR+PDM)")
print(f"  beta_aw:          {_beta_aw:>12.4f}")
print(f"  beta_co:          {_beta_co:>12.4f}")
print(f"  2025 target Aw:   ${_total_aw_incr_2025/1e6:>8.2f}M")
print(f"  2025 target Co:   ${_total_co_incr_2025/1e6:>8.2f}M")
print(f"  2025 target Tot:  ${_total_media_2025/1e6:>8.2f}M")
print(f"  Draws:            {_n}")
print(f"\n  Verification at 2025 spend:")
_rev_v = lbe_media_rev(_aw_weekly_2025 * 52, _co_weekly_2025 * 52)
print(f"    Reproduced media contribution: ${_rev_v/1e6:.2f}M (target: ${_total_media_2025/1e6:.2f}M) ✓")

In [0]:
# ── Load posterior InferenceData (needed for posterior-based scenario cells) ──
_local_tmp = "/local_disk0/tmp_idata"
os.makedirs(_local_tmp, exist_ok=True)

for brand in ["TRR", "PDM"]:
    for tgt in ["rpt", "txn"]:
        local_path = f"{_local_tmp}/{brand.lower()}_idata_{tgt}.nc"
        with open(f"{ARTIFACT_PATH}_{brand.lower()}_idata_{tgt}.nc", "rb") as _src, open(local_path, "wb") as _dst:
            _dst.write(_src.read())
        results[brand][f"idata_{tgt}"] = az.from_netcdf(local_path)
print("✓ Posterior idata loaded (idata_rpt / idata_txn per brand)")

In [0]:
# ━━ LBE BUDGET SCENARIO — DEPENDENCIES (POSTERIOR-BASED) ━━━━━━━━━━━━━━━━━━━━━━━━
# Scenarios now flow through the fitted V5e posteriors — NOT a re-derived
# single-target LBE curve. For a given annual (Aw, Co) budget per brand we:
#   1. scale that brand's ACTUAL 2025 weekly spend shape by (scenario/2025)
#   2. push every posterior draw through the exact V5e reconstruction
#      (adstock → Hill w/ fixed anchor → beta → incr RPT & incr Txns → revenue)
#   3. sum TRR + PDM per draw → LBE incremental-revenue distribution
# CIs are genuine posterior credible intervals, dual-target, per-brand.

# ── 1. Per-brand 2025 weekly spend shapes + normalization (fitted space) ──────
_mask_2025 = d["week_date"].dt.year == 2025
SCEN_BRANDS = ["TRR", "PDM"]

scen_ctx = {}   # per-brand cached context for the scenario reconstruction
for brand in SCEN_BRANDS:
    M = d[[f"{brand}_Awareness_spend", f"{brand}_Consideration_spend"]].values.astype(float)
    X = d[CONTROLS].values.astype(float)
    n_tr = int(len(d) * TRAIN_FRAC)
    m_s = M[:n_tr].max(0); m_s[m_s == 0] = 1.0          # SAME normalizer as fit
    x_mu, x_sd = X[:n_tr].mean(0), X[:n_tr].std(0); x_sd[x_sd == 0] = 1.0
    Xz = np.nan_to_num((X - x_mu) / x_sd)

    # 2025 weekly spend shape (actual $ per week, this brand)
    aw_wk_2025 = d.loc[_mask_2025, f"{brand}_Awareness_spend"].values.astype(float)
    co_wk_2025 = d.loc[_mask_2025, f"{brand}_Consideration_spend"].values.astype(float)

    # avg_rpt/avg_txn: not persisted in artifacts.json (only in results.json),
    # so recompute directly from d — same definition as the original fit (cell 3).
    _rev = d[f"{brand}_Rev"].values.astype(float)
    _txn = d[f"{brand}_Txns"].values.astype(float)
    _rpt = np.where(_txn > 0, _rev / _txn, np.nan)
    _rpt_clean = np.where(~np.isnan(_rpt), _rpt, np.nanmedian(_rpt))
    avg_rpt = _rpt_clean.mean()
    avg_txn = _txn.mean()

    scen_ctx[brand] = {
        "m_s": m_s, "Xz": Xz,
        "aw_wk_2025": aw_wk_2025, "co_wk_2025": co_wk_2025,
        "aw_annual_2025": float(aw_wk_2025.sum()),
        "co_annual_2025": float(co_wk_2025.sum()),
        "anchor_aw": results[brand]["anchor_aw"],
        "anchor_co": results[brand]["anchor_co"],
        "avg_rpt":   avg_rpt,
        "avg_txn":   avg_txn,
    }

# LBE 2025 annual budget (for baseline reference / scenario multipliers)
lbe_aw_2025 = sum(scen_ctx[b]["aw_annual_2025"] for b in SCEN_BRANDS)
lbe_co_2025 = sum(scen_ctx[b]["co_annual_2025"] for b in SCEN_BRANDS)

# ── 2. Pull posterior draws once per brand/target (paired RPT & TXN) ───────────
_N_SCEN_DRAWS = 1000
def _stack_draws(idata, keys, n_draws=_N_SCEN_DRAWS, seed=42):
    po = idata.posterior
    if hasattr(po, "to_dataset"):
        po = po.to_dataset()
    po = po.stack(sample=("chain", "draw"))
    total = po.sizes["sample"]
    idx = np.random.default_rng(seed).choice(total, size=min(n_draws, total), replace=False)
    out = {}
    for k in keys:
        v = po[k].values
        out[k] = v[..., idx]      # last axis = sample
    return out, len(idx)

scen_draws = {}
for brand in SCEN_BRANDS:
    r = results[brand]
    d_rpt, nR = _stack_draws(r["idata_rpt"], ["adstock_aw","adstock_co","beta_media","beta_ctrl","intercept"])
    d_txn, nT = _stack_draws(r["idata_txn"], ["adstock_aw","adstock_co","beta_media","beta_ctrl","intercept"])
    scen_draws[brand] = {"rpt": d_rpt, "txn": d_txn, "n": min(nR, nT)}

# ── 3. Core: per-draw LBE incremental revenue for a scenario budget ───────────
def lbe_media_rev_draws(aw_annual, co_annual):
    """Return per-draw array of LBE incremental revenue ($) for an annual
    (Awareness, Consideration) LBE budget. Splits the LBE budget across brands
    by each brand's 2025 funnel-spend share, scales that brand's 2025 weekly
    shape, and runs every posterior draw through the V5e reconstruction."""
    lbe_incr = None
    for brand in SCEN_BRANDS:
        ctx = scen_ctx[brand]; dr = scen_draws[brand]; nD = dr["n"]

        # LBE budget → brand budget via 2025 funnel share
        aw_share = ctx["aw_annual_2025"] / lbe_aw_2025 if lbe_aw_2025 > 0 else 0.0
        co_share = ctx["co_annual_2025"] / lbe_co_2025 if lbe_co_2025 > 0 else 0.0
        aw_brand = aw_annual * aw_share
        co_brand = co_annual * co_share

        # Scenario weekly shape = 2025 shape × multiplier (approach a)
        aw_mult = aw_brand / ctx["aw_annual_2025"] if ctx["aw_annual_2025"] > 0 else 0.0
        co_mult = co_brand / ctx["co_annual_2025"] if ctx["co_annual_2025"] > 0 else 0.0
        aw_wk = ctx["aw_wk_2025"] * aw_mult
        co_wk = ctx["co_wk_2025"] * co_mult

        # Normalize into fitted space (same m_s as fit)
        aw_nz = aw_wk / ctx["m_s"][0]
        co_nz = co_wk / ctx["m_s"][1]

        # Controls at 2025 weeks (align with the weekly window we're scoring)
        Xz_2025 = ctx["Xz"][_mask_2025.values]

        h_aw, h_co = ctx["anchor_aw"], ctx["anchor_co"]
        avg_rpt, avg_txn = ctx["avg_rpt"], ctx["avg_txn"]

        aw_incr_rpt = np.empty(nD); co_incr_rpt = np.empty(nD)
        aw_incr_txn = np.empty(nD); co_incr_txn = np.empty(nD)

        for tgt, a_out in (("rpt", (aw_incr_rpt, co_incr_rpt)),
                           ("txn", (aw_incr_txn, co_incr_txn))):
            g = dr[tgt]
            for i in range(nD):
                a_aw = float(g["adstock_aw"][i]); a_co = float(g["adstock_co"][i])
                b = g["beta_media"][:, i]; gam = g["beta_ctrl"][:, i]; ic = float(g["intercept"][i])
                aw_full = adstock_np(aw_nz, a_aw)
                co_full = adstock_np(co_nz, a_co)
                aw_sat = aw_full / (aw_full + h_aw)
                co_sat = co_full / (co_full + h_co)
                m_aw = b[0] * aw_sat; m_co = b[1] * co_sat
                log_pred = ic + Xz_2025 @ gam + m_aw + m_co
                pred = np.exp(log_pred)
                a_out[0][i] = (pred - np.exp(log_pred - m_aw)).sum()
                a_out[1][i] = (pred - np.exp(log_pred - m_co)).sum()

        # Combine to revenue per draw (dual-target: RPT × Txns)
        aw_rev_d = aw_incr_txn * avg_rpt + avg_txn * aw_incr_rpt
        co_rev_d = co_incr_txn * avg_rpt + avg_txn * co_incr_rpt
        brand_rev_d = aw_rev_d + co_rev_d

        lbe_incr = brand_rev_d if lbe_incr is None else lbe_incr + brand_rev_d
    return lbe_incr

# ── 4. Point-estimate helper (posterior median) — drop-in for old lbe_media_rev
def lbe_media_rev(aw_annual, co_annual):
    """Posterior-median LBE incremental revenue ($) for an annual budget.
    Signature-compatible with the old deterministic helper so downstream
    cells (marginal ROI, tier split) keep working — but now posterior-backed."""
    return float(np.median(lbe_media_rev_draws(aw_annual, co_annual)))

# ── 5. Verification: baseline scenario must reconcile to 2025 decomp ──────────
_base_draws = lbe_media_rev_draws(lbe_aw_2025, lbe_co_2025)
_base_med = float(np.median(_base_draws))
_base_lo, _base_hi = np.percentile(_base_draws, [5, 95])

# 2025 decomp target (same source as before)
if "decomp_df" in dir():
    _dec_2025 = decomp_df[decomp_df["year"] == 2025]
    _decomp_target = float(_dec_2025["media_Awareness"].sum() + _dec_2025["media_Consideration"].sum())
else:
    _decomp_target = float("nan")

print("LBE Budget Scenario — POSTERIOR-BASED (per-brand V5e reconstruction)")
print(f"  Brands:              {SCEN_BRANDS}")
print(f"  Draws per brand:     {scen_draws['TRR']['n']}")
print(f"  LBE 2025 Aw budget:  ${lbe_aw_2025/1e6:>8.2f}M")
print(f"  LBE 2025 Co budget:  ${lbe_co_2025/1e6:>8.2f}M")
print(f"\n  Baseline (2025 spend) LBE incremental revenue:")
print(f"    posterior median:  ${_base_med/1e6:>8.2f}M")
print(f"    90% CI:            [${_base_lo/1e6:.2f}M, ${_base_hi/1e6:.2f}M]")
print(f"    2025 decomp target: ${_decomp_target/1e6:>8.2f}M")
if _decomp_target == _decomp_target:  # not nan
    _reconc = _base_med / _decomp_target if _decomp_target else float("nan")
    print(f"    reconciliation:     {_reconc:.2f}× (median / decomp target)")
    print(f"    → NOTE: posterior baseline need not equal the point-estimate decomp exactly;")
    print(f"      the decomp uses posterior-mean params, this uses the full draw distribution.")

In [0]:
# ━━ LBE 2027 SCENARIOS — TABLE, TIER SPEND/REVENUE PLOTS, AND UPSIDE LADDER ━━━━
# Standalone: only requires cell 33 ("DEPENDENCIES (POSTERIOR-BASED)") to have
# run first (needs lbe_aw_2025, lbe_co_2025, lbe_media_rev_draws, _decomp_target,
# decomp_df, FORECAST_2027, d, results, eff_spend, CPX_* colors, save_fig).
import matplotlib.patches as mpatches
import matplotlib.ticker as mticker
from matplotlib.lines import Line2D
import re

# ══════════════════════════════════════════════════════════════════════════════
# PART 1 — SCENARIO TABLE (posterior-based, 2027 projection)
# ══════════════════════════════════════════════════════════════════════════════
lbe_baseline_aw = lbe_aw_2025
lbe_baseline_co = lbe_co_2025

_base_draws_33 = lbe_media_rev_draws(lbe_baseline_aw, lbe_baseline_co)
_posterior_base_med = float(np.median(_base_draws_33))
recon_factor = _decomp_target / _posterior_base_med if _posterior_base_med > 0 else 1.0
lbe_baseline_rev = _decomp_target

def lbe_scenario_ci(aw_annual, co_annual):
    """Posterior point + 90% CI for a scenario budget, reconciled to decomp."""
    draws = lbe_media_rev_draws(aw_annual, co_annual) * recon_factor
    point   = float(np.median(draws))
    ci_low  = float(np.percentile(draws, 5))
    ci_high = float(np.percentile(draws, 95))
    return point, ci_low, ci_high

def confidence_label(ci_low, ci_high, point):
    if point == 0:
        return "Low"
    ci_width_pct = (ci_high - ci_low) / abs(point) * 100
    if ci_width_pct < 40:
        return "High"
    elif ci_width_pct < 80:
        return "Medium"
    else:
        return "Low"

_lbe_rev_2025 = float((d.loc[d["week_date"].dt.year == 2025, "TRR_Rev"].sum()
                       + d.loc[d["week_date"].dt.year == 2025, "PDM_Rev"].sum()))

_media_ratio_27   = lbe_baseline_rev / _lbe_rev_2025
proj_2027_total   = FORECAST_2027
proj_2027_media   = FORECAST_2027 * _media_ratio_27
proj_2027_nonmedia = proj_2027_total - proj_2027_media

_, lbe_base_ci_lo, lbe_base_ci_hi = lbe_scenario_ci(lbe_baseline_aw, lbe_baseline_co)

scenarios = []
scenarios.append({"Scenario": "2025 Baseline", "Type": "Baseline",
                  "aw": lbe_baseline_aw, "co": lbe_baseline_co})

for pct in [3, 5, 10, 20, 30]:
    factor = 1 + pct / 100
    scenarios.append({"Scenario": f"+{pct}% Budget Uplift", "Type": "Budget",
                      "aw": lbe_baseline_aw * factor, "co": lbe_baseline_co * factor})

shift = lbe_baseline_co * 0.20
scenarios.append({"Scenario": "20% Shift: Co -> Aw", "Type": "Reshuffle",
                  "aw": lbe_baseline_aw + shift, "co": lbe_baseline_co - shift})

for pct in [10, 20, 30]:
    factor = 1 + pct / 100
    aw_up = lbe_baseline_aw * factor
    co_up = lbe_baseline_co * factor
    shift_c = co_up * 0.20
    scenarios.append({"Scenario": f"+{pct}% + 20% Shift", "Type": "Combined",
                      "aw": aw_up + shift_c, "co": co_up - shift_c})

tbl_rows = []
for s in scenarios:
    rev, ci_lo, ci_hi = lbe_scenario_ci(s["aw"], s["co"])
    lift = rev - lbe_baseline_rev
    lift_pct = (lift / lbe_baseline_rev * 100) if lbe_baseline_rev != 0 else 0.0
    conf = confidence_label(ci_lo, ci_hi, rev)

    projected_rev = FORECAST_2027 + lift
    ci_lo_proj    = FORECAST_2027 + (ci_lo - lbe_baseline_rev)
    ci_hi_proj    = FORECAST_2027 + (ci_hi - lbe_baseline_rev)

    tbl_rows.append({
        "Scenario":              s["Scenario"],
        "Type":                  s["Type"],
        "2027 Base Rev ($M)":    round(proj_2027_total / 1e6, 2),
        "Scenario Lift ($M)":    round(lift / 1e6, 2),
        "Projected Rev ($M)":    round(projected_rev / 1e6, 2),
        "CI Low ($M)":           round(ci_lo_proj / 1e6, 2),
        "CI High ($M)":          round(ci_hi_proj / 1e6, 2),
        "Lift vs Base %":        f"{lift_pct:+.1f}%",
        "Confidence":            conf,
    })

scenario_df = pd.DataFrame(tbl_rows)
print(f"LBE 2025 Media Budget: ${(lbe_baseline_aw + lbe_baseline_co)/1e6:.2f}M  |  "
      f"Baseline media rev: ${lbe_baseline_rev/1e6:.2f}M (= decomp)  |  "
      f"2027 base: ${proj_2027_total/1e6:.1f}M (media ${proj_2027_media/1e6:.1f}M)  |  "
      f"recon: {recon_factor:.3f}×  |  CI: 90% posterior\n")
display(scenario_df)

# ══════════════════════════════════════════════════════════════════════════════
# PART 2 — TIER SPEND/REVENUE SHUFFLE PLOTS (one chart per scenario)
# ══════════════════════════════════════════════════════════════════════════════
lbe_funnel_contribution_ci = lbe_scenario_ci

FUNNELS_T      = ["Awareness", "Consideration"]
BRANDS_T       = ["TRR", "PDM"]
brand_tiers_t  = {"TRR": ["HPL", "LPL", "NL"], "PDM": ["OL", "NL"]}
FUNNEL_COLOR_T = {"Awareness": CPX_NAVY, "Consideration": CPX_RED}

tier_list = []
for funnel in FUNNELS_T:
    for brand in BRANDS_T:
        for tier in brand_tiers_t[brand]:
            tier_list.append((funnel, brand, tier))

n_tiers_total = len(tier_list)

actual_spend_tier = {}
for funnel, brand, tier in tier_list:
    actual_spend_tier[(funnel, brand, tier)] = eff_spend.get((brand, funnel, tier, 2025), 0.0)

actual_aw_total = sum(v for (f, b, t), v in actual_spend_tier.items() if f == "Awareness")
actual_co_total = sum(v for (f, b, t), v in actual_spend_tier.items() if f == "Consideration")

def _funnel_rev_split(aw_annual, co_annual):
    total_rev, _, _ = lbe_funnel_contribution_ci(aw_annual, co_annual)
    aw_only, _, _ = lbe_funnel_contribution_ci(aw_annual, 0.0)
    co_only, _, _ = lbe_funnel_contribution_ci(0.0, co_annual)
    denom = aw_only + co_only if (aw_only + co_only) > 0 else 1.0
    aw_share = aw_only / denom
    co_share = co_only / denom
    return total_rev * aw_share, total_rev * co_share

_aw_rev_2025, _co_rev_2025 = _funnel_rev_split(lbe_baseline_aw, lbe_baseline_co)
actual_rev_tier = {}
for funnel, brand, tier in tier_list:
    funnel_spend = actual_aw_total if funnel == "Awareness" else actual_co_total
    w = actual_spend_tier[(funnel, brand, tier)] / funnel_spend if funnel_spend > 0 else 0.0
    funnel_rev = _aw_rev_2025 if funnel == "Awareness" else _co_rev_2025
    actual_rev_tier[(funnel, brand, tier)] = funnel_rev * w

def scenario_tier_alloc(aw_annual, co_annual):
    aw_factor = aw_annual / actual_aw_total if actual_aw_total > 0 else 1.0
    co_factor = co_annual / actual_co_total if actual_co_total > 0 else 1.0

    scen_spend = {}
    for funnel, brand, tier in tier_list:
        factor = aw_factor if funnel == "Awareness" else co_factor
        scen_spend[(funnel, brand, tier)] = actual_spend_tier[(funnel, brand, tier)] * factor

    aw_rev_total, co_rev_total = _funnel_rev_split(aw_annual, co_annual)

    scen_rev = {}
    for funnel, brand, tier in tier_list:
        funnel_spend = sum(scen_spend[(f, b, t)] for f, b, t in tier_list if f == funnel)
        w = scen_spend[(funnel, brand, tier)] / funnel_spend if funnel_spend > 0 else 0.0
        funnel_rev = aw_rev_total if funnel == "Awareness" else co_rev_total
        scen_rev[(funnel, brand, tier)] = funnel_rev * w

    return scen_spend, scen_rev

fmt_K = mticker.FuncFormatter(lambda v, _: f"${v/1e3:.0f}K")

for s in scenarios:
    if s["Type"] == "Baseline":
        continue

    scen_spend, scen_rev = scenario_tier_alloc(s["aw"], s["co"])
    total_rev_scen, ci_lo, ci_hi = lbe_funnel_contribution_ci(s["aw"], s["co"])
    lift = total_rev_scen - lbe_baseline_rev
    lift_pct = lift / lbe_baseline_rev * 100

    ci_lo_pct = (ci_lo - lbe_baseline_rev) / lbe_baseline_rev * 100
    ci_hi_pct = (ci_hi - lbe_baseline_rev) / lbe_baseline_rev * 100

    fig, (ax_sp, ax_rv) = plt.subplots(1, 2, figsize=(18, 7))
    fig.patch.set_facecolor(WHITE)

    bar_w = 0.35
    x = np.arange(n_tiers_total)

    actual_vals_sp = np.array([actual_spend_tier[(f, b, t)] for f, b, t in tier_list])
    scen_vals_sp   = np.array([scen_spend[(f, b, t)] for f, b, t in tier_list])
    colors         = [FUNNEL_COLOR_T[f] for f, b, t in tier_list]

    for i in range(n_tiers_total):
        ax_sp.bar(x[i] - bar_w/2, actual_vals_sp[i], bar_w,
                  color=colors[i], alpha=0.9, edgecolor=WHITE, lw=0.4)
        ax_sp.bar(x[i] + bar_w/2, scen_vals_sp[i], bar_w,
                  color=colors[i], alpha=0.45, edgecolor=colors[i],
                  lw=0.6, hatch="///")

    for i in range(n_tiers_total):
        delta = scen_vals_sp[i] - actual_vals_sp[i]
        if abs(delta) > 0:
            sign = "+" if delta >= 0 else ""
            lbl = f"{sign}${delta/1e3:.0f}K" if abs(delta) < 1e6 else f"{sign}${delta/1e6:.1f}M"
            ax_sp.annotate(lbl, xy=(x[i] + bar_w/2, scen_vals_sp[i]),
                           xytext=(0, 4), textcoords="offset points",
                           ha="center", fontsize=7.5, fontweight="bold",
                           color=CPX_NAVY if delta >= 0 else CPX_RED)

    xlabels = [f"{b}\n{t}" for f, b, t in tier_list]
    ax_sp.set_xticks(x)
    ax_sp.set_xticklabels(xlabels, fontsize=10, fontweight="bold")
    ax_sp.set_ylabel("Spend ($K)", fontsize=12, fontweight="bold")
    ax_sp.yaxis.set_major_formatter(fmt_K)
    ax_sp.spines["top"].set_visible(False)
    ax_sp.spines["right"].set_visible(False)
    plt.setp(ax_sp.get_yticklabels(), fontsize=10, fontweight="bold")
    ax_sp.grid(False)
    n_aw = sum(1 for f, _, _ in tier_list if f == "Awareness")
    ax_sp.axvline(n_aw - 0.5, color="#CCCCCC", lw=1.2, linestyle="--")
    y_top_sp = max(actual_vals_sp.max(), scen_vals_sp.max()) * 1.25
    ax_sp.set_ylim(0, y_top_sp)
    ax_sp.text(n_aw / 2 - 0.5, y_top_sp * 0.96, "Awareness",
               ha="center", fontsize=10, fontweight="bold", color=CPX_NAVY)
    ax_sp.text(n_aw + (n_tiers_total - n_aw) / 2 - 0.5, y_top_sp * 0.96, "Consideration",
               ha="center", fontsize=10, fontweight="bold", color=CPX_RED)
    ax_sp.set_title("Spend by Tier", fontsize=13, fontweight="bold", color=CPX_NAVY, pad=8)

    lift_per_tier = np.array([
        scen_rev[(f, b, t)] - actual_rev_tier[(f, b, t)]
        for f, b, t in tier_list
    ])
    total_lift_check = lift_per_tier.sum()

    for i in range(n_tiers_total):
        bar_color = colors[i]
        ax_rv.bar(x[i], lift_per_tier[i], bar_w * 1.5,
                  color=bar_color,
                  alpha=0.75 if lift_per_tier[i] >= 0 else 0.5,
                  edgecolor=bar_color, lw=0.6)

    for i in range(n_tiers_total):
        val = lift_per_tier[i]
        sign = "+" if val >= 0 else ""
        lbl = f"{sign}${val/1e6:.2f}M"
        va = "bottom" if val >= 0 else "top"
        ax_rv.annotate(lbl, xy=(x[i], val),
                       xytext=(0, 4 if val >= 0 else -4),
                       textcoords="offset points",
                       ha="center", va=va, fontsize=7.5, fontweight="bold",
                       color=CPX_NAVY if val >= 0 else CPX_RED)

    ax_rv.set_xticks(x)
    ax_rv.set_xticklabels(xlabels, fontsize=10, fontweight="bold")
    ax_rv.set_ylabel("Revenue Lift ($M)", fontsize=12, fontweight="bold")
    ax_rv.yaxis.set_major_formatter(mticker.FuncFormatter(lambda v, _: f"${v/1e6:.2f}M"))
    ax_rv.spines["top"].set_visible(False)
    ax_rv.spines["right"].set_visible(False)
    plt.setp(ax_rv.get_yticklabels(), fontsize=10, fontweight="bold")
    ax_rv.grid(False)
    ax_rv.axhline(0, color="#666666", lw=0.8)
    ax_rv.axvline(n_aw - 0.5, color="#CCCCCC", lw=1.2, linestyle="--")
    y_max_rv = max(abs(lift_per_tier.max()), abs(lift_per_tier.min()), 0.01) * 1.4
    ax_rv.set_ylim(-y_max_rv if lift_per_tier.min() < 0 else 0, y_max_rv)
    ax_rv.text(n_aw / 2 - 0.5, y_max_rv * 0.92, "Awareness",
               ha="center", fontsize=10, fontweight="bold", color=CPX_NAVY)
    ax_rv.text(n_aw + (n_tiers_total - n_aw) / 2 - 0.5, y_max_rv * 0.92, "Consideration",
               ha="center", fontsize=10, fontweight="bold", color=CPX_RED)
    ax_rv.set_title(f"Total Revenue Lift by Tier (Total: +${total_lift_check/1e6:.2f}M)",
                    fontsize=13, fontweight="bold", color=CPX_NAVY, pad=8)

    legend_handles = [
        mpatches.Patch(facecolor=CPX_NAVY, alpha=0.9, label="Awareness"),
        mpatches.Patch(facecolor=CPX_RED, alpha=0.9, label="Consideration"),
        mpatches.Patch(facecolor="#666666", alpha=0.9, label="2025 Actual (solid)"),
        mpatches.Patch(facecolor="#666666", alpha=0.45, hatch="///", label="Scenario (hatched)"),
    ]
    fig.legend(handles=legend_handles, fontsize=11, loc="upper right",
              bbox_to_anchor=(0.98, 0.98), frameon=False)

    proj_total_scen = proj_2027_total + lift
    fig.suptitle(
        f"LBE MMM 2027 Projection \u2014 {s['Scenario']}\n"
        f"2027 base ${proj_2027_total/1e6:.1f}M  |  Scenario media lift: +${lift/1e6:.2f}M ({lift_pct:+.1f}%)  "
        f"|  Projected 2027: ${proj_total_scen/1e6:.1f}M  |  90% CI: [{ci_lo_pct:+.1f}%, {ci_hi_pct:+.1f}%]",
        fontsize=13.5, fontweight="bold", color=CPX_NAVY, y=1.03
    )
    fig.text(0.5, -0.02,
        "Tier revenue = 2027 forecast split to media (2025 ratio) \u2192 funnel (model) \u2192 tier (2025 spend share). "
        "Tier detail is proportional, not modeled at tier level.",
        ha="center", fontsize=8, color=CPX_MGREY, style="italic")

    plt.tight_layout()
    save_fig(fig, f"lbe_2027_scenario_{s['Scenario'].lower().replace(' ', '_').replace('%','pct')}.png")
    display(fig)
    plt.close(fig)

    tbl = []
    for f, b, t in tier_list:
        act_sp = actual_spend_tier[(f, b, t)]
        scn_sp = scen_spend[(f, b, t)]
        act_rv = actual_rev_tier[(f, b, t)]
        scn_rv_val = scen_rev[(f, b, t)]
        tbl.append({
            "Funnel": f, "Brand": b, "Tier": t,
            "2025 Spend ($K)": round(act_sp / 1e3, 1),
            "Scenario Spend ($K)": round(scn_sp / 1e3, 1),
            "Spend Δ ($K)": round((scn_sp - act_sp) / 1e3, 1),
            "2025 Media Rev ($M)": round(act_rv / 1e6, 3),
            "Scenario Media Rev ($M)": round(scn_rv_val / 1e6, 3),
            "Rev Lift ($M)": round((scn_rv_val - act_rv) / 1e6, 3),
        })
    tbl.append({
        "Funnel": "", "Brand": "", "Tier": "TOTAL",
        "2025 Spend ($K)": round(sum(actual_spend_tier.values()) / 1e3, 1),
        "Scenario Spend ($K)": round(sum(scen_spend.values()) / 1e3, 1),
        "Spend Δ ($K)": round((sum(scen_spend.values()) - sum(actual_spend_tier.values())) / 1e3, 1),
        "2025 Media Rev ($M)": round(sum(actual_rev_tier.values()) / 1e6, 2),
        "Scenario Media Rev ($M)": round(sum(scen_rev.values()) / 1e6, 2),
        "Rev Lift ($M)": round(total_lift_check / 1e6, 2),
    })
    print(f"\n{s['Scenario']} — Tier Breakdown")
    display(pd.DataFrame(tbl).astype(str))

# ══════════════════════════════════════════════════════════════════════════════
# PART 3 — 2027 SCENARIO UPSIDE LADDER
# ══════════════════════════════════════════════════════════════════════════════
CPX_GREEN = "#2E7D32"

_ld = scenario_df[scenario_df["Type"] != "Baseline"].copy()
_ld["gain"]   = _ld["Scenario Lift ($M)"].astype(float)
_ld["ci_lo"]  = _ld["CI Low ($M)"].astype(float)  - _ld["Projected Rev ($M)"].astype(float) + _ld["gain"]
_ld["ci_hi"]  = _ld["CI High ($M)"].astype(float) - _ld["Projected Rev ($M)"].astype(float) + _ld["gain"]

df_plot = _ld.sort_values("gain", ascending=True).reset_index(drop=True)

labels  = [re.sub(r'\d+%\s*Shift', 'Reshuffle', l) for l in df_plot["Scenario"].tolist()]
labels  = [re.sub(r'[:\s]*Co\s*[->→]+\s*Aw', '', l).strip() for l in labels]
labels  = ["Reshuffle Only" if l == "Reshuffle" else l for l in labels]
gains   = df_plot["gain"].to_numpy()
lows    = df_plot["ci_lo"].to_numpy()
highs   = df_plot["ci_hi"].to_numpy()
n       = len(df_plot)
ceiling = gains.max()

FORECAST_2027_M = FORECAST_2027 / 1e6

def bar_color(row):
    t = str(row["Type"]).lower()
    if t == "budget":       return CPX_NAVY
    if t == "combined":     return CPX_GREEN
    if t == "reshuffle":    return CPX_RED
    return CPX_MGREY

colors = [bar_color(df_plot.iloc[i]) for i in range(n)]
RECOMMENDED = labels[int(np.argmax(gains))]
for i in range(n):
    if labels[i] == RECOMMENDED:
        colors[i] = CPX_GOLD
    elif str(df_plot.iloc[i]["Type"]).lower() == "reshuffle":
        colors[i] = CPX_RED

fig, ax = plt.subplots(figsize=(13, 4.6))
y      = np.arange(n)
floor  = min(gains.min(), 0.0)
span   = ceiling - floor
lab_pad = span * 0.012

pad     = ceiling * 0.06
x_lo    = floor - pad
x_hi    = x_lo + (ceiling - x_lo) / 0.65
band    = x_hi - ceiling
x_ci    = ceiling * 1.02
x_2026  = x_ci + band * 0.42
x_max   = x_hi

for i in range(n):
    is_rec = labels[i] == RECOMMENDED
    c = colors[i]
    g = gains[i]
    ax.barh(y[i], span, left=floor, height=0.72, color=CPX_LGREY, alpha=0.28,
            edgecolor="none", zorder=1)
    ax.barh(y[i], g, height=0.72, color=c,
            alpha=0.92 if is_rec else 0.8, edgecolor="none", zorder=2)
    if g < 0:
        ax.text(lab_pad, y[i], f"{g:+.2f}M",
                ha="left", va="center", fontsize=9, fontweight="bold", color=c, zorder=4)
    elif g >= span * 0.10:
        ax.text(g - lab_pad, y[i], f"{g:+.2f}M",
                ha="right", va="center",
                fontsize=10 if is_rec else 9, fontweight="bold", color=WHITE, zorder=4)
    else:
        ax.text(g + lab_pad, y[i], f"{g:+.2f}M",
                ha="left", va="center",
                fontsize=9, fontweight="bold", color=c, zorder=4)
    rev_27 = FORECAST_2027_M + g
    ax.text(x_ci, y[i], f"[{lows[i]:.2f}–{highs[i]:.2f}]",
            ha="left", va="center", fontsize=9, fontweight="bold", color=c, zorder=4, clip_on=False)
    ax.text(x_2026, y[i], f"2027: ${rev_27:.1f}M",
            ha="left", va="center", fontsize=8, fontweight="bold", color=CPX_GREEN, zorder=4, clip_on=False)

if floor < 0:
    ax.axvline(0, color=CPX_MGREY, lw=1.0, ls="-", alpha=0.6, zorder=3)

ax.axvline(ceiling, color=CPX_RED, lw=1.4, ls="--", alpha=0.8, zorder=5)
ax.text(ceiling, n - 0.42, f"Best case  +${ceiling:.2f}M",
        ha="center", va="bottom", fontsize=8.5, fontweight="bold",
        color=CPX_RED, zorder=6)

ax.set_yticks(y)
ytick_labels = [f"★ {l}" if l == RECOMMENDED else l for l in labels]
ax.set_yticklabels(ytick_labels, fontsize=9.5, fontweight="bold", color=CPX_DGREY)
ax.tick_params(axis="y", length=0, pad=8)
ax.set_xlim(x_lo, x_max)
ax.set_ylim(-0.7, n + 0.7)
ax.xaxis.set_major_formatter(mticker.FuncFormatter(lambda v, _: f"{v:+.1f}M"))
plt.setp(ax.get_xticklabels(), fontsize=8.5, fontweight="bold", color=CPX_DGREY)
for sp in ("top", "right", "left", "bottom"):
    ax.spines[sp].set_visible(False)
ax.grid(axis="x", alpha=0.14, color=CPX_LGREY)
ax.set_axisbelow(True)
ax.tick_params(axis="x", labelbottom=False, length=0)

ax.text(x_lo, n + 0.35, f"2027 forecast base: ${FORECAST_2027_M:.1f}M   |   bars = scenario media lift on top of base",
        ha="left", va="bottom", fontsize=8, fontweight="bold", color=CPX_MGREY, zorder=4)

leg = [mpatches.Patch(facecolor=CPX_NAVY,  label="Budget uplift"),
       mpatches.Patch(facecolor=CPX_GREEN, label="Combined"),
       mpatches.Patch(facecolor=CPX_GOLD,  label="Best case scenario"),
       mpatches.Patch(facecolor=CPX_RED,   label="Reshuffle"),
       mpatches.Patch(facecolor=CPX_LGREY, alpha=0.28, label="Gap to best case"),
       Line2D([0], [0], color=CPX_RED,   lw=1.4, ls="--", alpha=0.8, label="Best case line"),
       Line2D([0], [0], color=CPX_DGREY, lw=1.5, ls="-",  label="[low–high] = 90% CI (lift)")]
fig.legend(handles=leg, fontsize=7.5, loc="upper center", bbox_to_anchor=(0.5, 0.90),
           ncol=4, frameon=True, framealpha=0.9, edgecolor=CPX_LGREY,
           borderpad=0.5, labelspacing=0.5, handlelength=1.5)

fig.suptitle("LBE 2027 Scenario Upside — Climb to Best Case (Revenue)",
             fontsize=13, fontweight="bold", color=CPX_NAVY, y=0.995)
fig.text(0.99, 0.005, "Cineplex Marketing Analytics  |  LBE MMM V5e",
         ha="right", va="bottom", fontsize=6, color=CPX_LGREY)

plt.tight_layout(rect=[0, 0, 1, 0.82])
save_fig(fig, "FINAL_2027_lbe_scenario_ladder_compact.png")
display(fig)
plt.close(fig)

In [0]:
# ━━ LBE BUDGET SCENARIO ANALYSIS (POSTERIOR-BASED, 2027 PROJECTION) ━━━━━━━━━━━
# Uses lbe_media_rev_draws() (cell 32) — real V5e posterior draws, dual-target,
# per-brand. CIs are posterior credible intervals, NOT a MAPE band.
# Baseline media point reconciled to 2025 decomp; scenario media LIFT projected
# onto the 2027 forecast base (FORECAST_2027) to match cells 16/35/36.

# 2025 baseline spend (LBE annual) — the spend base the scenarios scale from
lbe_baseline_aw = lbe_aw_2025
lbe_baseline_co = lbe_co_2025

# ── Reconciliation factor: anchor posterior median to decomp point estimate ──
_base_draws_33 = lbe_media_rev_draws(lbe_baseline_aw, lbe_baseline_co)
_posterior_base_med = float(np.median(_base_draws_33))
recon_factor = _decomp_target / _posterior_base_med if _posterior_base_med > 0 else 1.0
lbe_baseline_rev = _decomp_target   # reported baseline media = decomp target exactly

def lbe_scenario_ci(aw_annual, co_annual):
    """Posterior point + 90% CI for a scenario budget, reconciled to decomp.
    point = median * recon_factor ; CI = posterior percentiles * recon_factor."""
    draws = lbe_media_rev_draws(aw_annual, co_annual) * recon_factor
    point   = float(np.median(draws))
    ci_low  = float(np.percentile(draws, 5))
    ci_high = float(np.percentile(draws, 95))
    return point, ci_low, ci_high

def confidence_label(ci_low, ci_high, point):
    """High / Medium / Low from posterior CI width relative to point."""
    if point == 0:
        return "Low"
    ci_width_pct = (ci_high - ci_low) / abs(point) * 100
    if ci_width_pct < 40:
        return "High"
    elif ci_width_pct < 80:
        return "Medium"
    else:
        return "Low"

# ── 2025 Actual Total Revenue (TRR + PDM) — reference only ──────────────────
_lbe_rev_2025 = float((d.loc[d["week_date"].dt.year == 2025, "TRR_Rev"].sum()
                       + d.loc[d["week_date"].dt.year == 2025, "PDM_Rev"].sum()))

# ── 2027 projection base (matches cell 16/35/36) ────────────────────────────
# 2027 media = FORECAST_2027 × (2025 media ratio); non-media = forecast − media.
# Scenario media (from the model) replaces the base media on top of this base.
_media_ratio_27   = lbe_baseline_rev / _lbe_rev_2025           # 2025 media share
proj_2027_total   = FORECAST_2027
proj_2027_media   = FORECAST_2027 * _media_ratio_27
proj_2027_nonmedia = proj_2027_total - proj_2027_media

# ── Baseline media contribution (point = decomp; CI = posterior) ─────────────
_, lbe_base_ci_lo, lbe_base_ci_hi = lbe_scenario_ci(lbe_baseline_aw, lbe_baseline_co)

# ── Define all scenarios ──────────────────────────────────────────────────────
scenarios = []
scenarios.append({"Scenario": "2025 Baseline", "Type": "Baseline",
                  "aw": lbe_baseline_aw, "co": lbe_baseline_co})

for pct in [3, 5, 10, 20, 30]:
    factor = 1 + pct / 100
    scenarios.append({"Scenario": f"+{pct}% Budget Uplift", "Type": "Budget",
                      "aw": lbe_baseline_aw * factor, "co": lbe_baseline_co * factor})

shift = lbe_baseline_co * 0.20
scenarios.append({"Scenario": "20% Shift: Co -> Aw", "Type": "Reshuffle",
                  "aw": lbe_baseline_aw + shift, "co": lbe_baseline_co - shift})

for pct in [10, 20, 30]:
    factor = 1 + pct / 100
    aw_up = lbe_baseline_aw * factor
    co_up = lbe_baseline_co * factor
    shift_c = co_up * 0.20
    scenarios.append({"Scenario": f"+{pct}% + 20% Shift", "Type": "Combined",
                      "aw": aw_up + shift_c, "co": co_up - shift_c})

# ── Compute results for each scenario — projected onto 2027 base ──────────────
tbl_rows = []
for s in scenarios:
    rev, ci_lo, ci_hi = lbe_scenario_ci(s["aw"], s["co"])   # scenario media + CI
    lift = rev - lbe_baseline_rev                            # media lift vs baseline (base-agnostic)
    lift_pct = (lift / lbe_baseline_rev * 100) if lbe_baseline_rev != 0 else 0.0
    conf = confidence_label(ci_lo, ci_hi, rev)

    # 2027 projected total = 2027 non-media base + scenario media contribution
    projected_rev = FORECAST_2027 + lift
    ci_lo_proj    = FORECAST_2027 + (ci_lo - lbe_baseline_rev)
    ci_hi_proj    = FORECAST_2027 + (ci_hi - lbe_baseline_rev)
    
    tbl_rows.append({
        "Scenario":              s["Scenario"],
        "Type":                  s["Type"],
        "2027 Base Rev ($M)":    round(proj_2027_total / 1e6, 2),
        "Scenario Lift ($M)":    round(lift / 1e6, 2),
        "Projected Rev ($M)":    round(projected_rev / 1e6, 2),
        "CI Low ($M)":           round(ci_lo_proj / 1e6, 2),
        "CI High ($M)":          round(ci_hi_proj / 1e6, 2),
        "Lift vs Base %":        f"{lift_pct:+.1f}%",
        "Confidence":            conf,
    })

scenario_df = pd.DataFrame(tbl_rows)
print(f"LBE 2025 Media Budget: ${(lbe_baseline_aw + lbe_baseline_co)/1e6:.2f}M  |  "
      f"Baseline media rev: ${lbe_baseline_rev/1e6:.2f}M (= decomp)  |  "
      f"2027 base: ${proj_2027_total/1e6:.1f}M (media ${proj_2027_media/1e6:.1f}M)  |  "
      f"recon: {recon_factor:.3f}×  |  CI: 90% posterior\n")
display(scenario_df)

In [0]:
#━━ LBE BUDGET SCENARIO ANALYSIS ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

# 2025 baseline spend
df_lbe_2025 = df_lbe_all[df_lbe_all["week_date"].dt.year == 2025]
lbe_baseline_aw = float(df_lbe_2025["Awareness"].sum())
lbe_baseline_co = float(df_lbe_2025["Consideration"].sum())

# ── Model uncertainty from V5e MAPE (used for CI estimation) ──────────────────
_mape_rpt_trr = results["TRR"].get("mape_rpt", 7.0)
_mape_txn_trr = results["TRR"].get("mape_txn", 7.0)
_mape_rpt_pdm = results["PDM"].get("mape_rpt", 7.0)
_mape_txn_pdm = results["PDM"].get("mape_txn", 7.0)
_model_cv = np.sqrt((_mape_rpt_trr**2 + _mape_txn_trr**2
                     + _mape_rpt_pdm**2 + _mape_txn_pdm**2) / 4) / 100

def lbe_funnel_contribution_ci(aw_annual, co_annual, n_sim=2000):
    """Point estimate + 90% CI via model uncertainty propagation.
    Uses lbe_media_rev() helper — Hill + finite-L adstock (matches V5e)."""
    contribution_point = lbe_media_rev(aw_annual, co_annual)

    # Monte Carlo CI using model uncertainty
    np.random.seed(42)
    noise = np.random.normal(1.0, _model_cv, n_sim)
    contributions_sim = contribution_point * noise
    ci_low  = float(np.percentile(contributions_sim, 5))
    ci_high = float(np.percentile(contributions_sim, 95))

    return contribution_point, ci_low, ci_high

# ── Confidence classification ─────────────────────────────────────────────────
def confidence_label(ci_low, ci_high, point):
    """High / Medium / Low based on CI width relative to point estimate."""
    if point == 0:
        return "Low"
    ci_width_pct = (ci_high - ci_low) / abs(point) * 100
    if ci_width_pct < 20:
        return "High"
    elif ci_width_pct < 35:
        return "Medium"
    else:
        return "Low"

# ── 2025 Actual Total Revenue (TRR + PDM) ───────────────────────────────────
_lbe_rev_2025 = float((d.loc[d["week_date"].dt.year == 2025, "TRR_Rev"].sum()
                       + d.loc[d["week_date"].dt.year == 2025, "PDM_Rev"].sum()))

# ── Baseline media contribution ─────────────────────────────────────────────
lbe_baseline_rev, lbe_base_ci_lo, lbe_base_ci_hi = lbe_funnel_contribution_ci(
    lbe_baseline_aw, lbe_baseline_co
)

# ── Define all scenarios ──────────────────────────────────────────────────────
scenarios = []

scenarios.append({
    "Scenario": "2025 Baseline",
    "Type": "Baseline",
    "aw": lbe_baseline_aw,
    "co": lbe_baseline_co,
})

for pct in [3, 5, 10, 20, 30]:
    factor = 1 + pct / 100
    scenarios.append({
        "Scenario": f"+{pct}% Budget Uplift",
        "Type": "Budget",
        "aw": lbe_baseline_aw * factor,
        "co": lbe_baseline_co * factor,
    })

shift = lbe_baseline_co * 0.20
scenarios.append({
    "Scenario": "20% Shift: Co -> Aw",
    "Type": "Reshuffle",
    "aw": lbe_baseline_aw + shift,
    "co": lbe_baseline_co - shift,
})

for pct in [10, 20, 30]:
    factor = 1 + pct / 100
    aw_up = lbe_baseline_aw * factor
    co_up = lbe_baseline_co * factor
    shift_c = co_up * 0.20
    scenarios.append({
        "Scenario": f"+{pct}% + 20% Shift",
        "Type": "Combined",
        "aw": aw_up + shift_c,
        "co": co_up - shift_c,
    })

# ── Compute results for each scenario ─────────────────────────────────────────
tbl_rows = []
for s in scenarios:
    rev, ci_lo, ci_hi = lbe_funnel_contribution_ci(s["aw"], s["co"])
    lift = rev - lbe_baseline_rev
    lift_pct = (lift / lbe_baseline_rev * 100) if lbe_baseline_rev != 0 else 0.0
    conf = confidence_label(ci_lo, ci_hi, rev)

    _baseline_non_media = _lbe_rev_2025 - lbe_baseline_rev
    projected_rev = _baseline_non_media + rev
    ci_lo_proj = _baseline_non_media + ci_lo
    ci_hi_proj = _baseline_non_media + ci_hi

    tbl_rows.append({
        "Scenario":              s["Scenario"],
        "Type":                  s["Type"],
        "2025 Actual Rev ($M)":  round(_lbe_rev_2025 / 1e6, 2),
        "Scenario Lift ($M)":    round(lift / 1e6, 2),
        "Projected Rev ($M)":    round(projected_rev / 1e6, 2),
        "CI Low ($M)":           round(ci_lo_proj / 1e6, 2),
        "CI High ($M)":          round(ci_hi_proj / 1e6, 2),
        "Lift vs Actual %":      f"{lift_pct:+.1f}%",
        "Confidence":            conf,
    })

scenario_df = pd.DataFrame(tbl_rows)
print(f"LBE 2025 Media Budget: ${(lbe_baseline_aw + lbe_baseline_co)/1e6:.2f}M  |  "
      f"Model CV: {_model_cv*100:.1f}%  |  CI: 90% (5th-95th percentile)\n")
display(scenario_df)

In [0]:
# ━━ LBE SCENARIO: SPEND & REVENUE BY TIER ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
# One chart per scenario (from cell 29). Each chart shows:
#   • Spend by tier: 2025 actual (solid) vs scenario (hatched)
#   • Revenue by tier: 2025 actual (solid) vs scenario (hatched)
#   • Delta labels on scenario bars
#   • Title includes total revenue lift & CI
import matplotlib.patches as mpatches
import matplotlib.ticker as mticker

# ── Compat shim: old name → new posterior scenario CI (cell 33) ──────────────
lbe_funnel_contribution_ci = lbe_scenario_ci

# ── 2027 projection base: forecast media contribution (matches cell 16/36) ──
# 2027 media revenue = FORECAST_2027 × 2025 media_ratio (same as decomp/ladder).
# Scenario media LIFT (from the model) is added on top of this forecast base.
_media_ratio_27 = (_media_2025 / _actual_2025) if ('_media_2025' in dir() and _actual_2025) \
                  else (lbe_baseline_rev / _lbe_rev_2025)
proj_2027_total   = FORECAST_2027
proj_2027_media   = FORECAST_2027 * _media_ratio_27
proj_2027_nonmedia = proj_2027_total - proj_2027_media

# ── Tier structure ───────────────────────────────────────────────────────────
FUNNELS_T      = ["Awareness", "Consideration"]
BRANDS_T       = ["TRR", "PDM"]
brand_tiers_t  = {"TRR": ["HPL", "LPL", "NL"], "PDM": ["OL", "NL"]}
FUNNEL_COLOR_T = {"Awareness": CPX_NAVY, "Consideration": CPX_RED}

# Build tier list: (funnel, brand, tier)
tier_list = []
for funnel in FUNNELS_T:
    for brand in BRANDS_T:
        for tier in brand_tiers_t[brand]:
            tier_list.append((funnel, brand, tier))

n_tiers_total = len(tier_list)

# ── 2025 actual spend & revenue per tier ────────────────────────────────────
actual_spend_tier = {}
for funnel, brand, tier in tier_list:
    actual_spend_tier[(funnel, brand, tier)] = eff_spend.get((brand, funnel, tier, 2025), 0.0)

# Funnel totals for proportional allocation
actual_aw_total = sum(v for (f, b, t), v in actual_spend_tier.items() if f == "Awareness")
actual_co_total = sum(v for (f, b, t), v in actual_spend_tier.items() if f == "Consideration")

# ── Funnel-level revenue function (for splitting total rev to Aw/Co) ────────
def _funnel_rev_split(aw_annual, co_annual):
    """Returns (aw_contribution, co_contribution) using the model."""
    # Total contribution
    total_rev, _, _ = lbe_funnel_contribution_ci(aw_annual, co_annual)
    # Awareness-only contribution
    aw_only, _, _ = lbe_funnel_contribution_ci(aw_annual, 0.0)
    # Consideration-only contribution
    co_only, _, _ = lbe_funnel_contribution_ci(0.0, co_annual)
    # Proportional split (handles interaction)
    denom = aw_only + co_only if (aw_only + co_only) > 0 else 1.0
    aw_share = aw_only / denom
    co_share = co_only / denom
    return total_rev * aw_share, total_rev * co_share

# ── 2025 actual media-attributed revenue per tier (model-consistent) ───────
# Uses the same funnel-split + spend-weight methodology as scenario allocation
# so that baseline and scenario are on the same basis and lifts sum correctly.
_aw_rev_2025, _co_rev_2025 = _funnel_rev_split(lbe_baseline_aw, lbe_baseline_co)
actual_rev_tier = {}
for funnel, brand, tier in tier_list:
    funnel_spend = actual_aw_total if funnel == "Awareness" else actual_co_total
    if funnel_spend > 0:
        w = actual_spend_tier[(funnel, brand, tier)] / funnel_spend
    else:
        w = 0.0
    funnel_rev = _aw_rev_2025 if funnel == "Awareness" else _co_rev_2025
    actual_rev_tier[(funnel, brand, tier)] = funnel_rev * w

# ── Compute scenario tier allocations ────────────────────────────────────────
def scenario_tier_alloc(aw_annual, co_annual):
    """Allocate scenario spend & revenue to tiers proportionally."""
    # Spend allocation: scale each tier by funnel ratio
    aw_factor = aw_annual / actual_aw_total if actual_aw_total > 0 else 1.0
    co_factor = co_annual / actual_co_total if actual_co_total > 0 else 1.0

    scen_spend = {}
    for funnel, brand, tier in tier_list:
        factor = aw_factor if funnel == "Awareness" else co_factor
        scen_spend[(funnel, brand, tier)] = actual_spend_tier[(funnel, brand, tier)] * factor

    # Revenue allocation: total scenario rev split by funnel, then by tier spend weight
    aw_rev_total, co_rev_total = _funnel_rev_split(aw_annual, co_annual)

    scen_rev = {}
    for funnel, brand, tier in tier_list:
        funnel_spend = sum(scen_spend[(f, b, t)] for f, b, t in tier_list if f == funnel)
        if funnel_spend > 0:
            w = scen_spend[(funnel, brand, tier)] / funnel_spend
        else:
            w = 0.0
        funnel_rev = aw_rev_total if funnel == "Awareness" else co_rev_total
        scen_rev[(funnel, brand, tier)] = funnel_rev * w

    return scen_spend, scen_rev

# ── Plotting ─────────────────────────────────────────────────────────────────
fmt_K = mticker.FuncFormatter(lambda v, _: f"${v/1e3:.0f}K")

for s in scenarios:
    if s["Type"] == "Baseline":
        continue  # skip baseline (no change to show)

    scen_spend, scen_rev = scenario_tier_alloc(s["aw"], s["co"])
    total_rev_scen, ci_lo, ci_hi = lbe_funnel_contribution_ci(s["aw"], s["co"])
    lift = total_rev_scen - lbe_baseline_rev
    lift_pct = lift / lbe_baseline_rev * 100

    # CI as % of baseline
    ci_lo_pct = (ci_lo - lbe_baseline_rev) / lbe_baseline_rev * 100
    ci_hi_pct = (ci_hi - lbe_baseline_rev) / lbe_baseline_rev * 100

    fig, (ax_sp, ax_rv) = plt.subplots(1, 2, figsize=(18, 7))
    fig.patch.set_facecolor(WHITE)

    bar_w = 0.35
    x = np.arange(n_tiers_total)

    # ── LEFT PANEL: Spend by Tier (actual vs scenario) ─────────────────────
    actual_vals_sp = np.array([actual_spend_tier[(f, b, t)] for f, b, t in tier_list])
    scen_vals_sp   = np.array([scen_spend[(f, b, t)] for f, b, t in tier_list])
    colors         = [FUNNEL_COLOR_T[f] for f, b, t in tier_list]

    for i in range(n_tiers_total):
        ax_sp.bar(x[i] - bar_w/2, actual_vals_sp[i], bar_w,
                  color=colors[i], alpha=0.9, edgecolor=WHITE, lw=0.4)
        ax_sp.bar(x[i] + bar_w/2, scen_vals_sp[i], bar_w,
                  color=colors[i], alpha=0.45, edgecolor=colors[i],
                  lw=0.6, hatch="///")

    # Spend delta labels
    for i in range(n_tiers_total):
        delta = scen_vals_sp[i] - actual_vals_sp[i]
        if abs(delta) > 0:
            sign = "+" if delta >= 0 else ""
            lbl = f"{sign}${delta/1e3:.0f}K" if abs(delta) < 1e6 else f"{sign}${delta/1e6:.1f}M"
            ax_sp.annotate(lbl, xy=(x[i] + bar_w/2, scen_vals_sp[i]),
                           xytext=(0, 4), textcoords="offset points",
                           ha="center", fontsize=7.5, fontweight="bold",
                           color=CPX_NAVY if delta >= 0 else CPX_RED)

    xlabels = [f"{b}\n{t}" for f, b, t in tier_list]
    ax_sp.set_xticks(x)
    ax_sp.set_xticklabels(xlabels, fontsize=10, fontweight="bold")
    ax_sp.set_ylabel("Spend ($K)", fontsize=12, fontweight="bold")
    ax_sp.yaxis.set_major_formatter(fmt_K)
    ax_sp.spines["top"].set_visible(False)
    ax_sp.spines["right"].set_visible(False)
    plt.setp(ax_sp.get_yticklabels(), fontsize=10, fontweight="bold")
    ax_sp.grid(False)
    n_aw = sum(1 for f, _, _ in tier_list if f == "Awareness")
    ax_sp.axvline(n_aw - 0.5, color="#CCCCCC", lw=1.2, linestyle="--")
    y_top_sp = max(actual_vals_sp.max(), scen_vals_sp.max()) * 1.25
    ax_sp.set_ylim(0, y_top_sp)
    ax_sp.text(n_aw / 2 - 0.5, y_top_sp * 0.96, "Awareness",
               ha="center", fontsize=10, fontweight="bold", color=CPX_NAVY)
    ax_sp.text(n_aw + (n_tiers_total - n_aw) / 2 - 0.5, y_top_sp * 0.96, "Consideration",
               ha="center", fontsize=10, fontweight="bold", color=CPX_RED)
    ax_sp.set_title("Spend by Tier", fontsize=13, fontweight="bold", color=CPX_NAVY, pad=8)

    # ── RIGHT PANEL: Total Revenue Lift by Tier ──────────────────────────────
    # Shows how the total revenue lift distributes across tiers
    lift_per_tier = np.array([
        scen_rev[(f, b, t)] - actual_rev_tier[(f, b, t)]
        for f, b, t in tier_list
    ])
    total_lift_check = lift_per_tier.sum()

    for i in range(n_tiers_total):
        bar_color = colors[i]
        ax_rv.bar(x[i], lift_per_tier[i], bar_w * 1.5,
                  color=bar_color,
                  alpha=0.75 if lift_per_tier[i] >= 0 else 0.5,
                  edgecolor=bar_color, lw=0.6)

    # Value labels on each bar
    for i in range(n_tiers_total):
        val = lift_per_tier[i]
        sign = "+" if val >= 0 else ""
        lbl = f"{sign}${val/1e6:.2f}M"
        y_pos = val if val >= 0 else val
        va = "bottom" if val >= 0 else "top"
        ax_rv.annotate(lbl, xy=(x[i], y_pos),
                       xytext=(0, 4 if val >= 0 else -4),
                       textcoords="offset points",
                       ha="center", va=va, fontsize=7.5, fontweight="bold",
                       color=CPX_NAVY if val >= 0 else CPX_RED)

    ax_rv.set_xticks(x)
    ax_rv.set_xticklabels(xlabels, fontsize=10, fontweight="bold")
    ax_rv.set_ylabel("Revenue Lift ($M)", fontsize=12, fontweight="bold")
    ax_rv.yaxis.set_major_formatter(mticker.FuncFormatter(lambda v, _: f"${v/1e6:.2f}M"))
    ax_rv.spines["top"].set_visible(False)
    ax_rv.spines["right"].set_visible(False)
    plt.setp(ax_rv.get_yticklabels(), fontsize=10, fontweight="bold")
    ax_rv.grid(False)
    ax_rv.axhline(0, color="#666666", lw=0.8)
    ax_rv.axvline(n_aw - 0.5, color="#CCCCCC", lw=1.2, linestyle="--")
    y_max_rv = max(abs(lift_per_tier.max()), abs(lift_per_tier.min()), 0.01) * 1.4
    ax_rv.set_ylim(-y_max_rv if lift_per_tier.min() < 0 else 0, y_max_rv)
    ax_rv.text(n_aw / 2 - 0.5, y_max_rv * 0.92, "Awareness",
               ha="center", fontsize=10, fontweight="bold", color=CPX_NAVY)
    ax_rv.text(n_aw + (n_tiers_total - n_aw) / 2 - 0.5, y_max_rv * 0.92, "Consideration",
               ha="center", fontsize=10, fontweight="bold", color=CPX_RED)
    ax_rv.set_title(f"Total Revenue Lift by Tier (Total: +${total_lift_check/1e6:.2f}M)",
                    fontsize=13, fontweight="bold", color=CPX_NAVY, pad=8)

    # Legend
    legend_handles = [
        mpatches.Patch(facecolor=CPX_NAVY, alpha=0.9, label="Awareness"),
        mpatches.Patch(facecolor=CPX_RED, alpha=0.9, label="Consideration"),
        mpatches.Patch(facecolor="#666666", alpha=0.9, label="2025 Actual (solid)"),
        mpatches.Patch(facecolor="#666666", alpha=0.45, hatch="///", label="Scenario (hatched)"),
    ]
    fig.legend(handles=legend_handles, fontsize=11, loc="upper right",
              bbox_to_anchor=(0.98, 0.98), frameon=False)

    proj_total_scen = proj_2027_total + lift          # 2027 forecast + scenario media lift
    fig.suptitle(
        f"LBE MMM 2027 Projection \u2014 {s['Scenario']}\n"
        f"2027 base ${proj_2027_total/1e6:.1f}M  |  Scenario media lift: +${lift/1e6:.2f}M ({lift_pct:+.1f}%)  "
        f"|  Projected 2027: ${proj_total_scen/1e6:.1f}M  |  90% CI: [{ci_lo_pct:+.1f}%, {ci_hi_pct:+.1f}%]",
        fontsize=13.5, fontweight="bold", color=CPX_NAVY, y=1.03
    )
    fig.text(0.5, -0.02,
        "Tier revenue = 2027 forecast split to media (2025 ratio) \u2192 funnel (model) \u2192 tier (2025 spend share). "
        "Tier detail is proportional, not modeled at tier level.",
        ha="center", fontsize=8, color=CPX_MGREY, style="italic")

    plt.tight_layout()
    save_fig(fig, f"lbe_2027_scenario_{s['Scenario'].lower().replace(' ', '_').replace('%','pct')}.png")
    display(fig)
    plt.close(fig)

    # ── Table: Spend & Revenue Lift by Tier ───────────────────────────────────
    tbl = []
    for f, b, t in tier_list:
        act_sp = actual_spend_tier[(f, b, t)]
        scn_sp = scen_spend[(f, b, t)]
        act_rv = actual_rev_tier[(f, b, t)]
        scn_rv_val = scen_rev[(f, b, t)]
        tbl.append({
            "Funnel": f,
            "Brand": b,
            "Tier": t,
            "2025 Spend ($K)": round(act_sp / 1e3, 1),
            "Scenario Spend ($K)": round(scn_sp / 1e3, 1),
            "Spend Δ ($K)": round((scn_sp - act_sp) / 1e3, 1),
            "2025 Media Rev ($M)": round(act_rv / 1e6, 3),
            "Scenario Media Rev ($M)": round(scn_rv_val / 1e6, 3),
            "Rev Lift ($M)": round((scn_rv_val - act_rv) / 1e6, 3),
        })
    # Totals row
    tbl.append({
        "Funnel": "", "Brand": "", "Tier": "TOTAL",
        "2025 Spend ($K)": round(sum(actual_spend_tier.values()) / 1e3, 1),
        "Scenario Spend ($K)": round(sum(scen_spend.values()) / 1e3, 1),
        "Spend Δ ($K)": round((sum(scen_spend.values()) - sum(actual_spend_tier.values())) / 1e3, 1),
        "2025 Media Rev ($M)": round(sum(actual_rev_tier.values()) / 1e6, 2),
        "Scenario Media Rev ($M)": round(sum(scen_rev.values()) / 1e6, 2),
        "Rev Lift ($M)": round(total_lift_check / 1e6, 2),
    })
    print(f"\n{s['Scenario']} — Tier Breakdown")
    display(pd.DataFrame(tbl).astype(str))

In [0]:
# ── 2027 SCENARIO UPSIDE — Ladder to Best Case (LBE, revenue) ────────────────
# EXH "2027 Scenario Upside" ported to LBE: revenue lift (not admissions),
# laddered over FORECAST_2027, 90% model-uncertainty CI.
import matplotlib.patches as mpatches
from matplotlib.lines import Line2D
import re

# CPX_GREEN isn't in the LBE Plots-Config palette (only EXH's) — define it here.
CPX_GREEN = "#2E7D32"

# ── Build plot frame from scenario_df (drop the zero-lift Baseline row) ───────
_ld = scenario_df[scenario_df["Type"] != "Baseline"].copy()

# Numeric lift ($M) and gain-CI ($M), re-centered on the lift.
# scenario_df CIs are on PROJECTED TOTAL; convert to CI around the GAIN by
# subtracting the projected point and adding the lift.
_ld["gain"]   = _ld["Scenario Lift ($M)"].astype(float)
_ld["ci_lo"]  = _ld["CI Low ($M)"].astype(float)  - _ld["Projected Rev ($M)"].astype(float) + _ld["gain"]
_ld["ci_hi"]  = _ld["CI High ($M)"].astype(float) - _ld["Projected Rev ($M)"].astype(float) + _ld["gain"]

df_plot = _ld.sort_values("gain", ascending=True).reset_index(drop=True)

# Replace "XX% Shift ..." with "Reshuffle" and strip any remaining "Co→Aw" / "Co->Aw" text
labels  = [re.sub(r'\d+%\s*Shift', 'Reshuffle', l) for l in df_plot["Scenario"].tolist()]
labels  = [re.sub(r'[:\s]*Co\s*[->→]+\s*Aw', '', l).strip() for l in labels]
labels  = ["Reshuffle Only" if l == "Reshuffle" else l for l in labels]
gains   = df_plot["gain"].to_numpy()
lows    = df_plot["ci_lo"].to_numpy()
highs   = df_plot["ci_hi"].to_numpy()
n       = len(df_plot)
ceiling = gains.max()

FORECAST_2027_M = FORECAST_2027 / 1e6   # 2027 projection base (Plots Config)

# ── Colour by Type (LBE casing) + best-case override ─────────────────────────
def bar_color(row):
    t = str(row["Type"]).lower()
    if t == "budget":       return CPX_NAVY
    if t == "combined":     return CPX_GREEN
    if t == "reshuffle":    return CPX_RED
    return CPX_MGREY

colors = [bar_color(df_plot.iloc[i]) for i in range(n)]
RECOMMENDED = labels[int(np.argmax(gains))]
for i in range(n):
    if labels[i] == RECOMMENDED:
        colors[i] = CPX_GOLD
    elif str(df_plot.iloc[i]["Type"]).lower() == "reshuffle":
        colors[i] = CPX_RED

# ── Plot ──────────────────────────────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(13, 4.6))
y      = np.arange(n)
floor  = min(gains.min(), 0.0)
span   = ceiling - floor
lab_pad = span * 0.012

pad     = ceiling * 0.06
x_lo    = floor - pad
x_hi    = x_lo + (ceiling - x_lo) / 0.65
band    = x_hi - ceiling
x_ci    = ceiling * 1.02
x_2026  = x_ci + band * 0.42
x_max   = x_hi

for i in range(n):
    is_rec = labels[i] == RECOMMENDED
    c = colors[i]
    g = gains[i]
    ax.barh(y[i], span, left=floor, height=0.72, color=CPX_LGREY, alpha=0.28,
            edgecolor="none", zorder=1)
    ax.barh(y[i], g, height=0.72, color=c,
            alpha=0.92 if is_rec else 0.8, edgecolor="none", zorder=2)
    if g < 0:
        ax.text(lab_pad, y[i], f"{g:+.2f}M",
                ha="left", va="center", fontsize=9, fontweight="bold", color=c, zorder=4)
    elif g >= span * 0.10:
        ax.text(g - lab_pad, y[i], f"{g:+.2f}M",
                ha="right", va="center",
                fontsize=10 if is_rec else 9, fontweight="bold", color=WHITE, zorder=4)
    else:
        ax.text(g + lab_pad, y[i], f"{g:+.2f}M",
                ha="left", va="center",
                fontsize=9, fontweight="bold", color=c, zorder=4)
    # Per-row 2027 projected total = 2027 forecast base + scenario media lift
    rev_27 = FORECAST_2027_M + g
    ax.text(x_ci, y[i], f"[{lows[i]:.2f}–{highs[i]:.2f}]",
            ha="left", va="center", fontsize=9, fontweight="bold", color=c, zorder=4, clip_on=False)
    ax.text(x_2026, y[i], f"2027: ${rev_27:.1f}M",
            ha="left", va="center", fontsize=8, fontweight="bold", color=CPX_GREEN, zorder=4, clip_on=False)

if floor < 0:
    ax.axvline(0, color=CPX_MGREY, lw=1.0, ls="-", alpha=0.6, zorder=3)

ax.axvline(ceiling, color=CPX_RED, lw=1.4, ls="--", alpha=0.8, zorder=5)
ax.text(ceiling, n - 0.42, f"Best case  +${ceiling:.2f}M",
        ha="center", va="bottom", fontsize=8.5, fontweight="bold",
        color=CPX_RED, zorder=6)

ax.set_yticks(y)
ytick_labels = [f"★ {l}" if l == RECOMMENDED else l for l in labels]
ax.set_yticklabels(ytick_labels, fontsize=9.5, fontweight="bold", color=CPX_DGREY)
ax.tick_params(axis="y", length=0, pad=8)
ax.set_xlim(x_lo, x_max)
ax.set_ylim(-0.7, n + 0.7)
ax.xaxis.set_major_formatter(mticker.FuncFormatter(lambda v, _: f"{v:+.1f}M"))
plt.setp(ax.get_xticklabels(), fontsize=8.5, fontweight="bold", color=CPX_DGREY)
for sp in ("top", "right", "left", "bottom"):
    ax.spines[sp].set_visible(False)
ax.grid(axis="x", alpha=0.14, color=CPX_LGREY)
ax.set_axisbelow(True)
ax.tick_params(axis="x", labelbottom=False, length=0)

# Header: make the 2027 base explicit
ax.text(x_lo, n + 0.35, f"2027 forecast base: ${FORECAST_2027_M:.1f}M   |   bars = scenario media lift on top of base",
        ha="left", va="bottom", fontsize=8, fontweight="bold", color=CPX_MGREY, zorder=4)

leg = [mpatches.Patch(facecolor=CPX_NAVY,  label="Budget uplift"),
       mpatches.Patch(facecolor=CPX_GREEN, label="Combined"),
       mpatches.Patch(facecolor=CPX_GOLD,  label="Best case scenario"),
       mpatches.Patch(facecolor=CPX_RED,   label="Reshuffle"),
       mpatches.Patch(facecolor=CPX_LGREY, alpha=0.28, label="Gap to best case"),
       Line2D([0], [0], color=CPX_RED,   lw=1.4, ls="--", alpha=0.8, label="Best case line"),
       Line2D([0], [0], color=CPX_DGREY, lw=1.5, ls="-",  label="[low–high] = 90% CI (lift)")]
fig.legend(handles=leg, fontsize=7.5, loc="upper center", bbox_to_anchor=(0.5, 0.90),
           ncol=4, frameon=True, framealpha=0.9, edgecolor=CPX_LGREY,
           borderpad=0.5, labelspacing=0.5, handlelength=1.5)

fig.suptitle("LBE 2027 Scenario Upside — Climb to Best Case (Revenue)",
             fontsize=13, fontweight="bold", color=CPX_NAVY, y=0.995)
fig.text(0.99, 0.005, "Cineplex Marketing Analytics  |  LBE MMM V5e",
         ha="right", va="bottom", fontsize=6, color=CPX_LGREY)

plt.tight_layout(rect=[0, 0, 1, 0.82])
save_fig(fig, "FINAL_2027_lbe_scenario_ladder_compact.png")
display(fig)
plt.close(fig)

In [0]:
# ── 2027 REVENUE BUILD — Baseline + Media Split (Pre-Scenario Lift) ────────────
# Single-panel LBE version of the EXH attendance-build strip.
# Revenue (not attendance); 2027 forecast only.
import matplotlib.patches as mpatches

CPX_GREY = "#8A8A8A"   # not in the LBE palette (only L/M/D grey) — define locally

# ── 2027 media share, reusing the annual-decomp logic ─────────────────────────
# decomp_df (from the Annual Revenue Decomposition cell) holds weekly media_*;
# 2025 media share of actual revenue is applied to the 2027 forecast, exactly as
# the decomposition cell does at its forecast-inheritance step.
_2025 = decomp_df[decomp_df["year"] == 2025]
_media_2025  = (_2025["media_Awareness"].sum() + _2025["media_Consideration"].sum())
_actual_2025 = _2025["total_rev"].sum()
_ms          = _media_2025 / _actual_2025            # 2025 media share, stable assumption

total = FORECAST_2027 / 1e6
media = FORECAST_2027 * _ms / 1e6
nm    = total - media

# ── Plot: single horizontal build bar ─────────────────────────────────────────
fig, ax = plt.subplots(figsize=(7, 2.4))

ax.barh(0, nm, height=0.32, color=CPX_GREY, alpha=0.65, edgecolor="none", zorder=2)
ax.barh(0, media, height=0.32, color=CPX_NAVY, alpha=0.85, edgecolor="none", zorder=2, left=nm)

ax.text(nm * 0.5, 0, f"Non-Media\n${nm:.1f}M  ({nm/total*100:.0f}%)",
        ha="center", va="center", fontsize=11, fontweight="bold", color="black", zorder=3)

_media_frac = media / total
if _media_frac >= 0.15:
    ax.text(nm + media * 0.5, 0, f"Media\n${media:.1f}M  ({_media_frac*100:.0f}%)",
            ha="center", va="center", fontsize=11, fontweight="bold", color=WHITE, zorder=3)
else:
    ax.text(total + total * 0.015, 0, f"Media\n${media:.1f}M  ({_media_frac*100:.0f}%)",
            ha="left", va="center", fontsize=11, fontweight="bold", color=CPX_NAVY, zorder=3)

ax.set_title(f"2027 (Forecast)\n${total:.1f}M", fontsize=12, fontweight="bold", color="black", pad=4)
ax.set_yticks([])
ax.set_ylim(-0.22, 0.22)
ax.set_xlim(0, total * 1.04)
ax.tick_params(axis="x", labelbottom=False, length=0)
for sp in ("top", "right", "left", "bottom"):
    ax.spines[sp].set_visible(False)
ax.grid(axis="x", alpha=0.12, color=CPX_LGREY)
ax.set_axisbelow(True)

leg = [mpatches.Patch(facecolor=CPX_GREY, alpha=0.65, label="Baseline / Non-Media"),
       mpatches.Patch(facecolor=CPX_NAVY, alpha=0.85, label="Media Attribution")]
fig.legend(handles=leg, fontsize=9, loc="lower center", bbox_to_anchor=(0.5, -0.02),
           ncol=2, framealpha=0.9, edgecolor=CPX_LGREY, borderpad=0.5)

fig.suptitle("LBE 2027 Revenue Build — Baseline + Media Split (Pre-Scenario Lift)",
             fontsize=13, fontweight="bold", color=CPX_NAVY, y=1.06)
fig.text(0.99, -0.10, "Cineplex Marketing Analytics  |  LBE MMM V5e",
         ha="right", va="bottom", fontsize=6, color=CPX_LGREY)

plt.tight_layout(rect=[0, 0.06, 1, 0.94])
save_fig(fig, "FINAL_lbe_revenue_build_2027.png")
display(fig)
plt.close(fig)

In [0]:
# ━━ LBE SCENARIO: SPEND & REVENUE BY TIER ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
# One chart per scenario (from cell 29). Each chart shows:
#   • Spend by tier: 2025 actual (solid) vs scenario (hatched)
#   • Revenue by tier: 2025 actual (solid) vs scenario (hatched)
#   • Delta labels on scenario bars
#   • Title includes total revenue lift & CI
import matplotlib.patches as mpatches
import matplotlib.ticker as mticker

# ── Compat shim: old name → new posterior scenario CI (cell 33) ──────────────
lbe_funnel_contribution_ci = lbe_scenario_ci

# ── 2027 projection base: forecast media contribution (matches cell 16/36) ──
# 2027 media revenue = FORECAST_2027 × 2025 media_ratio (same as decomp/ladder).
# Scenario media LIFT (from the model) is added on top of this forecast base.
_media_ratio_27 = (_media_2025 / _actual_2025) if ('_media_2025' in dir() and _actual_2025) \
                  else (lbe_baseline_rev / _lbe_rev_2025)
proj_2027_total   = FORECAST_2027
proj_2027_media   = FORECAST_2027 * _media_ratio_27
proj_2027_nonmedia = proj_2027_total - proj_2027_media

# ── Tier structure ───────────────────────────────────────────────────────────
FUNNELS_T      = ["Awareness", "Consideration"]
BRANDS_T       = ["TRR", "PDM"]
brand_tiers_t  = {"TRR": ["HPL", "LPL", "NL"], "PDM": ["OL", "NL"]}
FUNNEL_COLOR_T = {"Awareness": CPX_NAVY, "Consideration": CPX_RED}

# Build tier list: (funnel, brand, tier)
tier_list = []
for funnel in FUNNELS_T:
    for brand in BRANDS_T:
        for tier in brand_tiers_t[brand]:
            tier_list.append((funnel, brand, tier))

n_tiers_total = len(tier_list)

# ── 2025 actual spend & revenue per tier ────────────────────────────────────
actual_spend_tier = {}
for funnel, brand, tier in tier_list:
    actual_spend_tier[(funnel, brand, tier)] = eff_spend.get((brand, funnel, tier, 2025), 0.0)

# Funnel totals for proportional allocation
actual_aw_total = sum(v for (f, b, t), v in actual_spend_tier.items() if f == "Awareness")
actual_co_total = sum(v for (f, b, t), v in actual_spend_tier.items() if f == "Consideration")

# ── Funnel-level revenue function (for splitting total rev to Aw/Co) ────────
def _funnel_rev_split(aw_annual, co_annual):
    """Returns (aw_contribution, co_contribution) using the model."""
    # Total contribution
    total_rev, _, _ = lbe_funnel_contribution_ci(aw_annual, co_annual)
    # Awareness-only contribution
    aw_only, _, _ = lbe_funnel_contribution_ci(aw_annual, 0.0)
    # Consideration-only contribution
    co_only, _, _ = lbe_funnel_contribution_ci(0.0, co_annual)
    # Proportional split (handles interaction)
    denom = aw_only + co_only if (aw_only + co_only) > 0 else 1.0
    aw_share = aw_only / denom
    co_share = co_only / denom
    return total_rev * aw_share, total_rev * co_share

# ── 2025 actual media-attributed revenue per tier (model-consistent) ───────
# Uses the same funnel-split + spend-weight methodology as scenario allocation
# so that baseline and scenario are on the same basis and lifts sum correctly.
_aw_rev_2025, _co_rev_2025 = _funnel_rev_split(lbe_baseline_aw, lbe_baseline_co)
actual_rev_tier = {}
for funnel, brand, tier in tier_list:
    funnel_spend = actual_aw_total if funnel == "Awareness" else actual_co_total
    if funnel_spend > 0:
        w = actual_spend_tier[(funnel, brand, tier)] / funnel_spend
    else:
        w = 0.0
    funnel_rev = _aw_rev_2025 if funnel == "Awareness" else _co_rev_2025
    actual_rev_tier[(funnel, brand, tier)] = funnel_rev * w

# ── Compute scenario tier allocations ────────────────────────────────────────
def scenario_tier_alloc(aw_annual, co_annual):
    """Allocate scenario spend & revenue to tiers proportionally."""
    # Spend allocation: scale each tier by funnel ratio
    aw_factor = aw_annual / actual_aw_total if actual_aw_total > 0 else 1.0
    co_factor = co_annual / actual_co_total if actual_co_total > 0 else 1.0

    scen_spend = {}
    for funnel, brand, tier in tier_list:
        factor = aw_factor if funnel == "Awareness" else co_factor
        scen_spend[(funnel, brand, tier)] = actual_spend_tier[(funnel, brand, tier)] * factor

    # Revenue allocation: total scenario rev split by funnel, then by tier spend weight
    aw_rev_total, co_rev_total = _funnel_rev_split(aw_annual, co_annual)

    scen_rev = {}
    for funnel, brand, tier in tier_list:
        funnel_spend = sum(scen_spend[(f, b, t)] for f, b, t in tier_list if f == funnel)
        if funnel_spend > 0:
            w = scen_spend[(funnel, brand, tier)] / funnel_spend
        else:
            w = 0.0
        funnel_rev = aw_rev_total if funnel == "Awareness" else co_rev_total
        scen_rev[(funnel, brand, tier)] = funnel_rev * w

    return scen_spend, scen_rev

# ── Plotting ─────────────────────────────────────────────────────────────────
fmt_K = mticker.FuncFormatter(lambda v, _: f"${v/1e3:.0f}K")

for s in scenarios:
    if s["Type"] == "Baseline":
        continue  # skip baseline (no change to show)

    scen_spend, scen_rev = scenario_tier_alloc(s["aw"], s["co"])
    total_rev_scen, ci_lo, ci_hi = lbe_funnel_contribution_ci(s["aw"], s["co"])
    lift = total_rev_scen - lbe_baseline_rev
    lift_pct = lift / lbe_baseline_rev * 100

    # CI as % of baseline
    ci_lo_pct = (ci_lo - lbe_baseline_rev) / lbe_baseline_rev * 100
    ci_hi_pct = (ci_hi - lbe_baseline_rev) / lbe_baseline_rev * 100

    fig, (ax_sp, ax_rv) = plt.subplots(1, 2, figsize=(18, 7))
    fig.patch.set_facecolor(WHITE)

    bar_w = 0.35
    x = np.arange(n_tiers_total)

    # ── LEFT PANEL: Spend by Tier (actual vs scenario) ─────────────────────
    actual_vals_sp = np.array([actual_spend_tier[(f, b, t)] for f, b, t in tier_list])
    scen_vals_sp   = np.array([scen_spend[(f, b, t)] for f, b, t in tier_list])
    colors         = [FUNNEL_COLOR_T[f] for f, b, t in tier_list]

    for i in range(n_tiers_total):
        ax_sp.bar(x[i] - bar_w/2, actual_vals_sp[i], bar_w,
                  color=colors[i], alpha=0.9, edgecolor=WHITE, lw=0.4)
        ax_sp.bar(x[i] + bar_w/2, scen_vals_sp[i], bar_w,
                  color=colors[i], alpha=0.45, edgecolor=colors[i],
                  lw=0.6, hatch="///")

    # Spend delta labels
    for i in range(n_tiers_total):
        delta = scen_vals_sp[i] - actual_vals_sp[i]
        if abs(delta) > 0:
            sign = "+" if delta >= 0 else ""
            lbl = f"{sign}${delta/1e3:.0f}K" if abs(delta) < 1e6 else f"{sign}${delta/1e6:.1f}M"
            ax_sp.annotate(lbl, xy=(x[i] + bar_w/2, scen_vals_sp[i]),
                           xytext=(0, 4), textcoords="offset points",
                           ha="center", fontsize=7.5, fontweight="bold",
                           color=CPX_NAVY if delta >= 0 else CPX_RED)

    xlabels = [f"{b}\n{t}" for f, b, t in tier_list]
    ax_sp.set_xticks(x)
    ax_sp.set_xticklabels(xlabels, fontsize=10, fontweight="bold")
    ax_sp.set_ylabel("Spend ($K)", fontsize=12, fontweight="bold")
    ax_sp.yaxis.set_major_formatter(fmt_K)
    ax_sp.spines["top"].set_visible(False)
    ax_sp.spines["right"].set_visible(False)
    plt.setp(ax_sp.get_yticklabels(), fontsize=10, fontweight="bold")
    ax_sp.grid(False)
    n_aw = sum(1 for f, _, _ in tier_list if f == "Awareness")
    ax_sp.axvline(n_aw - 0.5, color="#CCCCCC", lw=1.2, linestyle="--")
    y_top_sp = max(actual_vals_sp.max(), scen_vals_sp.max()) * 1.25
    ax_sp.set_ylim(0, y_top_sp)
    ax_sp.text(n_aw / 2 - 0.5, y_top_sp * 0.96, "Awareness",
               ha="center", fontsize=10, fontweight="bold", color=CPX_NAVY)
    ax_sp.text(n_aw + (n_tiers_total - n_aw) / 2 - 0.5, y_top_sp * 0.96, "Consideration",
               ha="center", fontsize=10, fontweight="bold", color=CPX_RED)
    ax_sp.set_title("Spend by Tier", fontsize=13, fontweight="bold", color=CPX_NAVY, pad=8)

    # ── RIGHT PANEL: Total Revenue Lift by Tier ──────────────────────────────
    # Shows how the total revenue lift distributes across tiers
    lift_per_tier = np.array([
        scen_rev[(f, b, t)] - actual_rev_tier[(f, b, t)]
        for f, b, t in tier_list
    ])
    total_lift_check = lift_per_tier.sum()

    for i in range(n_tiers_total):
        bar_color = colors[i]
        ax_rv.bar(x[i], lift_per_tier[i], bar_w * 1.5,
                  color=bar_color,
                  alpha=0.75 if lift_per_tier[i] >= 0 else 0.5,
                  edgecolor=bar_color, lw=0.6)

    # Value labels on each bar
    for i in range(n_tiers_total):
        val = lift_per_tier[i]
        sign = "+" if val >= 0 else ""
        lbl = f"{sign}${val/1e6:.2f}M"
        y_pos = val if val >= 0 else val
        va = "bottom" if val >= 0 else "top"
        ax_rv.annotate(lbl, xy=(x[i], y_pos),
                       xytext=(0, 4 if val >= 0 else -4),
                       textcoords="offset points",
                       ha="center", va=va, fontsize=7.5, fontweight="bold",
                       color=CPX_NAVY if val >= 0 else CPX_RED)

    ax_rv.set_xticks(x)
    ax_rv.set_xticklabels(xlabels, fontsize=10, fontweight="bold")
    ax_rv.set_ylabel("Revenue Lift ($M)", fontsize=12, fontweight="bold")
    ax_rv.yaxis.set_major_formatter(mticker.FuncFormatter(lambda v, _: f"${v/1e6:.2f}M"))
    ax_rv.spines["top"].set_visible(False)
    ax_rv.spines["right"].set_visible(False)
    plt.setp(ax_rv.get_yticklabels(), fontsize=10, fontweight="bold")
    ax_rv.grid(False)
    ax_rv.axhline(0, color="#666666", lw=0.8)
    ax_rv.axvline(n_aw - 0.5, color="#CCCCCC", lw=1.2, linestyle="--")
    y_max_rv = max(abs(lift_per_tier.max()), abs(lift_per_tier.min()), 0.01) * 1.4
    ax_rv.set_ylim(-y_max_rv if lift_per_tier.min() < 0 else 0, y_max_rv)
    ax_rv.text(n_aw / 2 - 0.5, y_max_rv * 0.92, "Awareness",
               ha="center", fontsize=10, fontweight="bold", color=CPX_NAVY)
    ax_rv.text(n_aw + (n_tiers_total - n_aw) / 2 - 0.5, y_max_rv * 0.92, "Consideration",
               ha="center", fontsize=10, fontweight="bold", color=CPX_RED)
    ax_rv.set_title(f"Total Revenue Lift by Tier (Total: +${total_lift_check/1e6:.2f}M)",
                    fontsize=13, fontweight="bold", color=CPX_NAVY, pad=8)

    # Legend
    legend_handles = [
        mpatches.Patch(facecolor=CPX_NAVY, alpha=0.9, label="Awareness"),
        mpatches.Patch(facecolor=CPX_RED, alpha=0.9, label="Consideration"),
        mpatches.Patch(facecolor="#666666", alpha=0.9, label="2025 Actual (solid)"),
        mpatches.Patch(facecolor="#666666", alpha=0.45, hatch="///", label="Scenario (hatched)"),
    ]
    fig.legend(handles=legend_handles, fontsize=11, loc="upper right",
              bbox_to_anchor=(0.98, 0.98), frameon=False)

    proj_total_scen = proj_2027_total + lift          # 2027 forecast + scenario media lift
    fig.suptitle(
        f"LBE MMM 2027 Projection \u2014 {s['Scenario']}\n"
        f"2027 base ${proj_2027_total/1e6:.1f}M  |  Scenario media lift: +${lift/1e6:.2f}M ({lift_pct:+.1f}%)  "
        f"|  Projected 2027: ${proj_total_scen/1e6:.1f}M  |  90% CI: [{ci_lo_pct:+.1f}%, {ci_hi_pct:+.1f}%]",
        fontsize=13.5, fontweight="bold", color=CPX_NAVY, y=1.03
    )
    fig.text(0.5, -0.02,
        "Tier revenue = 2027 forecast split to media (2025 ratio) \u2192 funnel (model) \u2192 tier (2025 spend share). "
        "Tier detail is proportional, not modeled at tier level.",
        ha="center", fontsize=8, color=CPX_MGREY, style="italic")

    plt.tight_layout()
    save_fig(fig, f"lbe_2027_scenario_{s['Scenario'].lower().replace(' ', '_').replace('%','pct')}.png")
    display(fig)
    plt.close(fig)

    # ── Table: Spend & Revenue Lift by Tier ───────────────────────────────────
    tbl = []
    for f, b, t in tier_list:
        act_sp = actual_spend_tier[(f, b, t)]
        scn_sp = scen_spend[(f, b, t)]
        act_rv = actual_rev_tier[(f, b, t)]
        scn_rv_val = scen_rev[(f, b, t)]
        tbl.append({
            "Funnel": f,
            "Brand": b,
            "Tier": t,
            "2025 Spend ($K)": round(act_sp / 1e3, 1),
            "Scenario Spend ($K)": round(scn_sp / 1e3, 1),
            "Spend Δ ($K)": round((scn_sp - act_sp) / 1e3, 1),
            "2025 Media Rev ($M)": round(act_rv / 1e6, 3),
            "Scenario Media Rev ($M)": round(scn_rv_val / 1e6, 3),
            "Rev Lift ($M)": round((scn_rv_val - act_rv) / 1e6, 3),
        })
    # Totals row
    tbl.append({
        "Funnel": "", "Brand": "", "Tier": "TOTAL",
        "2025 Spend ($K)": round(sum(actual_spend_tier.values()) / 1e3, 1),
        "Scenario Spend ($K)": round(sum(scen_spend.values()) / 1e3, 1),
        "Spend Δ ($K)": round((sum(scen_spend.values()) - sum(actual_spend_tier.values())) / 1e3, 1),
        "2025 Media Rev ($M)": round(sum(actual_rev_tier.values()) / 1e6, 2),
        "Scenario Media Rev ($M)": round(sum(scen_rev.values()) / 1e6, 2),
        "Rev Lift ($M)": round(total_lift_check / 1e6, 2),
    })
    print(f"\n{s['Scenario']} — Tier Breakdown")
    display(pd.DataFrame(tbl).astype(str))

In [0]:
# ━━ LBE BUDGET SCENARIO — DEPENDENCIES (POSTERIOR-BASED) ━━━━━━━━━━━━━━━━━━━━━━━━
# Scenarios now flow through the fitted V5e posteriors — NOT a re-derived
# single-target LBE curve. For a given annual (Aw, Co) budget per brand we:
#   1. scale that brand's ACTUAL 2025 weekly spend shape by (scenario/2025)
#   2. push every posterior draw through the exact V5e reconstruction
#      (adstock → Hill w/ fixed anchor → beta → incr RPT & incr Txns → revenue)
#   3. sum TRR + PDM per draw → LBE incremental-revenue distribution
# CIs are genuine posterior credible intervals, dual-target, per-brand.

# ── 1. Per-brand 2025 weekly spend shapes + normalization (fitted space) ──────
_mask_2025 = d["week_date"].dt.year == 2025
SCEN_BRANDS = ["TRR", "PDM"]

scen_ctx = {}   # per-brand cached context for the scenario reconstruction
for brand in SCEN_BRANDS:
    M = d[[f"{brand}_Awareness_spend", f"{brand}_Consideration_spend"]].values.astype(float)
    X = d[CONTROLS].values.astype(float)
    n_tr = int(len(d) * TRAIN_FRAC)
    m_s = M[:n_tr].max(0); m_s[m_s == 0] = 1.0          # SAME normalizer as fit
    x_mu, x_sd = X[:n_tr].mean(0), X[:n_tr].std(0); x_sd[x_sd == 0] = 1.0
    Xz = np.nan_to_num((X - x_mu) / x_sd)

    # 2025 weekly spend shape (actual $ per week, this brand)
    aw_wk_2025 = d.loc[_mask_2025, f"{brand}_Awareness_spend"].values.astype(float)
    co_wk_2025 = d.loc[_mask_2025, f"{brand}_Consideration_spend"].values.astype(float)

    scen_ctx[brand] = {
        "m_s": m_s, "Xz": Xz,
        "aw_wk_2025": aw_wk_2025, "co_wk_2025": co_wk_2025,
        "aw_annual_2025": float(aw_wk_2025.sum()),
        "co_annual_2025": float(co_wk_2025.sum()),
        "anchor_aw": results[brand]["anchor_aw"],
        "anchor_co": results[brand]["anchor_co"],
        "avg_rpt":   results[brand]["avg_rpt"],
        "avg_txn":   results[brand]["avg_txn"],
    }

# LBE 2025 annual budget (for baseline reference / scenario multipliers)
lbe_aw_2025 = sum(scen_ctx[b]["aw_annual_2025"] for b in SCEN_BRANDS)
lbe_co_2025 = sum(scen_ctx[b]["co_annual_2025"] for b in SCEN_BRANDS)

# ── 2. Pull posterior draws once per brand/target (paired RPT & TXN) ───────────
_N_SCEN_DRAWS = 1000
def _stack_draws(idata, keys, n_draws=_N_SCEN_DRAWS, seed=42):
    po = idata.posterior
    if hasattr(po, "to_dataset"):
        po = po.to_dataset()
    po = po.stack(sample=("chain", "draw"))
    total = po.sizes["sample"]
    idx = np.random.default_rng(seed).choice(total, size=min(n_draws, total), replace=False)
    out = {}
    for k in keys:
        v = po[k].values
        out[k] = v[..., idx]      # last axis = sample
    return out, len(idx)

scen_draws = {}
for brand in SCEN_BRANDS:
    r = results[brand]
    d_rpt, nR = _stack_draws(r["idata_rpt"], ["adstock_aw","adstock_co","beta_media","beta_ctrl","intercept"])
    d_txn, nT = _stack_draws(r["idata_txn"], ["adstock_aw","adstock_co","beta_media","beta_ctrl","intercept"])
    scen_draws[brand] = {"rpt": d_rpt, "txn": d_txn, "n": min(nR, nT)}

# ── 3. Core: per-draw LBE incremental revenue for a scenario budget ───────────
def lbe_media_rev_draws(aw_annual, co_annual):
    """Return per-draw array of LBE incremental revenue ($) for an annual
    (Awareness, Consideration) LBE budget. Splits the LBE budget across brands
    by each brand's 2025 funnel-spend share, scales that brand's 2025 weekly
    shape, and runs every posterior draw through the V5e reconstruction."""
    lbe_incr = None
    for brand in SCEN_BRANDS:
        ctx = scen_ctx[brand]; dr = scen_draws[brand]; nD = dr["n"]

        # LBE budget → brand budget via 2025 funnel share
        aw_share = ctx["aw_annual_2025"] / lbe_aw_2025 if lbe_aw_2025 > 0 else 0.0
        co_share = ctx["co_annual_2025"] / lbe_co_2025 if lbe_co_2025 > 0 else 0.0
        aw_brand = aw_annual * aw_share
        co_brand = co_annual * co_share

        # Scenario weekly shape = 2025 shape × multiplier (approach a)
        aw_mult = aw_brand / ctx["aw_annual_2025"] if ctx["aw_annual_2025"] > 0 else 0.0
        co_mult = co_brand / ctx["co_annual_2025"] if ctx["co_annual_2025"] > 0 else 0.0
        aw_wk = ctx["aw_wk_2025"] * aw_mult
        co_wk = ctx["co_wk_2025"] * co_mult

        # Normalize into fitted space (same m_s as fit)
        aw_nz = aw_wk / ctx["m_s"][0]
        co_nz = co_wk / ctx["m_s"][1]

        # Controls at 2025 weeks (align with the weekly window we're scoring)
        Xz_2025 = ctx["Xz"][_mask_2025.values]

        h_aw, h_co = ctx["anchor_aw"], ctx["anchor_co"]
        avg_rpt, avg_txn = ctx["avg_rpt"], ctx["avg_txn"]

        aw_incr_rpt = np.empty(nD); co_incr_rpt = np.empty(nD)
        aw_incr_txn = np.empty(nD); co_incr_txn = np.empty(nD)

        for tgt, a_out in (("rpt", (aw_incr_rpt, co_incr_rpt)),
                           ("txn", (aw_incr_txn, co_incr_txn))):
            g = dr[tgt]
            for i in range(nD):
                a_aw = float(g["adstock_aw"][i]); a_co = float(g["adstock_co"][i])
                b = g["beta_media"][:, i]; gam = g["beta_ctrl"][:, i]; ic = float(g["intercept"][i])
                aw_full = adstock_np(aw_nz, a_aw)
                co_full = adstock_np(co_nz, a_co)
                aw_sat = aw_full / (aw_full + h_aw)
                co_sat = co_full / (co_full + h_co)
                m_aw = b[0] * aw_sat; m_co = b[1] * co_sat
                log_pred = ic + Xz_2025 @ gam + m_aw + m_co
                pred = np.exp(log_pred)
                a_out[0][i] = (pred - np.exp(log_pred - m_aw)).sum()
                a_out[1][i] = (pred - np.exp(log_pred - m_co)).sum()

        # Combine to revenue per draw (dual-target: RPT × Txns)
        aw_rev_d = aw_incr_txn * avg_rpt + avg_txn * aw_incr_rpt
        co_rev_d = co_incr_txn * avg_rpt + avg_txn * co_incr_rpt
        brand_rev_d = aw_rev_d + co_rev_d

        lbe_incr = brand_rev_d if lbe_incr is None else lbe_incr + brand_rev_d
    return lbe_incr

# ── 4. Point-estimate helper (posterior median) — drop-in for old lbe_media_rev
def lbe_media_rev(aw_annual, co_annual):
    """Posterior-median LBE incremental revenue ($) for an annual budget.
    Signature-compatible with the old deterministic helper so downstream
    cells (marginal ROI, tier split) keep working — but now posterior-backed."""
    return float(np.median(lbe_media_rev_draws(aw_annual, co_annual)))

# ── 5. Verification: baseline scenario must reconcile to 2025 decomp ──────────
_base_draws = lbe_media_rev_draws(lbe_aw_2025, lbe_co_2025)
_base_med = float(np.median(_base_draws))
_base_lo, _base_hi = np.percentile(_base_draws, [5, 95])

# 2025 decomp target (same source as before)
if "decomp_df" in dir():
    _dec_2025 = decomp_df[decomp_df["year"] == 2025]
    _decomp_target = float(_dec_2025["media_Awareness"].sum() + _dec_2025["media_Consideration"].sum())
else:
    _decomp_target = float("nan")

print("LBE Budget Scenario — POSTERIOR-BASED (per-brand V5e reconstruction)")
print(f"  Brands:              {SCEN_BRANDS}")
print(f"  Draws per brand:     {scen_draws['TRR']['n']}")
print(f"  LBE 2025 Aw budget:  ${lbe_aw_2025/1e6:>8.2f}M")
print(f"  LBE 2025 Co budget:  ${lbe_co_2025/1e6:>8.2f}M")
print(f"\n  Baseline (2025 spend) LBE incremental revenue:")
print(f"    posterior median:  ${_base_med/1e6:>8.2f}M")
print(f"    90% CI:            [${_base_lo/1e6:.2f}M, ${_base_hi/1e6:.2f}M]")
print(f"    2025 decomp target: ${_decomp_target/1e6:>8.2f}M")
if _decomp_target == _decomp_target:  # not nan
    _reconc = _base_med / _decomp_target if _decomp_target else float("nan")
    print(f"    reconciliation:     {_reconc:.2f}× (median / decomp target)")
    print(f"    → NOTE: posterior baseline need not equal the point-estimate decomp exactly;")
    print(f"      the decomp uses posterior-mean params, this uses the full draw distribution.")

In [0]:
# ━━ LBE RESHUFFLE — MULTI-OBJECTIVE PARETO (VECTORIZED / PRECOMPUTED) ━━━━━━━━━━
# 4 independent levers: TRR-Aw, TRR-Co, PDM-Aw, PDM-Co.
# Objectives: (1) minimize total spend, (2) maximize posterior-median attributable
# revenue. Reports Pareto set + exact $ recommendations, each with 90% posterior CI.
#
# SPEED: exploits two facts so 4000 candidates run in ~10s (was ~9min):
#   • adstock is LINEAR in spend  → adstock(shape×mult) = mult×adstock(shape).
#     Convolve the unit 2025 shape ONCE per draw; scale per candidate.
#   • controls (Xz@gamma) + intercept are spend-independent → precompute once.
#   • Hill saturation (nonlinear) + beta applied per candidate (cheap).
# Numerically identical to the per-draw loop (validated to 1e-12).
try:
    from tqdm.notebook import tqdm
except Exception:
    from tqdm import tqdm

# ── 1. Precompute per-brand draw-level base tensors (once) ────────────────────
# For each brand: adstock of the UNIT 2025 weekly shape per draw (D,W), plus the
# spend-independent log baseline (intercept + Xz@gamma) per draw (D,W).
_ADSTOCK_L = ADSTOCK_L

def _adstock_draws(x, alphas, L=_ADSTOCK_L):
    """x:(W,) unit weekly shape ; alphas:(D,) per-draw decay → (D,W) adstock."""
    D = alphas.shape[0]
    out = np.tile(x.astype(float), (D, 1))
    lags = np.arange(1, L)
    ap = alphas[:, None] ** lags[None, :]
    for j, lag in enumerate(lags):
        out[:, lag:] += ap[:, j][:, None] * x[None, :-lag]
    return out

precomp = {}
for brand in SCEN_BRANDS:
    ctx = scen_ctx[brand]; dr = scen_draws[brand]; nD = dr["n"]
    Xz_2025 = ctx["Xz"][_mask_2025.values]           # (W, K)
    # unit-normalized 2025 weekly shape (spend/m_s at mult=1)
    aw_unit = ctx["aw_wk_2025"] / ctx["m_s"][0]      # (W,)
    co_unit = ctx["co_wk_2025"] / ctx["m_s"][1]      # (W,)

    pc = {}
    for tgt in ("rpt", "txn"):
        g = dr[tgt]
        a_aw = g["adstock_aw"]; a_co = g["adstock_co"]      # (D,)
        b = g["beta_media"]; gam = g["beta_ctrl"]; ic = g["intercept"]  # (2,D),(K,D),(D,)
        aw_ad = _adstock_draws(aw_unit, a_aw)               # (D,W)  unit adstock
        co_ad = _adstock_draws(co_unit, a_co)               # (D,W)
        ctrl  = (Xz_2025 @ gam).T                           # (D,W)  spend-independent
        base_log = ic[:, None] + ctrl                       # (D,W)
        pc[tgt] = {
            "aw_ad": aw_ad, "co_ad": co_ad, "base_log": base_log,
            "b_aw": b[0], "b_co": b[1],                      # (D,)
            "h_aw": ctx["anchor_aw"], "h_co": ctx["anchor_co"],
        }
    precomp[brand] = {"rpt": pc["rpt"], "txn": pc["txn"],
                      "avg_rpt": ctx["avg_rpt"], "avg_txn": ctx["avg_txn"], "nD": nD}

# ── 2. Fast per-candidate LBE revenue draws (4 levers) ───────────────────────
def _incr_for_target(pc, aw_mult, co_mult):
    """Return (aw_incr(D,), co_incr(D,)) for one target given spend multipliers."""
    aw_ad = pc["aw_ad"] * aw_mult          # linearity: scale precomputed adstock
    co_ad = pc["co_ad"] * co_mult
    aw_sat = aw_ad / (aw_ad + pc["h_aw"])
    co_sat = co_ad / (co_ad + pc["h_co"])
    m_aw = pc["b_aw"][:, None] * aw_sat
    m_co = pc["b_co"][:, None] * co_sat
    lp = pc["base_log"] + m_aw + m_co
    pr = np.exp(lp)
    aw_incr = (pr - np.exp(lp - m_aw)).sum(1)
    co_incr = (pr - np.exp(lp - m_co)).sum(1)
    return aw_incr, co_incr

def brandfunnel_rev_draws(budgets):
    """budgets: {(brand,funnel): annual_$}. → per-draw LBE incremental revenue ($),
    reconciled to decomp. Vectorized/precomputed; matches the loop to 1e-12."""
    lbe = None
    for brand in SCEN_BRANDS:
        ctx = scen_ctx[brand]; P = precomp[brand]
        aw_mult = budgets[(brand,"aw")] / ctx["aw_annual_2025"] if ctx["aw_annual_2025"]>0 else 0.0
        co_mult = budgets[(brand,"co")] / ctx["co_annual_2025"] if ctx["co_annual_2025"]>0 else 0.0
        aw_ir, co_ir = _incr_for_target(P["rpt"], aw_mult, co_mult)
        aw_it, co_it = _incr_for_target(P["txn"], aw_mult, co_mult)
        aw_rev = aw_it * P["avg_rpt"] + P["avg_txn"] * aw_ir
        co_rev = co_it * P["avg_rpt"] + P["avg_txn"] * co_ir
        brand_rev = aw_rev + co_rev
        lbe = brand_rev if lbe is None else lbe + brand_rev
    return lbe * recon_factor

# ── 3. Current 2025 per-lever budgets (reference) ────────────────────────────
LEVERS = [("TRR","aw"), ("TRR","co"), ("PDM","aw"), ("PDM","co")]
base_budget = {
    ("TRR","aw"): scen_ctx["TRR"]["aw_annual_2025"],
    ("TRR","co"): scen_ctx["TRR"]["co_annual_2025"],
    ("PDM","aw"): scen_ctx["PDM"]["aw_annual_2025"],
    ("PDM","co"): scen_ctx["PDM"]["co_annual_2025"],
}
base_total = sum(base_budget.values())
_cur = brandfunnel_rev_draws(base_budget)
cur_med = float(np.median(_cur)); cur_lo, cur_hi = np.percentile(_cur, [5, 95])

# ── 4. Sample candidates ─────────────────────────────────────────────────────
N_CAND = 4000
rng = np.random.default_rng(7)
LO_MULT, HI_MULT = 0.5, 1.5

cand_rows = []
for _ in tqdm(range(N_CAND), desc="Pareto search", unit="alloc"):
    budgets = {lv: base_budget[lv] * rng.uniform(LO_MULT, HI_MULT) for lv in LEVERS}
    draws = brandfunnel_rev_draws(budgets)
    cand_rows.append({
        "budgets": budgets, "total": sum(budgets.values()),
        "rev_med": float(np.median(draws)),
        "rev_lo":  float(np.percentile(draws, 5)),
        "rev_hi":  float(np.percentile(draws, 95)),
    })
cand_rows.append({"budgets": base_budget, "total": base_total,
                  "rev_med": cur_med, "rev_lo": cur_lo, "rev_hi": cur_hi})

# ── 5. Pareto front: minimize total, maximize rev_med ────────────────────────
def pareto_mask(rows):
    tot = np.array([r["total"] for r in rows]); rev = np.array([r["rev_med"] for r in rows])
    n = len(rows); dominated = np.zeros(n, dtype=bool)
    for i in range(n):
        dom = (tot <= tot[i]) & (rev >= rev[i]) & ((tot < tot[i]) | (rev > rev[i]))
        dominated[i] = dom.any()
    return ~dominated

pareto = [r for r, m in zip(cand_rows, pareto_mask(cand_rows)) if m]
pareto.sort(key=lambda r: r["total"])

# ── 6. Recommendations ───────────────────────────────────────────────────────
def alloc_str(b): return "  ".join(f"{lv[0]}-{lv[1].upper()}:${b[lv]/1e3:.0f}K" for lv in LEVERS)
def distinguishable(r):
    if r is None: return "n/a"
    overlap = not (r["rev_lo"] > cur_hi or r["rev_hi"] < cur_lo)
    return "OVERLAPS current (not distinguishable)" if overlap else "DISTINCT from current"

match_pts  = [r for r in pareto if r["rev_med"] >= cur_med]
rec_minspend = min(match_pts, key=lambda r: r["total"]) if match_pts else None
budget_pts = [r for r in pareto if r["total"] <= base_total * 1.001]
rec_maxrev = max(budget_pts, key=lambda r: r["rev_med"]) if budget_pts else None

print("="*84)
print("  LBE RESHUFFLE — MULTI-OBJECTIVE PARETO (min spend / max attributable revenue)")
print("="*84)
print(f"  Levers: {[f'{b}-{f}' for b,f in LEVERS]}")
print(f"  Candidates: {N_CAND}  |  Pareto-efficient: {len(pareto)}  |  range {LO_MULT}-{HI_MULT}x of 2025")
print()
print(f"  CURRENT 2025 MIX (reference)")
print(f"    {alloc_str(base_budget)}")
print(f"    Total spend: ${base_total/1e6:.2f}M   Attr. revenue: ${cur_med/1e6:.2f}M  "
      f"[90% CI ${cur_lo/1e6:.2f}-${cur_hi/1e6:.2f}M]")
print()
for tag, rec, mode in [("A — hold revenue, minimize spend", rec_minspend, "save"),
                       ("B — hold spend, maximize revenue", rec_maxrev, "gain")]:
    print(f"  ── RECOMMENDATION {tag} ─────────────────")
    if rec:
        b = rec
        if mode == "save":
            delta = base_total - b["total"]
            print(f"    {alloc_str(b['budgets'])}")
            print(f"    Total spend: ${b['total']/1e6:.2f}M  (saves ${delta/1e6:.2f}M, {delta/base_total*100:.1f}% less)")
        else:
            delta = b["rev_med"] - cur_med
            print(f"    {alloc_str(b['budgets'])}")
            print(f"    Total spend: ${b['total']/1e6:.2f}M (≈current)  (+${delta/1e6:.2f}M rev vs current)")
        print(f"    Attr. revenue: ${b['rev_med']/1e6:.2f}M  [90% CI ${b['rev_lo']/1e6:.2f}-${b['rev_hi']/1e6:.2f}M]")
        print(f"    vs current: {distinguishable(b)}")
        for lv in LEVERS:
            dl = b['budgets'][lv] - base_budget[lv]
            print(f"      {lv[0]}-{lv[1].upper():<3} {dl/1e3:>+7.0f}K  ({dl/base_budget[lv]*100:>+5.1f}%)")
    else:
        print("    (no qualifying Pareto point in search range)")
    print()

# ── 7. Frontier table (thinned) ──────────────────────────────────────────────
step = max(1, len(pareto)//12)
frontier_tbl = [{
    "Total Spend ($M)": round(r["total"]/1e6,2), "Attr Rev ($M)": round(r["rev_med"]/1e6,2),
    "CI Low ($M)": round(r["rev_lo"]/1e6,2), "CI High ($M)": round(r["rev_hi"]/1e6,2),
    "TRR-Aw ($K)": round(r["budgets"][("TRR","aw")]/1e3), "TRR-Co ($K)": round(r["budgets"][("TRR","co")]/1e3),
    "PDM-Aw ($K)": round(r["budgets"][("PDM","aw")]/1e3), "PDM-Co ($K)": round(r["budgets"][("PDM","co")]/1e3),
} for r in pareto[::step]]
pareto_df = pd.DataFrame(frontier_tbl)
display(pareto_df)

print("\n  READ-OUT:")
print("  • 'OVERLAPS current' = point-estimate gain/saving is real but NOT separable")
print("    from current mix at 90% posterior. Directional, not a defensible commitment.")
print("  • Watch which lever the optimizer loads — PDM-Co loading corroborates the")
print("    cell-34 marginal-headroom finding from an independent direction.")

In [0]:
# ── LBE RESHUFFLE — PARETO FRONTIER PLOT (spend vs attributable revenue) ──────
# Median frontier + 90% posterior CI band. Current mix + both recommendations
# marked. Second panel: marginal efficiency (ΔRev/ΔSpend) along the frontier.
import matplotlib.ticker as mticker

# Full Pareto arrays (sorted by spend already)
p_spend = np.array([r["total"]   for r in pareto]) / 1e6
p_med   = np.array([r["rev_med"] for r in pareto]) / 1e6
p_lo    = np.array([r["rev_lo"]  for r in pareto]) / 1e6
p_hi    = np.array([r["rev_hi"]  for r in pareto]) / 1e6

fig, (ax, ax2) = plt.subplots(1, 2, figsize=(16, 6),
                              gridspec_kw={"width_ratios": [1.55, 1]})
fig.patch.set_facecolor(WHITE)

# ── LEFT: frontier + CI band ─────────────────────────────────────────────────
ax.fill_between(p_spend, p_lo, p_hi, color=CPX_NAVY, alpha=0.10,
                label="90% posterior CI")
ax.plot(p_spend, p_med, color=CPX_NAVY, lw=2.4, zorder=3,
        label="Pareto frontier (median)")

# Current mix
ax.scatter([base_total/1e6], [cur_med/1e6], s=130, color=CPX_MGREY,
           edgecolor="white", lw=1.5, zorder=6, label="Current 2025 mix")
ax.errorbar([base_total/1e6], [cur_med/1e6],
            yerr=[[ (cur_med-cur_lo)/1e6 ], [ (cur_hi-cur_med)/1e6 ]],
            fmt="none", ecolor=CPX_MGREY, elinewidth=1.2, capsize=4, zorder=5, alpha=0.7)

# Recommendations
if rec_minspend:
    ax.scatter([rec_minspend["total"]/1e6], [rec_minspend["rev_med"]/1e6],
               s=150, marker="D", color=CPX_GOLD, edgecolor=CPX_DGREY, lw=1.2,
               zorder=7, label="Rec A — min spend @ current rev")
if rec_maxrev:
    ax.scatter([rec_maxrev["total"]/1e6], [rec_maxrev["rev_med"]/1e6],
               s=150, marker="^", color=CPX_RED, edgecolor=CPX_DGREY, lw=1.2,
               zorder=7, label="Rec B — max rev @ current spend")

# Honest annotation: CI overlap
ax.annotate(
    "Recommendations sit inside the current mix's\nCI band → gains not statistically distinguishable",
    xy=(base_total/1e6, cur_med/1e6), xytext=(base_total/1e6 + 0.35, cur_med/1e6 - 3.4),
    fontsize=9, color=CPX_DGREY, fontweight="bold",
    arrowprops=dict(arrowstyle="->", color=CPX_DGREY, lw=1.0), zorder=8)

ax.set_xlabel("Total LBE media spend ($M)", fontsize=12, fontweight="bold")
ax.set_ylabel("Attributable revenue ($M)", fontsize=12, fontweight="bold")
ax.xaxis.set_major_formatter(mticker.FuncFormatter(lambda v,_: f"${v:.1f}M"))
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda v,_: f"${v:.0f}M"))
ax.set_title("LBE Reshuffle — Pareto Frontier (min spend / max revenue)",
             fontsize=13, fontweight="bold", color=CPX_NAVY, pad=8)
ax.spines[["top","right"]].set_visible(False)
ax.grid(alpha=0.15, color=CPX_LGREY); ax.set_axisbelow(True)
ax.legend(fontsize=9, loc="upper left", frameon=False)

# ── RIGHT: marginal efficiency along frontier ────────────────────────────────
# ΔRev/ΔSpend between consecutive frontier points (thin to reduce noise)
step_m = max(1, len(pareto)//30)
sp = p_spend[::step_m]; rv = p_med[::step_m]
marg_x = (sp[1:] + sp[:-1]) / 2
marg_y = np.diff(rv) / np.diff(sp)

ax2.axhline(1.0, color=CPX_RED, lw=1.2, ls="--", alpha=0.7, zorder=2,
            label="Break-even (1.0×)")
ax2.plot(marg_x, marg_y, color=CPX_NAVY, lw=2.0, marker="o", ms=3, zorder=3,
         label="Marginal ROI (ΔRev/ΔSpend)")
ax2.axvline(base_total/1e6, color=CPX_MGREY, lw=1.2, ls=":", zorder=2)
ax2.annotate(f"current\n${base_total/1e6:.2f}M", xy=(base_total/1e6, ax2.get_ylim()[1]),
             xytext=(base_total/1e6, marg_y.max()*0.95), fontsize=8.5,
             color=CPX_MGREY, fontweight="bold", ha="center")

ax2.set_xlabel("Total LBE media spend ($M)", fontsize=12, fontweight="bold")
ax2.set_ylabel("Marginal ROI (×)", fontsize=12, fontweight="bold")
ax2.xaxis.set_major_formatter(mticker.FuncFormatter(lambda v,_: f"${v:.1f}M"))
ax2.set_title("Marginal Efficiency Along Frontier", fontsize=13,
              fontweight="bold", color=CPX_NAVY, pad=8)
ax2.spines[["top","right"]].set_visible(False)
ax2.grid(alpha=0.15, color=CPX_LGREY); ax2.set_axisbelow(True)
ax2.legend(fontsize=9, loc="upper right", frameon=False)

fig.suptitle("LBE MMM V5e — Budget Reshuffle Analysis (posterior-based)",
             fontsize=14, fontweight="bold", color=CPX_NAVY, y=1.02)
fig.text(0.99, -0.02, "Cineplex Marketing Analytics  |  LBE MMM V5e",
         ha="right", va="bottom", fontsize=6, color=CPX_LGREY)

plt.tight_layout()
save_fig(fig, "FINAL_lbe_reshuffle_pareto.png")
display(fig)
plt.close(fig)